# Project 003 Phase 1A — Colab compute proof

This notebook regenerates two independent HG002/GRCh38 event-level benchmark runs for precisionFDA V2 submissions **60Z59** (Illumina) and **RU88N** (ONT). It never represents these observations as original precisionFDA event decisions.

Run from top to bottom in a fresh **free Google Colab CPU** runtime. Large temporary inputs stay under `/content/omicsedge_phase1a/`; only compact results are packaged from `/content/omicsedge_phase1a_results/`. The notebook fails closed on comparator identity, input integrity, annotation-gate, or locked-validation failures.


**Runtime-identity revision:** Section 4 establishes hap.py 0.3.15 from either exact in-container version evidence or, for the legacy BioContainer, the exact immutable OCI digest plus the exact Bioconda build tag and a successful executable probe. It never relabels missing CLI metadata as a reported version.


## 1. Environment information

Record the Colab host identity without changing it. CPU-only execution is required.

In [ ]:
import base64, csv, datetime as dt, hashlib, io, json, os, platform, re, shlex
import shutil, subprocess, sys, textwrap, time, traceback, uuid
from pathlib import Path

UTC = dt.timezone.utc
def now_iso():
    return dt.datetime.now(UTC).replace(microsecond=0).isoformat().replace("+00:00", "Z")

COMPUTE_STATUS = None
STATUS_EMITTED = False
PIPELINE_ABORTED = False
PIPELINE_ABORT_STATUS = None
PIPELINE_ABORT_STAGE = None

class Phase1AAbort(RuntimeError):
    """Intentional fail-closed notebook stop with a clean IPython traceback."""
    pass

def emit_status_once(status):
    global STATUS_EMITTED
    allowed = {"PHASE1A_COMPUTE_PASS", "PHASE1A_COMPUTE_FAIL", "COMPARATOR_ENVIRONMENT_BLOCKED"}
    if status not in allowed:
        raise ValueError(status)
    if not STATUS_EMITTED:
        print(status)
        STATUS_EMITTED = True

HOST_EUID = os.geteuid() if hasattr(os, "geteuid") else None
WHOAMI = subprocess.run(
    ["whoami"], text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE
).stdout.strip() or "UNKNOWN"
COLAB_HOST_ROOT = (HOST_EUID == 0)

print(json.dumps({
    "captured_at": now_iso(),
    "python": sys.version,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "colab_release_tag": os.environ.get("COLAB_RELEASE_TAG", "UNKNOWN"),
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES", "UNSET"),
    "euid": HOST_EUID,
    "whoami": WHOAMI,
    "colab_host_root": COLAB_HOST_ROOT,
}, indent=2))


## 2. Runtime/disk/RAM check

Require an x86-64 CPU runtime with at least 2 logical CPUs, 7 GiB RAM, and 6 GiB free under `/content`. No GPU or TPU is used.

In [ ]:
WORK_ROOT = Path("/content/omicsedge_phase1a")
SOURCE_DIR = WORK_ROOT / "source"
STAGED_DIR = WORK_ROOT / "staged"
TOOLS_DIR = WORK_ROOT / "tools"
RUNS_DIR = WORK_ROOT / "runs"
CONFIG_DIR = WORK_ROOT / "authoritative_config"
RESULTS_DIR = Path("/content/omicsedge_phase1a_results")
ARCHIVE_PATH = Path("/content/omicsedge_phase1a_results.tar.gz")
for p in (SOURCE_DIR, STAGED_DIR, TOOLS_DIR, RUNS_DIR, CONFIG_DIR, RESULTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Initialized before any guarded compute stage so early failures can be packaged safely.
FINAL_TAR_ARGV = ["tar", "-C", "/content", "-czf", str(ARCHIVE_PATH), RESULTS_DIR.name]

def mem_total_bytes():
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith("MemTotal:"):
            return int(line.split()[1]) * 1024
    return 0

runtime_resources = {
    "architecture": platform.machine(),
    "logical_cpus": os.cpu_count() or 0,
    "ram_bytes": mem_total_bytes(),
    "content_free_bytes": shutil.disk_usage("/content").free,
}
print(json.dumps(runtime_resources, indent=2))
resource_errors = []
if runtime_resources["architecture"] not in {"x86_64", "amd64"}:
    resource_errors.append("pinned comparator requires linux/amd64")
if runtime_resources["logical_cpus"] < 2:
    resource_errors.append("fewer than 2 logical CPUs")
if runtime_resources["ram_bytes"] < 7 * 1024**3:
    resource_errors.append("less than 7 GiB RAM")
if runtime_resources["content_free_bytes"] < 6 * 1024**3:
    resource_errors.append("less than 6 GiB free disk")
if resource_errors:
    emit_status_once("PHASE1A_COMPUTE_FAIL")
    raise RuntimeError("Colab resource preflight failed: " + "; ".join(resource_errors))


## 3. Immutable configuration

Materialize byte-for-byte snapshots of the checked-in lock, manifests, schemas, validation contract, semantics, and query. Verify each embedded SHA-256 before use. Coordinates remain: core BED `chr20:[10000000,11000000)`, display/VCF `chr20:10000001-11000000`, padded BED `chr20:[9999000,11001000)`, padded VCF `chr20:9999001-11001000`.


In [ ]:
EMBEDDED_B64 = {
  "environment/comparator-lock.yaml": "bG9ja192ZXJzaW9uOiAiMS4wLjAiCmxvY2tfaWQ6ICJwaGFzZTFhLWhhcHB5LTAuMy4xNS12Y2ZldmFsLTMuMTIuMSIKbG9ja2VkX2F0OiAiMjAyNi0wOS0yOCIKcGxhdGZvcm06CiAgb3M6IGxpbnV4CiAgYXJjaGl0ZWN0dXJlOiBhbWQ2NApoYXBfcHk6CiAgdmVyc2lvbjogIjAuMy4xNSIKICB1cHN0cmVhbV9yZXBvc2l0b3J5OiAiaHR0cHM6Ly9naXRodWIuY29tL0lsbHVtaW5hL2hhcC5weSIKICBnaXRfdGFnOiAidjAuMy4xNSIKICBnaXRfY29tbWl0OiAiODQwMTE2OTViMmZmMjQwNmMxNmEzMzUxMDZkYjY4MzFmYjY3ZmRmZSIKICBzb3VyY2VfYXJjaGl2ZV9zaGEyNTY6ICI4Mzg5MTJmMWFiODA1MjI0MTEwYWYzYzEwMzU4NDlkNTQ4M2I1Y2U0YzU2NWFlMGYwZGI3ZjcyODRhZjA2Njc2IgogIGNvbnRhaW5lcjoKICAgIHJlZmVyZW5jZTogInF1YXkuaW8vYmlvY29udGFpbmVycy9oYXAucHlAc2hhMjU2OmQ2M2I5NjNhNmNiMDFiNDgzMDM5M2IyMjM2OWU3YjkxZDI5OGU0MTU2ZGRlMzUzNzM5ZTc0ZTRjZmE0Zjk2ZDAiCiAgICB0YWdfZm9yX2h1bWFuc19vbmx5OiAiMC4zLjE1LS1weTI3aGNiNzNiM2RfMCIKICAgIG1hbmlmZXN0X2RpZ2VzdDogInNoYTI1NjpkNjNiOTYzYTZjYjAxYjQ4MzAzOTNiMjIzNjllN2I5MWQyOThlNDE1NmRkZTM1MzczOWU3NGU0Y2ZhNGY5NmQwIgogICAgY29tcHJlc3NlZF9sYXllcl9ieXRlczogMjMzNTY4NTE2CnJ0Z190b29sczoKICB2ZXJzaW9uOiAiMy4xMi4xIgogIHVwc3RyZWFtX3JlcG9zaXRvcnk6ICJodHRwczovL2dpdGh1Yi5jb20vUmVhbFRpbWVHZW5vbWljcy9ydGctdG9vbHMiCiAgZ2l0X3RhZzogIjMuMTIuMSIKICBnaXRfY29tbWl0OiAiYTlkMTZkODhiODBmYzY2MGJhM2FkN2Q2YjllN2FjYzQ1ZTBiZWZkNyIKICBsaW51eF94NjRfYXJjaGl2ZV91cmw6ICJodHRwczovL2dpdGh1Yi5jb20vUmVhbFRpbWVHZW5vbWljcy9ydGctdG9vbHMvcmVsZWFzZXMvZG93bmxvYWQvMy4xMi4xL3J0Zy10b29scy0zLjEyLjEtbGludXgteDY0LnppcCIKICBsaW51eF94NjRfYXJjaGl2ZV9ieXRlczogNTA3NDI1MzcKICBsaW51eF94NjRfYXJjaGl2ZV9zaGEyNTY6ICJiYTQzY2FkYmIzZTc5YmZmNzRlN2QyZjg1NDBkZDVlMjAxNTc0ZWM4ZTI5M2JhNTU1ZjNhZWVmZDEzNzVkZmM1Igpjb21wYXJpc29uOgogIGVuZ2luZTogInZjZmV2YWwiCiAgYW5ub3RhdGVkX3ZjZjogdHJ1ZQogIHRocmVhZHM6IDIKICBjb3JlX2ludGVydmFsOgogICAgYXNzZW1ibHk6ICJHUkNoMzgiCiAgICBjb250aWc6ICJjaHIyMCIKICAgIHN0YXJ0XzBiYXNlZDogMTAwMDAwMDAKICAgIGVuZF8wYmFzZWQ6IDExMDAwMDAwCiAgcGFkZGVkX2ludGVydmFsOgogICAgYXNzZW1ibHk6ICJHUkNoMzgiCiAgICBjb250aWc6ICJjaHIyMCIKICAgIHN0YXJ0XzBiYXNlZDogOTk5OTAwMAogICAgZW5kXzBiYXNlZDogMTEwMDEwMDAKICBpbW11dGFibGVfYXJnczoKICAgIC0gIi1WIgogICAgLSAiLS1lbmdpbmU9dmNmZXZhbCIKICAgIC0gIi0tZW5naW5lLXZjZmV2YWwtcGF0aD0vb3B0L3J0Zy10b29scy0zLjEyLjEvcnRnIgogICAgLSAiLS10aHJlYWRzPTIiCnJ1bnRpbWVfY2FwdHVyZV9yZXF1aXJlZDoKICAtIGhhcC5weV92ZXJzaW9uCiAgLSBydGdfdmVyc2lvbgogIC0gamF2YV92ZXJzaW9uCiAgLSBiY2Z0b29sc192ZXJzaW9uCiAgLSBzYW10b29sc192ZXJzaW9uCiAgLSBweXRob25fdmVyc2lvbgogIC0gZGVyaXZlZF9pbWFnZV9kaWdlc3QKICAtIGNvbW1hbmRfYXJndgogIC0gc3Rkb3V0X3NoYTI1NgogIC0gc3RkZXJyX3NoYTI1Ngo=",
  "data/manifests/source_artifacts.tsv": "c291cmNlX2FydGlmYWN0X2lkCWFydGlmYWN0X25hbWUJYXJ0aWZhY3Rfcm9sZQl1cmkJYnl0ZV9zaXplCXZlcnNpb24JY2hlY2tzdW1fYWxnb3JpdGhtCWNoZWNrc3VtX3ZhbHVlCWNoZWNrc3VtX3NvdXJjZQljaGVja3N1bV9zdGF0dXMJcmV0cmlldmFsX3N0YXR1cwlsaWNlbnNlX29yX3JldXNlX2Jhc2lzCW5vdGVzCmFydGlmYWN0LXBmZGEtcGRyLW1ldGFkYXRhCU5JU1QgUERSIG1hY2hpbmUgbWV0YWRhdGEJREFUQVNFVF9NQU5JRkVTVAlodHRwczovL2RhdGEubmlzdC5nb3Yvb2QvaWQvbWRzMi0yMzM2CTc1NjY0NgltZHMyLTIzMzYgc25hcHNob3QgMjAyNi0wOS0yOAlVTktOT1dOCVVOS05PV04JTk9ORQlOT1RfWUVUX1JFVFJJRVZFRAlOT1RfRE9XTkxPQURFRAlOSVNUIFBEUiB0ZXJtcwlNYWNoaW5lIGludmVudG9yeTsgZHluYW1pYyByZXByZXNlbnRhdGlvbiBtdXN0IGJlIGxvY2FsbHkgaGFzaGVkIGF0IGNvbXB1dGUgcHJlZmxpZ2h0LgphcnRpZmFjdC1wZmRhLXJlYWRtZQlSRUFETUVfcEZEQS1HSUFCX0NoYWxsZW5nZS1WMi5tZAlFWFBFUklNRU5UX01FVEFEQVRBCWh0dHBzOi8vZGF0YS5uaXN0Lmdvdi9vZC9kcy9hcms6Lzg4NDM0L21kczItMjMzNi9SRUFETUVfcEZEQS1HSUFCX0NoYWxsZW5nZS1WMi5tZAkzMTA2MwltZHMyLTIzMzYJc2hhMjU2CWVjMDhlNTU1ODA4ODEzNGNmNzZmZTQ4ZTQ3NTQ4ZDE5NmRmZTdhODg1Nzk2MzE4ZDM0MDFlYWVkNTEyZTM5OTAJUFVCTElTSEVSCVBVQkxJU0hFUl9WRVJJRklFRAlOT1RfRE9XTkxPQURFRAlDQzAgcGx1cyBOSVNUIERhdGEgVXNlIFBvbGljeQlEYXRhc2V0IGFuZCB0ZWNobm9sb2d5IGRvY3VtZW50YXRpb24uCmFydGlmYWN0LXBmZGEtc3VibWlzc2lvbi1tZXRob2RzCXN1Ym1pc3Npb25fbWV0aG9kcy50eHQJRVhQRVJJTUVOVF9NRVRBREFUQQlodHRwczovL2RhdGEubmlzdC5nb3Yvb2QvZHMvYXJrOi84ODQzNC9tZHMyLTIzMzYvc3VibWlzc2lvbl9tZXRob2RzLnR4dAkzNjkxMwltZHMyLTIzMzYJc2hhMjU2CWNmZWI1MTIzNTgxMzNjMTI4Zjc4ODQ1MzkzM2I5MzhmMTNiYTc3ODM1YTUzOWYxZWFjZjYyMjM4YmIxYzg3NDAJUFVCTElTSEVSCVBVQkxJU0hFUl9WRVJJRklFRAlOT1RfRE9XTkxPQURFRAlDQzAgcGx1cyBOSVNUIERhdGEgVXNlIFBvbGljeQlQYXJ0aWNpcGFudC1zdXBwbGllZCBwaXBlbGluZSBtZXRob2RzOyBzb21lIHZlcnNpb25zIGFyZSBleHBsaWNpdGx5IHVua25vd24uCmFydGlmYWN0LXBmZGEtYW5vbnltaXplZC1tZXRhZGF0YQlhbm9ueW1pemVkX21ldGFkYXRhX3RhYmxlLnRzdglFWFBFUklNRU5UX01FVEFEQVRBCWh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS91c25pc3Rnb3YvZ2lhYi1wRkRBLTJuZC1jaGFsbGVuZ2UvZGI5YmEwNjdkMzc0MTc2MjUyYzA2NjI4Nzk1ZDdlMzY4YWFkZTZlMC9kYXRhLXJhdy9hbm9ueW1pemVkX21ldGFkYXRhX3RhYmxlLnRzdgk1NDA4CWdpdCBkYjliYTA2N2QzNzQxNzYyNTJjMDY2Mjg3OTVkN2UzNjhhYWRlNmUwCXNoYTI1NglkZWUwMWMyZTcyMGEwNDUxYWIwNDI2Yzg1ZmU5ZjU0YzQxOTQyYzg5ZDk0M2VmNDBmZDEyMzU4N2ExMjYyZDVjCUNPTVBVVEVEX09OX1JFVFJJRVZBTAlDT01QVVRFRF9OT1RfWUVUX1ZFUklGSUVECU5PVF9ET1dOTE9BREVECU5JU1QgcmVwb3NpdG9yeSB0ZXJtcwlNYXBzIElEcyB0byB0ZWNobm9sb2d5IGFuZCBzdWJtaXNzaW9uIG5hbWVzOyBwaW5uZWQgYnkgY29tbWl0LgphcnRpZmFjdC1xdWVyeS02MHo1OS1oZzAwMgk2MFo1OV9IRzAwMi52Y2YuZ3oJUVVFUllfVkNGCWh0dHBzOi8vZGF0YS5uaXN0Lmdvdi9vZC9kcy9hcms6Lzg4NDM0L21kczItMjMzNi9zdWJtaXNzaW9uX3ZjZnMvNjBaNTkvNjBaNTlfSEcwMDIudmNmLmd6CTc4NDQ5Nzg0CXByZWNpc2lvbkZEQSBWMiBOSVNUIGRlcG9zaXQJc2hhMjU2CTMxOTg1MjAzMzkzZTQ3OTVmMTdhZDA5MDg1MGQxZDA2Y2JhMzc5MzViYWZjYTJkNzhjMWVlYmI3Njk0MjFlODIJUFVCTElTSEVSCVBVQkxJU0hFUl9WRVJJRklFRAlOT1RfRE9XTkxPQURFRAlDQzAgcGx1cyBOSVNUIERhdGEgVXNlIFBvbGljeQlObyBwdWJsaWMgVEJJIG9yIENTSTsgd2hvbGUgYXJ0aWZhY3QgbXVzdCBiZSByZXRyaWV2ZWQgYmVmb3JlIGxvY2FsIGluZGV4aW5nLgphcnRpZmFjdC1xdWVyeS1ydTg4bi1oZzAwMglSVTg4Tl9IRzAwMi52Y2YuZ3oJUVVFUllfVkNGCWh0dHBzOi8vZGF0YS5uaXN0Lmdvdi9vZC9kcy9hcms6Lzg4NDM0L21kczItMjMzNi9zdWJtaXNzaW9uX3ZjZnMvUlU4OE4vUlU4OE5fSEcwMDIudmNmLmd6CTkwMDEwODMwCXByZWNpc2lvbkZEQSBWMiBOSVNUIGRlcG9zaXQJc2hhMjU2CTFmM2JmMGY0NjQwNzg3ZmEwMzU1YjVlMmE0NGU2M2MwNzMxOTZmYzU5ZDNjNzVhMTc4YWM2MzY3NDg2ODVjYmMJUFVCTElTSEVSCVBVQkxJU0hFUl9WRVJJRklFRAlOT1RfRE9XTkxPQURFRAlDQzAgcGx1cyBOSVNUIERhdGEgVXNlIFBvbGljeQlObyBwdWJsaWMgVEJJIG9yIENTSTsgd2hvbGUgYXJ0aWZhY3QgbXVzdCBiZSByZXRyaWV2ZWQgYmVmb3JlIGxvY2FsIGluZGV4aW5nLgphcnRpZmFjdC10cnV0aC1oZzAwMi1ncmNoMzgtdjQyMQlIRzAwMl9HUkNoMzhfMV8yMl92NC4yLjFfYmVuY2htYXJrLnZjZi5neglUUlVUSF9WQ0YJaHR0cHM6Ly9mdHAtdHJhY2UubmNiaS5ubG0ubmloLmdvdi9SZWZlcmVuY2VTYW1wbGVzL2dpYWIvcmVsZWFzZS9Bc2hrZW5hemltVHJpby9IRzAwMl9OQTI0Mzg1X3Nvbi9OSVNUdjQuMi4xL0dSQ2gzOC9IRzAwMl9HUkNoMzhfMV8yMl92NC4yLjFfYmVuY2htYXJrLnZjZi5negkxNTYyNTI5NDQJR0lBQiBIRzAwMiB2NC4yLjEJVU5LTk9XTglVTktOT1dOCU5PTkUJTk9UX1lFVF9SRVRSSUVWRUQJTk9UX0RPV05MT0FERUQJTklTVCBvcGVuIHRlcm1zIHdpdGggYXR0cmlidXRpb24vY2hhbmdlIG5vdGljZQlQdWJsaXNoZXIgZGlyZWN0b3J5IHN1cHBsaWVzIG5vIHByaW1hcnktZmlsZSBjaGVja3N1bSBzaWRlY2FyOyBjb21wdXRlIFNIQS0yNTYgb24gcmV0cmlldmFsLgphcnRpZmFjdC10cnV0aC1oZzAwMi1ncmNoMzgtdjQyMS10YmkJSEcwMDJfR1JDaDM4XzFfMjJfdjQuMi4xX2JlbmNobWFyay52Y2YuZ3oudGJpCVRSVVRIX0lOREVYCWh0dHBzOi8vZnRwLXRyYWNlLm5jYmkubmxtLm5paC5nb3YvUmVmZXJlbmNlU2FtcGxlcy9naWFiL3JlbGVhc2UvQXNoa2VuYXppbVRyaW8vSEcwMDJfTkEyNDM4NV9zb24vTklTVHY0LjIuMS9HUkNoMzgvSEcwMDJfR1JDaDM4XzFfMjJfdjQuMi4xX2JlbmNobWFyay52Y2YuZ3oudGJpCTE2NTc3NDcJR0lBQiBIRzAwMiB2NC4yLjEJVU5LTk9XTglVTktOT1dOCU5PTkUJTk9UX1lFVF9SRVRSSUVWRUQJTk9UX0RPV05MT0FERUQJTklTVCBvcGVuIHRlcm1zIHdpdGggYXR0cmlidXRpb24vY2hhbmdlIG5vdGljZQlDb21wdXRlIFNIQS0yNTYgb24gcmV0cmlldmFsLgphcnRpZmFjdC1iZWQtaGcwMDItZ3JjaDM4LXY0MjEJSEcwMDJfR1JDaDM4XzFfMjJfdjQuMi4xX2JlbmNobWFya19ub2luY29uc2lzdGVudC5iZWQJQkVOQ0hNQVJLX0JFRAlodHRwczovL2Z0cC10cmFjZS5uY2JpLm5sbS5uaWguZ292L1JlZmVyZW5jZVNhbXBsZXMvZ2lhYi9yZWxlYXNlL0FzaGtlbmF6aW1UcmlvL0hHMDAyX05BMjQzODVfc29uL05JU1R2NC4yLjEvR1JDaDM4L0hHMDAyX0dSQ2gzOF8xXzIyX3Y0LjIuMV9iZW5jaG1hcmtfbm9pbmNvbnNpc3RlbnQuYmVkCTExNDk0MDIxCUdJQUIgSEcwMDIgdjQuMi4xCVVOS05PV04JVU5LTk9XTglOT05FCU5PVF9ZRVRfUkVUUklFVkVECU5PVF9ET1dOTE9BREVECU5JU1Qgb3BlbiB0ZXJtcyB3aXRoIGF0dHJpYnV0aW9uL2NoYW5nZSBub3RpY2UJRXhhY3QgbWF0Y2hpbmcgYmVuY2htYXJrIHJlZ2lvbnM7IGNvbXB1dGUgU0hBLTI1NiBvbiByZXRyaWV2YWwuCmFydGlmYWN0LXJlZmVyZW5jZS1ncmNoMzgtbm8tYWx0CUdDQV8wMDAwMDE0MDUuMTVfR1JDaDM4X25vX2FsdF9hbmFseXNpc19zZXQuZmFzdGEuZ3oJUkVGRVJFTkNFX0ZBU1RBCWh0dHBzOi8vZnRwLXRyYWNlLm5jYmkubmxtLm5paC5nb3YvUmVmZXJlbmNlU2FtcGxlcy9naWFiL3JlbGVhc2UvcmVmZXJlbmNlcy9HUkNoMzgvR0NBXzAwMDAwMTQwNS4xNV9HUkNoMzhfbm9fYWx0X2FuYWx5c2lzX3NldC5mYXN0YS5negk4ODYzNDQ2MTgJR0NBXzAwMDAwMTQwNS4xNSBHUkNoMzggbm8tYWx0IGFuYWx5c2lzIHNldAltZDUJM2EzMzQ3ZWFlMDg5M2Y5NmVjZjQ5NWQxYzM5ZTIyODQJUFVCTElTSEVSCVBVQkxJU0hFUl9WRVJJRklFRAlOT1RfRE9XTkxPQURFRAlDQzAgcGx1cyBOSVNUIERhdGEgVXNlIFBvbGljeQlCR1pGIHJlZmVyZW5jZTsgYWxzbyBjb21wdXRlIFNIQS0yNTYgb24gcmV0cmlldmFsLgphcnRpZmFjdC1yZWZlcmVuY2UtZ3JjaDM4LW5vLWFsdC1mYWkJR0NBXzAwMDAwMTQwNS4xNV9HUkNoMzhfbm9fYWx0X2FuYWx5c2lzX3NldC5mYXN0YS5nei5mYWkJUkVGRVJFTkNFX0lOREVYCWh0dHBzOi8vZnRwLXRyYWNlLm5jYmkubmxtLm5paC5nb3YvUmVmZXJlbmNlU2FtcGxlcy9naWFiL3JlbGVhc2UvcmVmZXJlbmNlcy9HUkNoMzgvR0NBXzAwMDAwMTQwNS4xNV9HUkNoMzhfbm9fYWx0X2FuYWx5c2lzX3NldC5mYXN0YS5nei5mYWkJNzgwNAlHQ0FfMDAwMDAxNDA1LjE1IEdSQ2gzOCBuby1hbHQgYW5hbHlzaXMgc2V0CW1kNQk1ZmRkYmMxMDljODI5ODBmOTQzNmFhNWMyMWE1N2M2MQlQVUJMSVNIRVIJUFVCTElTSEVSX1ZFUklGSUVECU5PVF9ET1dOTE9BREVECUNDMCBwbHVzIE5JU1QgRGF0YSBVc2UgUG9saWN5CUZBSSBmb3IgQkdaRiByZWZlcmVuY2UuCmFydGlmYWN0LXJlZmVyZW5jZS1ncmNoMzgtbm8tYWx0LWd6aQlHQ0FfMDAwMDAxNDA1LjE1X0dSQ2gzOF9ub19hbHRfYW5hbHlzaXNfc2V0LmZhc3RhLmd6Lmd6aQlSRUZFUkVOQ0VfSU5ERVgJaHR0cHM6Ly9mdHAtdHJhY2UubmNiaS5ubG0ubmloLmdvdi9SZWZlcmVuY2VTYW1wbGVzL2dpYWIvcmVsZWFzZS9yZWZlcmVuY2VzL0dSQ2gzOC9HQ0FfMDAwMDAxNDA1LjE1X0dSQ2gzOF9ub19hbHRfYW5hbHlzaXNfc2V0LmZhc3RhLmd6Lmd6aQk3NzA2NjQJR0NBXzAwMDAwMTQwNS4xNSBHUkNoMzggbm8tYWx0IGFuYWx5c2lzIHNldAltZDUJY2ZhMWVlMTFiMWVjYjI5ZjkzNjU3OGIyNzAxNGZiZTAJUFVCTElTSEVSCVBVQkxJU0hFUl9WRVJJRklFRAlOT1RfRE9XTkxPQURFRAlDQzAgcGx1cyBOSVNUIERhdGEgVXNlIFBvbGljeQlHWkkgZm9yIEJHWkYgcmFuZ2UgYWNjZXNzLgphcnRpZmFjdC1oYXBweS1jb250YWluZXIJaGFwLnB5IEJpb0NvbnRhaW5lcglTT0ZUV0FSRV9JTUFHRQlxdWF5LmlvL2Jpb2NvbnRhaW5lcnMvaGFwLnB5QHNoYTI1NjpkNjNiOTYzYTZjYjAxYjQ4MzAzOTNiMjIzNjllN2I5MWQyOThlNDE1NmRkZTM1MzczOWU3NGU0Y2ZhNGY5NmQwCTIzMzU2ODUxNgloYXAucHkgMC4zLjE1IHB5MjdoY2I3M2IzZF8wCW9jaS1tYW5pZmVzdC1kaWdlc3QJZDYzYjk2M2E2Y2IwMWI0ODMwMzkzYjIyMzY5ZTdiOTFkMjk4ZTQxNTZkZGUzNTM3MzllNzRlNGNmYTRmOTZkMAlSRUdJU1RSWQlQVUJMSVNIRVJfVkVSSUZJRUQJTk9UX0RPV05MT0FERUQJQlNELTItQ2xhdXNlIGZvciBoYXAucHk7IGJ1bmRsZWQgY29tcG9uZW50cyByZXRhaW4gdGhlaXIgbGljZW5zZXMJTWFuaWZlc3QgZGlnZXN0IHBpbm5lZDsgY29tcHJlc3NlZCBsYXllci1zaXplIHN1bS4KYXJ0aWZhY3QtcnRnLXRvb2xzLTMxMjEJcnRnLXRvb2xzLTMuMTIuMS1saW51eC14NjQuemlwCVNPRlRXQVJFX0FSQ0hJVkUJaHR0cHM6Ly9naXRodWIuY29tL1JlYWxUaW1lR2Vub21pY3MvcnRnLXRvb2xzL3JlbGVhc2VzL2Rvd25sb2FkLzMuMTIuMS9ydGctdG9vbHMtMy4xMi4xLWxpbnV4LXg2NC56aXAJNTA3NDI1MzcJUlRHIFRvb2xzIDMuMTIuMQlzaGEyNTYJYmE0M2NhZGJiM2U3OWJmZjc0ZTdkMmY4NTQwZGQ1ZTIwMTU3NGVjOGUyOTNiYTU1NWYzYWVlZmQxMzc1ZGZjNQlDT01QVVRFRF9PTl9SRVRSSUVWQUwJQ09NUFVURURfTk9UX1lFVF9WRVJJRklFRAlOT1RfRE9XTkxPQURFRAlCU0QtMi1DbGF1c2UJT2ZmaWNpYWwgcmVsZWFzZSBhcmNoaXZlIGluY2x1ZGVzIExpbnV4IHg2NCBKUkU7IGhhc2ggY29tcHV0ZWQgZHVyaW5nIGJvb3RzdHJhcCBtZXRhZGF0YSBhdWRpdC4K",
  "data/manifests/selected_experiments.tsv": "ZXhwZXJpbWVudF9pZAlzb3VyY2VfY29ycHVzCXNvdXJjZV9zdWJtaXNzaW9uX2lkCXNhbXBsZV9pZAl0ZWNobm9sb2d5CXBsYXRmb3JtCWNvdmVyYWdlCXBpcGVsaW5lX25hbWUJYWxpZ25lcl9uYW1lcwlhbGlnbmVyX3ZlcnNpb25zCWNhbGxlcl9uYW1lcwljYWxsZXJfdmVyc2lvbnMJcXVlcnlfYXJ0aWZhY3RfaWQJc2VsZWN0aW9uX2Jhc2lzCXBlcmZvcm1hbmNlX2NvbnN1bHRlZF9mb3Jfc2VsZWN0aW9uCW1ldGFkYXRhX3N0YXR1cwpleHAtcGZkYS12Mi02MHo1OS1oZzAwMglwcmVjaXNpb25GREEgVHJ1dGggQ2hhbGxlbmdlIFYyIC8gTklTVCBQRFIgbWRzMi0yMzM2CTYwWjU5CUhHMDAyCUlMTFVNSU5BCU5vdmFTZXEgNjAwMAkzNXgJUk4tSWxsdW1pbmEJQldBCTAuNy4xNQlOZXVTb21hdGljO0dBVEs7UGljYXJkCVVOS05PV047NC4xLjcuMDsyLjIyLjQJYXJ0aWZhY3QtcXVlcnktNjB6NTktaGcwMDIJUGhhc2UgMEIgaGFkIGFscmVhZHkgdmVyaWZpZWQgdGhlIGV4YWN0IHB1YmxpYyBCR1pGIGFydGlmYWN0IGhlYWRlciBjaGVja3N1bSBzaW5nbGUtdGVjaG5vbG9neSBtYXBwaW5nIGFuZCBtZXRob2RzOyBjaG9zZW4gYmVmb3JlIHNjb3JlIGluc3BlY3Rpb24JZmFsc2UJUEFSVElBTF9FWEFDVF9DQUxMRVJfVkVSU0lPTl9VTktOT1dOCmV4cC1wZmRhLXYyLXJ1ODhuLWhnMDAyCXByZWNpc2lvbkZEQSBUcnV0aCBDaGFsbGVuZ2UgVjIgLyBOSVNUIFBEUiBtZHMyLTIzMzYJUlU4OE4JSEcwMDIJT05UCVByb21ldGhJT04gUjkuNAk0N3gJUEVQUEVSLURlZXBWYXJpYW50IChPTlQpCW1pbmltYXAyCTIuMTctcjk0MQlQRVBQRVI7RGVlcFZhcmlhbnQJVU5LTk9XTjtVTktOT1dOCWFydGlmYWN0LXF1ZXJ5LXJ1ODhuLWhnMDAyCVVuYW1iaWd1b3VzIHB1YmxpYyBzaW5nbGUtdGVjaG5vbG9neSBPTlQgYXJ0aWZhY3QgYW5kIHBhcnRpY2lwYW50LXN1cHBsaWVkIG1ldGhvZHM7IGNob3NlbiBiZWZvcmUgc2NvcmUgaW5zcGVjdGlvbglmYWxzZQlQQVJUSUFMX0NBTExFUl9WRVJTSU9OU19VTktOT1dOCg==",
  "docs/phase1a_compute_plan.md": "IyBQaGFzZSAxQSBjb21wdXRlIGFuZCBHb29nbGUgQ29sYWIgcGxhbgoKVGhpcyBkb2N1bWVudCBkZWZpbmVzIGEgZnV0dXJlIGNvbXB1dGUgcHJvb2YuIFRoZSBib290c3RyYXAgZGlkIG5vdCBkb3dubG9hZApnZW5vbWljIGRhdGEgYW5kIGRpZCBub3QgcnVuIGhhcC5weSBvciB2Y2ZldmFsLgoKIyMgTG9ja2VkIHN1Ym1pc3Npb25zCgp8IFN1Ym1pc3Npb24gfCBUZWNobm9sb2d5IGNvbmRpdGlvbiB8IFB1YmxpYyBwaXBlbGluZSBtZXRhZGF0YSB8IFNlbGVjdGlvbiBiYXNpcyB8CnwtLS18LS0tfC0tLXwtLS18CnwgYDYwWjU5YCAvIEhHMDAyIHwgSWxsdW1pbmEgTm92YVNlcSA2MDAwLCBjaGFsbGVuZ2UgaW5wdXQgcmVwb3J0ZWQgYXQgMzV4IHwgYFJOLUlsbHVtaW5hYDsgQldBIDAuNy4xNTsgTmV1U29tYXRpYyB2ZXJzaW9uIGBVTktOT1dOYDsgR0FUSyA0LjEuNy4wOyBQaWNhcmQgMi4yMi40IHwgUGhhc2UgMEIgYWxyZWFkeSB2ZXJpZmllZCB0aGlzIGV4YWN0IEJHWkYgYXJ0aWZhY3QvaGVhZGVyL2NoZWNrc3VtIGFuZCBwdWJsaWMgbWV0aG9kczsgbm8gcmVzdWx0IG1ldHJpYyB3YXMgY29uc3VsdGVkLiB8CnwgYFJVODhOYCAvIEhHMDAyIHwgT05UIFByb21ldGhJT04gUjkuNCwgY2hhbGxlbmdlIGlucHV0IHJlcG9ydGVkIGF0IDQ3eDsgY2hhbGxlbmdlIEZBU1RRcyB3ZXJlIEd1cHB5IDMuNiBiYXNlY2FsbGVkIHwgYFBFUFBFUi1EZWVwVmFyaWFudCAoT05UKWA7IG1pbmltYXAyIDIuMTctcjk0MTsgUEVQUEVSIGFuZCBEZWVwVmFyaWFudCB2ZXJzaW9ucyBgVU5LTk9XTmAgfCBTaW5nbGUtdGVjaG5vbG9neSBPTlQgYXJ0aWZhY3Qgd2l0aCBleGFjdCBQRFIgbWFwcGluZyBhbmQgbWV0aG9kczsgbm8gcmVzdWx0IG1ldHJpYyB3YXMgY29uc3VsdGVkLiB8CgpUaGUgbWV0YWRhdGEgY2xhaW1zIGFib3ZlIGNvbWUgZnJvbSB0aGUgTklTVCBQRFIgZG9jdW1lbnRhdGlvbiwgdGhlIHBpbm5lZApOSVNUIGFuYWx5c2lzLXJlcG9zaXRvcnkgbWV0YWRhdGEgdGFibGUsIGFuZCBgc3VibWlzc2lvbl9tZXRob2RzLnR4dGAuIE1pc3NpbmcKdmVyc2lvbnMgcmVtYWluIGBVTktOT1dOYC4KCiMjIExvY2tlZCB0cnV0aCwgcmVmZXJlbmNlLCBhbmQgaW50ZXJ2YWwKCi0gVHJ1dGg6IGBIRzAwMl9HUkNoMzhfMV8yMl92NC4yLjFfYmVuY2htYXJrLnZjZi5nemAgcGx1cyBpdHMgcHVibGljIFRCSS4KLSBCZW5jaG1hcmsgcmVnaW9uczoKICBgSEcwMDJfR1JDaDM4XzFfMjJfdjQuMi4xX2JlbmNobWFya19ub2luY29uc2lzdGVudC5iZWRgLgotIFJlZmVyZW5jZToKICBgR0NBXzAwMDAwMTQwNS4xNV9HUkNoMzhfbm9fYWx0X2FuYWx5c2lzX3NldC5mYXN0YS5nemAgcGx1cyBGQUkvR1pJLgotIENvcmUgaW50ZXJ2YWwgKHJlZ2lzdHJ5IGNvb3JkaW5hdGVzKTogYGNocjIwOlsxMCwwMDAsMDAwLDExLDAwMCwwMDApYCwKICAwLWJhc2VkIGhhbGYtb3BlbiwgZXhhY3RseSAxLDAwMCwwMDAgYmFzZXMuCi0gSHVtYW4gZGlzcGxheSBpbnRlcnZhbDogYGNocjIwOjEwLDAwMCwwMDEtMTEsMDAwLDAwMGAsIDEtYmFzZWQgaW5jbHVzaXZlLgotIEV4dHJhY3Rpb24gaW50ZXJ2YWw6IGBjaHIyMDpbOSw5OTksMDAwLDExLDAwMSwwMDApYCwgcHJvdmlkaW5nIDEsMDAwIGJhc2VzCiAgb2YgcGFkZGluZyBvbiBlYWNoIHNpZGUuCgpUaGUgaW50ZXJ2YWwgd2FzIHNlbGVjdGVkIGJlZm9yZSBjb21wYXJpc29uIGJ5IGFuIGVuZ2luZWVyaW5nIHJ1bGU6IGEgZml4ZWQKMSBNYiB3aW5kb3cgb24gdGhlIHJlbGF0aXZlbHkgc21hbGwgYXV0b3NvbWUgY2hyMjAsIGF3YXkgZnJvbSBjaHJvbW9zb21lIGVuZHMKYW5kIHRoZSBtb2RlbGVkIGNlbnRyb21lcmUuIE5vIHN1Ym1pc3Npb24gcmVzdWx0IG9yIGVycm9yIGRpc3RyaWJ1dGlvbiB3YXMKaW5zcGVjdGVkLiBUaGUgYmVuY2htYXJrIEJFRCBvdmVybGFwIGhhcyBpbnRlbnRpb25hbGx5IG5vdCBiZWVuIGluc3BlY3RlZApkdXJpbmcgYm9vdHN0cmFwLiBJZiB0aGUgbG9ja2VkIGNvcmUgaGFzIG5vIGFzc2Vzc2FibGUgYmFzZXMsIHRoYXQgaXMgYSBsb2NrZWQKRkFJTCBhbmQgcmVxdWlyZXMgYSB2ZXJzaW9uZWQgZGVzaWduIHJldmlzaW9uLCBub3QgYW4gYWQgaG9jIGludGVydmFsIGNoYW5nZS4KCiMjIExvY2tlZCBjb21wYXJhdG9yIGVudmlyb25tZW50CgpgZW52aXJvbm1lbnQvY29tcGFyYXRvci1sb2NrLnlhbWxgIGlzIGF1dGhvcml0YXRpdmUuCgotIGhhcC5weSBgMC4zLjE1YCwgdXBzdHJlYW0gdGFnIGNvbW1pdAogIGA4NDAxMTY5NWIyZmYyNDA2YzE2YTMzNTEwNmRiNjgzMWZiNjdmZGZlYDsKLSBCaW9Db250YWluZXJzIGhhcC5weSBpbWFnZQogIGBxdWF5LmlvL2Jpb2NvbnRhaW5lcnMvaGFwLnB5QHNoYTI1NjpkNjNiOTYzYTZjYjAxYjQ4MzAzOTNiMjIzNjllN2I5MWQyOThlNDE1NmRkZTM1MzczOWU3NGU0Y2ZhNGY5NmQwYDsKLSBSVEcgVG9vbHMvdmNmZXZhbCBgMy4xMi4xYCwgdXBzdHJlYW0gdGFnIGNvbW1pdAogIGBhOWQxNmQ4OGI4MGZjNjYwYmEzYWQ3ZDZiOWU3YWNjNDVlMGJlZmQ3YDsKLSBvZmZpY2lhbCBSVEcgTGludXggeDY0IFpJUCBTSEEtMjU2CiAgYGJhNDNjYWRiYjNlNzliZmY3NGU3ZDJmODU0MGRkNWUyMDE1NzRlYzhlMjkzYmE1NTVmM2FlZWZkMTM3NWRmYzVgOwotIGNvbXBhcmlzb24gZW5naW5lIGB2Y2ZldmFsYDsgYW5kCi0gYW5ub3RhdGVkIGhhcC5weSBvdXRwdXQgZW5hYmxlZCB3aXRoIGAtVmAuCgpUaGUgaGFwLnB5IDAuMy4xNSBzb3VyY2UgcmVsZWFzZSBpdHNlbGYgdXBkYXRlZCBpdHMgYnVuZGxlZCBSVEcgZGVwZW5kZW5jeSB0bwozLjEyLjEuIFRoZSBzZXBhcmF0ZSBSVEcgZGlzdHJpYnV0aW9uIGlzIG5ldmVydGhlbGVzcyBsb2NrZWQgZXhwbGljaXRseSBzbwp0aGUgZXhlY3V0YWJsZSBpbnZva2VkIGluIENvbGFiIGlzIGF0dHJpYnV0YWJsZS4KCkJlZm9yZSBnZW5vbWljIGNvbXB1dGUsIHRoZSBydW50aW1lIG11c3QgcHJpbnQgYW5kIGNhcHR1cmUgaGFwLnB5LCBSVEcsIEphdmEsCmJjZnRvb2xzLCBzYW10b29scywgYW5kIFB5dGhvbiB2ZXJzaW9ucy4gQSBkZXJpdmVkIGltYWdlIGJ1aWx0IGZyb20KYGVudmlyb25tZW50L0RvY2tlcmZpbGUucGhhc2UxYWAgbXVzdCBiZSB0YWdnZWQgYnkgaXRzIGNvbnRlbnQgZGlnZXN0IGluIHRoZQpydW4gcmVjb3JkOyB0aGUgc291cmNlIGxvY2sgYWxvbmUgaXMgbm90IGEgc3Vic3RpdHV0ZSBmb3IgdGhlIGJ1aWx0IGRpZ2VzdC4KCiMjIENvbGFiIGV4ZWN1dGlvbiBzdHJhdGVneQoKVGhlIGZ1dHVyZSBub3RlYm9vayBpcyBkZWxpYmVyYXRlbHkgc3RhZ2VkIGFuZCBmYWlsLWNsb3NlZDoKCjEuIFN0YXJ0IGEgZnJlc2ggQ1BVIENvbGFiIHJ1bnRpbWUuIFJlY29yZCBydW50aW1lIG1ldGFkYXRhIGFuZCBhbGxvY2F0ZSBhCiAgIHBlcnNpc3RlbnQgd29yayBkaXJlY3RvcnksIHByZWZlcmFibHkgbW91bnRlZCBHb29nbGUgRHJpdmUgZm9yIGNhY2hlIG9ubHkuCjIuIEluc3RhbGwgYSBwaW5uZWQgcm9vdGxlc3MgY29udGFpbmVyIHJ1bm5lciBhbmQgcHVsbCB0aGUgbG9ja2VkIGhhcC5weSBpbWFnZQogICBieSBkaWdlc3QuIERvd25sb2FkIHRoZSBsb2NrZWQgUlRHIFpJUCwgdmVyaWZ5IGl0cyBTSEEtMjU2IGJlZm9yZSBleHRyYWN0aW9uLAogICBhbmQgZXhwb3NlIGl0cyBgcnRnYCBleGVjdXRhYmxlIHRvIHRoZSBjb250YWluZXIuIEFsdGVybmF0aXZlbHkgYnVpbGQgdGhlCiAgIGNoZWNrZWQtaW4gZGVyaXZlZCBEb2NrZXJmaWxlIGVsc2V3aGVyZSBhbmQgcHVsbCB0aGF0IGltYWdlIGJ5IGl0cyByZWNvcmRlZAogICBkaWdlc3Q7IG5ldmVyIHVzZSBhIG11dGFibGUgdGFnIGZvciBhIHNjaWVudGlmaWMgcnVuLgozLiBEb3dubG9hZCB0aGUgdHdvIHdob2xlIHF1ZXJ5IFZDRnMgYmVjYXVzZSB0aGUgUERSIGRlcG9zaXQgaGFzIG5vIFRCSS9DU0kuCiAgIFZlcmlmeSB0aGVpciBwdWJsaXNoZWQgU0hBLTI1NiB2YWx1ZXMgYmVmb3JlIHVzZS4KNC4gRG93bmxvYWQgdGhlIEdJQUIgQkVEIGFuZCBwdWJsaWMgaW5kZXhlcy4gVXNlIHRoZSBpbmRleGVkIEdJQUIgdHJ1dGggYW5kCiAgIEJHWkYgcmVmZXJlbmNlIHRvIHJldHJpZXZlIG9ubHkgdGhlIHBhZGRlZCBjaHIyMCBzbGljZSB3aGVyZSB0aGUgdG9vbGNoYWluCiAgIHN1cHBvcnRzIEhUVFAgcmFuZ2UgYWNjZXNzLiBJZiByYW5nZSByZXRyaWV2YWwgaXMgdW5yZWxpYWJsZSwgdXNlIHRoZQogICBkb2N1bWVudGVkIGZ1bGwtZG93bmxvYWQgZmFsbGJhY2sgd2l0aG91dCBjaGFuZ2luZyBhcnRpZmFjdCBpZGVudGl0eS4KNS4gQ29tcHV0ZSBhbmQgcmVjb3JkIFNIQS0yNTYgZm9yIGV2ZXJ5IGRvd25sb2FkZWQgZmlsZSwgaW5jbHVkaW5nIEdJQUIgZmlsZXMKICAgZm9yIHdoaWNoIG5vIHB1YmxpYyBTSEEtMjU2IHNpZGVjYXIgaXMgc3VwcGxpZWQuIFZhbGlkYXRlIGV4cGVjdGVkIGJ5dGUgc2l6ZXMKICAgYW5kIHRoZSBwdWJsaXNoZWQgcmVmZXJlbmNlIE1ENSB2YWx1ZXMuCjYuIE5vcm1hbGl6ZSBvbmx5IGNvbnRhaW5lci9zb3J0L2luZGV4IHByZXJlcXVpc2l0ZXMsIHJldGFpbmluZyB0aGUgb3JpZ2luYWwKICAgcXVlcnkgYXJ0aWZhY3RzIGFuZCByZWNvcmRpbmcgZXZlcnkgZGVyaXZlZCBzbGljZS9pbmRleCBhcyBhIG5ldwogICBgU09VUkNFX0FSVElGQUNUYC4gTmV2ZXIgb3ZlcndyaXRlIGEgc291cmNlIG9iamVjdC4KNy4gQnVpbGQgYW4gUlRHIFNERiBmcm9tIHRoZSBleGFjdCByZWZlcmVuY2Ugc2xpY2UuIEludGVyc2VjdCB0aGUgYmVuY2htYXJrIEJFRAogICB3aXRoIHRoZSBwYWRkZWQgaW50ZXJ2YWwuIFByZXNlcnZlIGNvcmUgYW5kIHBhZGRpbmcgYm91bmRhcmllcyBzZXBhcmF0ZWx5Lgo4LiBSdW4gdGhlIHNhbWUgaGFwLnB5L3ZjZmV2YWwgY29tbWFuZCB0ZW1wbGF0ZSBvbmNlIHBlciBleHBlcmltZW50IHdpdGggYC1WYCwKICAgdGhlIGxvY2tlZCB0cnV0aCwgZWZmZWN0aXZlIEJFRCwgcmVmZXJlbmNlL1NERiwgYW5kIHBhcmFtZXRlcnMuIENhcHR1cmUKICAgc3Rkb3V0LCBzdGRlcnIsIGV4aXQgc3RhdHVzLCB3YWxsIHRpbWUsIHBlYWsgUlNTIHdoZXJlIGF2YWlsYWJsZSwgYW5kIGNvbW1hbmQKICAgYXJndiBhcyBvdXRwdXQgYXJ0aWZhY3RzLgo5LiBOb3JtYWxpemUgYW5ub3RhdGVkIHJlY29yZHMgdG8gdGhlIHNpeCBzY2hlbWFzLCBleGNsdWRpbmcgYm91bmRhcnktb25seQogICBkaWFnbm9zdGljcyBmcm9tIHRoZSBldmlkZW5jZSBleHBvcnQgd2hpbGUgcmV0YWluaW5nIHRoZW0gaW4gYXVkaXQgb3V0cHV0LgoxMC4gV3JpdGUgdHlwZWQgUGFycXVldCB0YWJsZXMsIGNyZWF0ZSBhIER1Y2tEQiBkYXRhYmFzZS92aWV3cywgcnVuIHRoZSBsb2NrZWQKICAgIGV2ZW50IHF1ZXJ5LCBhbmQgZXhlY3V0ZSBhbGwgUGhhc2UgMUEgdmFsaWRhdGlvbiBjaGVja3MuCjExLiBQZXJzaXN0IG1hbmlmZXN0cyBhbmQgY2hlY2tzdW1zIGJlZm9yZSB0aGUgZXBoZW1lcmFsIHJ1bnRpbWUgZW5kcy4gRG8gbm90CiAgICBwdWJsaXNoIHJlbW90ZWx5IGFzIHBhcnQgb2YgUGhhc2UgMUEuCgpUaGUgbm90ZWJvb2sgZGVmYXVsdHMgdG8gYEVYRUNVVEVfQ09NUEFSSVNPTiA9IEZhbHNlYC4gQSB1c2VyIG11c3QgZXhwbGljaXRseQpjaGFuZ2UgdGhlIGZsYWcgaW4gYSBmdXR1cmUgY29tcHV0ZSBzZXNzaW9uIGFmdGVyIHRoZSBwcmVmbGlnaHQgbWFuaWZlc3QgcGFzc2VzLgoKIyMgUHJvcG9zZWQgY29tbWFuZCBjb250cmFjdAoKVGhlIGV4YWN0IHBhdGggbmFtZXMgYXJlIHJ1bi1sb2NhbCwgYnV0IHRoZSBzZW1hbnRpYyBhcmd2IGlzIGxvY2tlZDoKCmBgYHRleHQKaGFwLnB5IFRSVVRIX1NMSUNFIFFVRVJZX1NMSUNFCiAgLWYgRUZGRUNUSVZFX1BBRERFRF9CRUQKICAtciBSRUZFUkVOQ0VfQ0hSMjBfRkFTVEEKICAtbyBSVU5fT1VUUFVUX1BSRUZJWAogIC1WCiAgLS1lbmdpbmU9dmNmZXZhbAogIC0tZW5naW5lLXZjZmV2YWwtcGF0aCAvb3B0L3J0Zy10b29scy0zLjEyLjEvcnRnCiAgLS1lbmdpbmUtdmNmZXZhbC10ZW1wbGF0ZSBSRUZFUkVOQ0VfQ0hSMjBfU0RGCiAgLS10aHJlYWRzIDIKYGBgCgpObyBxdWFsaXR5IHRocmVzaG9sZCBpcyBhZGRlZC4gU291cmNlIEZJTFRFUiB2YWx1ZXMgYW5kIGNvbXBhcmF0b3Igbm90LWFzc2Vzc2VkCnN0YXRlcyBhcmUgcmV0YWluZWQuIEFueSByZXF1aXJlZCBzeW50YXgtb25seSBhZGp1c3RtZW50IGRpc2NvdmVyZWQgZHVyaW5nCnByZWZsaWdodCBtdXN0IGJlIGRvY3VtZW50ZWQgYXMgYSBsb2NrLWZpbGUgcmV2aXNpb24gYmVmb3JlIGNvbXBhcmlzb24uCgojIyBEb3dubG9hZCBhbmQgY29tcHV0ZSBlc3RpbWF0ZXMKCktub3duIGZ1bGwgZ2Vub21pYyBzb3VyY2Ugb2JqZWN0cyB0b3RhbCBhcHByb3hpbWF0ZWx5IDEuMjI1IEdCIGRlY2ltYWwgYmVmb3JlCnRvb2xzOiAxNjguNDYgTUIgZm9yIHRoZSB0d28gcXVlcnkgVkNGcywgMTU2LjI1IE1CIHRydXRoIFZDRiwgMS42NiBNQiB0cnV0aAppbmRleCwgMTEuNDkgTUIgQkVELCBhbmQgODg3LjEyIE1CIGZvciByZWZlcmVuY2UgcGx1cyBpbmRleGVzLiBUaGUgbG9ja2VkCmhhcC5weSBpbWFnZSBsYXllcnMgYXJlIGFwcHJveGltYXRlbHkgMjMzLjYgTUIgY29tcHJlc3NlZCBhbmQgdGhlIFJURyBaSVAgaXMKNTAuNyBNQiwgeWllbGRpbmcgYSBjb25zZXJ2YXRpdmUgY29sZC1ydW50aW1lIHRyYW5zZmVyIG5lYXIgKioxLjUxIEdCKiouCgpXaXRoIEhUVFAgcmFuZ2UgZXh0cmFjdGlvbiwgZXhwZWN0ZWQgZ2Vub21pYyB0cmFuc2ZlciBpcyBhcHByb3hpbWF0ZWx5CioqMjAw4oCTMjYwIE1CKio6IGJvdGggd2hvbGUgdW5pbmRleGVkIHF1ZXJpZXMgZG9taW5hdGUgKDE2OC40NiBNQiksIHBsdXMgdGhlIEJFRCwKaW5kZXhlcywgb25lIGNocjIwIHJlZmVyZW5jZSBzbGljZSwgYW5kIGEgc21hbGwgdHJ1dGggc2xpY2UuIEluY2x1ZGluZyBjb2xkCnRvb2wgYWNxdWlzaXRpb24gZ2l2ZXMgYW4gZXhwZWN0ZWQgb3B0aW1pemVkIGZpcnN0IHJ1biBvZiByb3VnaGx5CioqNDg14oCTNTUwIE1CKiouIFJhbmdlIGJlaGF2aW9yIGFuZCBjb250YWluZXItcnVubmVyIG92ZXJoZWFkIG1ha2UgdGhpcyBhbgplbmdpbmVlcmluZyBlc3RpbWF0ZSwgbm90IGEgbWVhc3VyZWQgYmVuY2htYXJrLgoKRXhwZWN0ZWQgcmVzb3VyY2VzIGZvciB0d28gMSBNYiBwYWRkZWQgY29tcGFyaXNvbnMgYXJlIDLigJM0IHZDUFVzLCBsZXNzIHRoYW4KOCBHQiBSQU0sIGFwcHJveGltYXRlbHkgNiBHQiB0ZW1wb3JhcnkgZGlzayAodG8gYWxsb3cgY29udGFpbmVyIGV4cGFuc2lvbiwKcmVmZXJlbmNlL1NERiwgc291cmNlIFZDRnMsIHNsaWNlcywgYW5kIG91dHB1dHMpLCBhbmQgMTDigJMzMCBtaW51dGVzIG9mIGNvbXBhcmlzb24KdGltZSBhZnRlciBzZXR1cC4gQWxsb3cgMzDigJM2MCBtaW51dGVzIGVuZCB0byBlbmQgb24gYSBmcmVlIENvbGFiIENQVSBydW50aW1lLgpUaGVzZSBlc3RpbWF0ZXMgbXVzdCBiZSByZXBsYWNlZCBieSBtZWFzdXJlZCB2YWx1ZXMgYWZ0ZXIgdGhlIHByb29mLgo=",
  "docs/phase1a_validation.md": "IyBMb2NrZWQgUGhhc2UgMUEgUEFTUy9GQUlMIGNyaXRlcmlhCgpUaGVzZSBjcml0ZXJpYSB3ZXJlIGxvY2tlZCBiZWZvcmUgZ2Vub21pYyBpbnB1dHMgd2VyZSBkb3dubG9hZGVkIG9yIGJlbmNobWFyawpyZXN1bHRzIHdlcmUgaW5zcGVjdGVkLiBQcmVjaXNpb24sIHJlY2FsbCwgRjEsIFRQIGNvdW50LCBGUCBjb3VudCwgRk4gY291bnQsCmFuZCBhZ3JlZW1lbnQgYmV0d2VlbiB0ZWNobm9sb2dpZXMgYXJlIG5vdCBzdWNjZXNzIHRocmVzaG9sZHMuCgpQaGFzZSAxQSBpcyAqKlBBU1MqKiBvbmx5IGlmIGV2ZXJ5IGNyaXRlcmlvbiBiZWxvdyBwYXNzZXMgZm9yIGJvdGggc2VsZWN0ZWQKc3VibWlzc2lvbnMuCgojIyBJbnB1dCBhbmQgcHJvdmVuYW5jZQoKMS4gVGhlIHF1ZXJ5IGFydGlmYWN0IFVSTCwgYnl0ZSBzaXplLCBhbmQgcHVibGlzaGVkIFNIQS0yNTYgbWF0Y2ggdGhlIG1hbmlmZXN0LgoyLiBUcnV0aCwgVEJJLCBCRUQsIHJlZmVyZW5jZSwgRkFJLCBhbmQgR1pJIG5hbWVzL1VSTHMgbWF0Y2ggdGhlIG1hbmlmZXN0OyB0aGVpcgogICBkb3dubG9hZGVkIGJ5dGUgc2l6ZXMgbWF0Y2ggdGhlIHB1YmxpYyBsaXN0aW5nIGFuZCBsb2NhbGx5IGNvbXB1dGVkIFNIQS0yNTYKICAgdmFsdWVzIGFyZSByZWNvcmRlZC4gUHVibGlzaGVkIHJlZmVyZW5jZSBNRDUgdmFsdWVzIGFsc28gbWF0Y2guCjMuIEV2ZXJ5IGRlcml2ZWQgc2xpY2UsIGluZGV4LCBTREYsIGFubm90YXRlZCBWQ0YsIGxvZywgUGFycXVldCBmaWxlLCBhbmQgRHVja0RCCiAgIGZpbGUgaXMgYSBzZXBhcmF0ZSBhcnRpZmFjdCB3aXRoIGEgU0hBLTI1NiBhbmQgYSBwcm92ZW5hbmNlIGVkZ2UuCjQuIEV4cGVyaW1lbnQgY2xhaW1zIHJlc29sdmUgdG8gdGhlIHBpbm5lZCBtZXRhZGF0YSBhcnRpZmFjdHMuIFVua25vd24gY2FsbGVyCiAgIHZlcnNpb25zIHJlbWFpbiB0aGUgbGl0ZXJhbCBgVU5LTk9XTmAuCjUuIEFsbCBvYnNlcnZhdGlvbnMgYXJlIG1hcmtlZCBgUkVHRU5FUkFURURgOyBub25lIGlzIGxhYmVsZWQgYXMgYW4gb3JpZ2luYWwKICAgcHJlY2lzaW9uRkRBIGV2ZW50IGRlY2lzaW9uLgoKIyMgTG9ja2VkIHJ1biBjb250cmFjdAoKNi4gQ29yZS9wYWRkZWQgaW50ZXJ2YWxzLCB0cnV0aCwgQkVELCByZWZlcmVuY2UsIGNvbXBhcmF0b3IgdmVyc2lvbnMsIGVuZ2luZSwKICAgYXJndW1lbnRzLCBhbmQgdGhyZWFkIGNvdW50IGVxdWFsIHRoZSBjaGVja2VkLWluIGxvY2suIE5vIGludGVydmFsIGlzIGNob3NlbgogICBvciBhbHRlcmVkIGFmdGVyIHZpZXdpbmcgb3V0Y29tZXMuCjcuIFRoZSBjb3JlIGludGVydmFsIGludGVyc2VjdHMgYXQgbGVhc3Qgb25lIGJhc2Ugb2YgdGhlIHY0LjIuMSBiZW5jaG1hcmsgQkVELgogICBaZXJvIGFzc2Vzc2FibGUgYmFzZXMgaXMgRkFJTCwgbm90IHBlcm1pc3Npb24gdG8gc3Vic3RpdHV0ZSBhbiBpbnRlcnZhbC4KOC4gUnVudGltZS1yZXBvcnRlZCBoYXAucHkgaXMgMC4zLjE1IGFuZCBSVEcgaXMgMy4xMi4xOyB0aGUgY29udGFpbmVyL2Rlcml2ZWQKICAgaW1hZ2UgY29udGVudCBkaWdlc3QgYW5kIGFsbCBhdXhpbGlhcnkgdG9vbCB2ZXJzaW9ucyBhcmUgcmVjb3JkZWQuCjkuIEVhY2ggaGFwLnB5IHByb2Nlc3MgZXhpdHMgemVybyBhbmQgcHJvZHVjZXMgYSByZWFkYWJsZSBpbmRleGVkIGFubm90YXRlZCBWQ0YsCiAgIG1ldHJpY3Mgb3V0cHV0LCBzdGRvdXQsIGFuZCBzdGRlcnIuIEVtcHR5LCB0cnVuY2F0ZWQsIG9yIHNpbGVudGx5IHNraXBwZWQKICAgY29tcGFyaXNvbiBvdXRwdXQgaXMgRkFJTC4KCiMjIEV2ZW50IHNlbWFudGljcyBhbmQgbm9ybWFsaXphdGlvbgoKMTAuIFRoZSBhbm5vdGF0ZWQgVkNGIGV4cG9zZXMgc2VwYXJhdGUgYFRSVVRIYCBhbmQgYFFVRVJZYCBzYW1wbGVzIGFuZCB0aGUKICAgIGV4cGVjdGVkIEdBNEdIIGZpZWxkcyBuZWVkZWQgZm9yIGRlY2lzaW9ucyBhbmQgZ3JvdXBpbmcgKGBCRGAsIGBCS2AsIGFuZAogICAgYEJTYCwgYWxsb3dpbmcgYSBkb2N1bWVudGVkIHNvdXJjZS1yZWNvcmQgZmFsbGJhY2sgb25seSB3aGVyZSBgQlNgIGlzCiAgICBhYnNlbnQpLgoxMS4gRXZlcnkgYW5ub3RhdGVkIHNvdXJjZSByZWNvcmQgaW4gdGhlIHBhZGRlZCBvdXRwdXQgaXMgZWl0aGVyIG5vcm1hbGl6ZWQgb3IKICAgIHJldGFpbmVkIGFzIGFuIGV4cGxpY2l0IHJlamVjdGVkL2JvdW5kYXJ5IGRpYWdub3N0aWM7IG5vIHJlY29yZCBkaXNhcHBlYXJzCiAgICBzaWxlbnRseS4KMTIuIFJhdyB0cnV0aC9xdWVyeSBkZWNpc2lvbiwgbWF0Y2gta2luZCwgZmlsdGVyLCBnZW5vdHlwZSwgc291cmNlIGNvb3JkaW5hdGVzLAogICAgYW5kIHNvdXJjZSByZWNvcmQgb3JkaW5hbCBhcmUgcmV0YWluZWQuIE5vcm1hbGl6ZWQgdmFsdWVzIGRvIG5vdCBvdmVyd3JpdGUKICAgIHJhdyB2YWx1ZXMuCjEzLiBFYWNoIGV4cG9ydGVkIG9ic2VydmF0aW9uIGlzIHdob2xseSBpbiB0aGUgY29yZSBpbnRlcnZhbCwgcmVzb2x2ZXMgdG8KICAgIGV4YWN0bHkgb25lIHJ1bi1zY29wZWQgZXZlbnQgYW5kIGJlbmNobWFyayBydW4sIGFuZCBoYXMgY29tcGxldGUgbGluZWFnZSB0bwogICAgYW4gYW5ub3RhdGVkIG91dHB1dCBhcnRpZmFjdCBhbmQgYWxsIGZvdXIgc2NpZW50aWZpYyBpbnB1dHMuCjE0LiBUaGVyZSBpcyBhdCBsZWFzdCBvbmUgZXhwb3J0ZWQgZXZlbnQgYW5kIGF0IGxlYXN0IG9uZSBvYnNlcnZhdGlvbiBmb3IgZWFjaAogICAgYmVuY2htYXJrIHJ1bi4gVGhpcyBpcyBhIHBpcGVsaW5lLXZpYWJpbGl0eSByZXF1aXJlbWVudCwgbm90IGEgbWluaW11bQogICAgYWNjdXJhY3kgcmVxdWlyZW1lbnQuCjE1LiBEZWNpc2lvbiBjb3VudHMgZGVyaXZlZCBmcm9tIG5vcm1hbGl6ZWQgb2JzZXJ2YXRpb25zIHJlY29uY2lsZSBleGFjdGx5IHdpdGgKICAgIHRoZSBjb3JyZXNwb25kaW5nIGNvcmUtaW50ZXJ2YWwgYW5ub3RhdGVkIHJlY29yZHMgYnkgc2lkZSBhbmQgcmF3IGRlY2lzaW9uLgoKIyMgU3RvcmFnZSBhbmQgcXVlcnkKCjE2LiBBbGwgc2l4IGVudGl0eSBjb2xsZWN0aW9ucyB2YWxpZGF0ZSBhZ2FpbnN0IHRoZWlyIHZlcnNpb25lZCBKU09OIFNjaGVtYXM7CiAgICBpZGVudGlmaWVycyBhcmUgdW5pcXVlIGFuZCBldmVyeSByZWZlcmVuY2UvcHJvdmVuYW5jZSBlbmRwb2ludCByZXNvbHZlcy4KMTcuIFBhcnF1ZXQgcm91bmQtdHJpcCBwcmVzZXJ2ZXMgcm93IGNvdW50cywgaWRlbnRpZmllcnMsIGludGVnZXJzLCBib29sZWFucywKICAgIGxpc3RzLCBleHBsaWNpdCBgVU5LTk9XTmAgc3RyaW5ncywgYW5kIG51bGwvaW5hcHBsaWNhYmxlIGRpc3RpbmN0aW9ucy4KMTguIER1Y2tEQiB2aWV3cyByZXR1cm4gdGhlIHNhbWUgY291bnRzIGFzIFBhcnF1ZXQgYW5kIHRoZSBldmVudC1sZXZlbCBxdWVyeQogICAgcmV0dXJucywgZm9yIGEgc2VsZWN0ZWQgcnVuLXNjb3BlZCBldmVudCwgYWxsIG9ic2VydmF0aW9ucyBwbHVzIGV4cGVyaW1lbnQsCiAgICBydW4sIGFydGlmYWN0LCBhbmQgcHJvdmVuYW5jZSBjb250ZXh0LgoxOS4gQ2FuZGlkYXRlIGNyb3NzLXJ1biBsb29rdXAgZG9lcyBub3QgbWVyZ2UgcnVuLXNjb3BlZCBldmVudCBJRHMgYW5kIGxhYmVscwogICAgY29vcmRpbmF0ZS9hbGxlbGUgaGl0cyBhcyBjYW5kaWRhdGVzIHJhdGhlciB0aGFuIGVzdGFibGlzaGVkIGVxdWl2YWxlbmNlLgoyMC4gVGhlIGNvbXBsZXRlIHJ1biBtYW5pZmVzdCwgdmFsaWRhdGlvbiByZXBvcnQsIGFuZCBvdXRwdXQgY2hlY2tzdW1zIGFyZQogICAgd3JpdHRlbiBiZWZvcmUgdGhlIENvbGFiIHNlc3Npb24gZW5kcy4KCkFueSBmYWlsZWQgaXRlbSBtYWtlcyB0aGUgb3ZlcmFsbCByZXN1bHQgKipGQUlMKiouIEEgZmFpbGVkIHJ1biBtYXkgYmUgcmV0YWluZWQKYXMgYW4gYXVkaXRhYmxlIGJlbmNobWFyay1ydW4gcmVjb3JkLCBidXQgaXRzIG9ic2VydmF0aW9ucyBjYW5ub3QgZW50ZXIgdGhlClBoYXNlIDFBIGV2aWRlbmNlIGV4cG9ydC4gQ2hhbmdpbmcgYSBsb2NrZWQgaW5wdXQsIGludGVydmFsLCB0b29sLCBvciBzZW1hbnRpYwpydWxlIHJlcXVpcmVzIGEgbmV3IGRvY3VtZW50ZWQgUGhhc2UgMUEgcmV2aXNpb24gYmVmb3JlIHJlcnVubmluZy4KCg==",
  "docs/evidence_semantics.md": "IyBFdmlkZW5jZSBzZW1hbnRpY3MKCiMjIE9ic2VydmF0aW9uLCBub3QgbGFiZWwKCmBUUGAsIGBGUGAsIGBGTmAsIGBOYCwgYFVOS2AsIGFuZCBgSUdOYCBhcmUgb3V0Y29tZXMgb2YgYSBwYXJ0aWN1bGFyCmNvbXBhcmlzb24uIFRoZXkgYXJlIG5vdCBpbnRyaW5zaWMgcHJvcGVydGllcyBvZiBhIGdlbm9taWMgYWxsZWxlLiBUaGUgc2FtZQpzb3VyY2UgcmVjb3JkIGNhbiByZWNlaXZlIGEgZGlmZmVyZW50IG91dGNvbWUgdW5kZXIgYSBkaWZmZXJlbnQgdHJ1dGggcmVsZWFzZSwKYmVuY2htYXJrIEJFRCwgcHJlcHJvY2Vzc2luZyBydWxlLCBjb21wYXJhdG9yLCBjb21wYXJhdG9yIHZlcnNpb24sIG9yIHBhcmFtZXRlcgpzZXQuCgpFYWNoIG5vcm1hbGl6ZWQgb2JzZXJ2YXRpb24gdGhlcmVmb3JlIG5hbWVzIGV4YWN0bHkgb25lIGJlbmNobWFyayBydW4sIG9uZQpydW4tc2NvcGVkIGV2ZW50LCBvbmUgc2lkZSAoYFRSVVRIYCBvciBgUVVFUllgKSwgYW5kIG9uZSBvciBtb3JlIHNvdXJjZSByZWNvcmRzLgpUcnV0aCBhbmQgcXVlcnkgb3V0Y29tZXMgYXJlIHN0b3JlZCBzZXBhcmF0ZWx5LiBBIGdlbm90eXBlIG1pc21hdGNoIG1heSBwcm9kdWNlCnRydXRoLXNpZGUgYEZOYCBhbmQgcXVlcnktc2lkZSBgRlBgIG9ic2VydmF0aW9ucyB3aXRoaW4gdGhlIHNhbWUgc3VwZXJsb2N1cy4KCiMjIEV2ZW50IGlkZW50aXR5CgpGb3IgUGhhc2UgMUEsIGFuIGBFVkVOVGAgaXMgYSBjb21wYXJpc29uLXNjb3BlZCBncm91cGluZywgbm9ybWFsbHkgdGhlIGhhcC5weQpgQlNgIHN1cGVybG9jdXMgZW1pdHRlZCBieSBvbmUgYEJFTkNITUFSS19SVU5gLiBJdCBpcyBzYWZlIHRvIHVzZSBhcyBhIHVuaXQgb2YKcmV0cmlldmFsIHdpdGhpbiB0aGF0IHJ1bi4gSXRzIGlkZW50aWZpZXIgaXMgbmV2ZXIgcmV1c2VkIGFjcm9zcyBydW5zLgoKQSBub3JtYWxpemVkIGFsbGVsZSBrZXkgbWF5IGJlIHN0b3JlZCBhcyBhIGxvb2t1cCBhaWQgZm9yIGEgc2luZ2xlIHNvdXJjZQpyZWNvcmQuIEl0IGRvZXMgbm90IGVzdGFibGlzaCBiaW9sb2dpY2FsIGVxdWl2YWxlbmNlLiBDb29yZGluYXRlIG92ZXJsYXAsCmlkZW50aWNhbCBub3JtYWxpemVkIGFsbGVsZXMsIG9yIGxpZnRlZCBjb29yZGluYXRlcyBtYXkgaWRlbnRpZnkgY2FuZGlkYXRlcwpmb3IgbGF0ZXIgcmV2aWV3LCBidXQgdGhleSBtdXN0IG5vdCBzaWxlbnRseSBtZXJnZSBldmVudHMuIFBoYXNlIDFBIGRlZmluZXMgbm8KY3Jvc3MtcnVuIGNhbm9uaWNhbC1ldmVudCBhc3NlcnRpb24uCgpJZiBoYXAucHkgZW1pdHMgbm8gYEJTYCB2YWx1ZSBmb3IgYSByZWNvcmQsIG5vcm1hbGl6YXRpb24gY3JlYXRlcyBhIGRldGVybWluaXN0aWMKcnVuLXNjb3BlZCBzb3VyY2UtcmVjb3JkIGV2ZW50IGFuZCByZWNvcmRzIHRoYXQgZGVyaXZhdGlvbi4gSXQgbXVzdCBub3QgaW52ZW50IGEKY3Jvc3MtcmVjb3JkIHN1cGVybG9jdXMuCgojIyBSZWZlcmVuY2UgYW5kIGNvb3JkaW5hdGUgcnVsZXMKCi0gQXNzZW1ibHkgaXMgYEdSQ2gzOGAsIGJvdW5kIHRvIHRoZSBleGFjdCByZWZlcmVuY2UgYXJ0aWZhY3QgaW4gdGhlIHJ1bi4KLSBSZWdpc3RyeSBpbnRlcnZhbHMgYXJlIDAtYmFzZWQsIGhhbGYtb3Blbi4KLSBPcmlnaW5hbCBWQ0YgYENIUk9NYCBhbmQgMS1iYXNlZCBgUE9TYCBhcmUgcmV0YWluZWQgdmVyYmF0aW0gaW4gb2JzZXJ2YXRpb24KICBzb3VyY2UgZmllbGRzLgotIE5vcm1hbGl6ZWQgc3RhcnQvZW5kIGZpZWxkcyBhcmUgZGVyaXZlZCBhbmQgbmFtZSB0aGVpciBub3JtYWxpemF0aW9uIG1ldGhvZAogIGFuZCB2ZXJzaW9uLgotIExpZnRvdmVyIGlzIG5vdCBpZGVudGl0eSBhbmQgaXMgbm90IHVzZWQgaW4gUGhhc2UgMUEuCi0gRXZlbnRzIGludGVyc2VjdGluZyB0aGUgcGFkZGVkIGV4dHJhY3Rpb24gYm91bmRhcnkgYnV0IG5vdCB3aG9sbHkgY29udGFpbmVkCiAgaW4gdGhlIGxvY2tlZCBjb3JlIGludGVydmFsIGFyZSByZXRhaW5lZCBhcyBib3VuZGFyeSBkaWFnbm9zdGljcyBhbmQgZXhjbHVkZWQKICBmcm9tIHRoZSBQaGFzZSAxQSBldmlkZW5jZSBleHBvcnQuCgojIyBSZWdpb24gYW5kIGFic2VuY2Ugc2VtYW50aWNzCgpUaGUgZm9sbG93aW5nIHN0YXRlcyBhcmUgZGlzdGluY3Q6CgotIGBJTlNJREVgOiB3aG9sbHkgaW5zaWRlIHRoZSBlZmZlY3RpdmUgYmVuY2htYXJrIHJlZ2lvbjsKLSBgT1VUU0lERWA6IHdob2xseSBvdXRzaWRlIGl0OwotIGBCT1JERVJgOiB0b3VjaGVzIG9yIGNyb3NzZXMgaXRzIGJvdW5kYXJ5OwotIGBVTktOT1dOYDogbWVtYmVyc2hpcCB3YXMgbm90IGNvbXB1dGVkIG9yIGNhbm5vdCBiZSBlc3RhYmxpc2hlZDsKLSBjb21wYXJhdG9yIGRlY2lzaW9uIGBOYCwgYFVOS2AsIG9yIGBJR05gIGFzIGVtaXR0ZWQgYnkgdGhlIHRvb2w7Ci0gYSBmaWx0ZXJlZCBzb3VyY2UgY2FsbDsKLSBubyBzb3VyY2UgcmVjb3JkIGluIGEgc2lkZSBvZiBhIHN1cGVybG9jdXM7IGFuZAotIGFuIGFydGlmYWN0IG9yIGZpZWxkIGFic2VudCBmcm9tIHRoZSBwdWJsaWMgY29ycHVzLgoKQWJzZW5jZSBtdXN0IG5ldmVyIGJlIHJld3JpdHRlbiBhcyBgRk5gLCBgRlBgLCBgTmAsIG9yIGBVTktgIHdpdGhvdXQgY29tcGFyYXRvcgpldmlkZW5jZS4gVGhlIHJhdyBjb21wYXJhdG9yIHZhbHVlIGFuZCB0aGUgbm9ybWFsaXplZCBkZWNpc2lvbiBhcmUgYm90aCBrZXB0LgoKIyMgVW5rbm93bnMKCmBVTktOT1dOYCBtZWFucyB0aGUgdmFsdWUgaXMgbm90IGVzdGFibGlzaGVkIGJ5IGF0dHJpYnV0YWJsZSBldmlkZW5jZS4gSXQgaXMgYQpwb3NpdGl2ZSBzZW1hbnRpYyBzdGF0ZSwgbm90IGEgbnVsbCBzaG9ydGN1dC4gTnVsbCBpcyByZXNlcnZlZCBmb3IgYSBmaWVsZCB0aGF0CmlzIHN0cnVjdHVyYWxseSBpbmFwcGxpY2FibGUuIE5vdGFibHksIHRoZSBleGFjdCBOZXVTb21hdGljIHZlcnNpb24gZm9yIGA2MFo1OWAKYW5kIHRoZSBQRVBQRVIvRGVlcFZhcmlhbnQgdmVyc2lvbnMgZm9yIGBSVTg4TmAgcmVtYWluIGBVTktOT1dOYC4KCiMjIFJlZ2VuZXJhdGlvbiBhbmQgcHJvdmVuYW5jZQoKQWxsIFBoYXNlIDFBIG9ic2VydmF0aW9ucyBoYXZlIGBvYnNlcnZhdGlvbl9vcmlnaW4gPSBSRUdFTkVSQVRFRGAuIFByb3ZlbmFuY2UKbXVzdCBjb25uZWN0IGVhY2ggb2JzZXJ2YXRpb24gdG86CgoxLiBpdHMgYW5ub3RhdGVkIGNvbXBhcmF0b3Igb3V0cHV0IHJlY29yZDsKMi4gdGhlIGJlbmNobWFyayBydW4gYW5kIGV4YWN0IGNvbW1hbmQ7CjMuIHRoZSBzZWxlY3RlZCBxdWVyeSwgdHJ1dGgsIEJFRCwgYW5kIHJlZmVyZW5jZSBhcnRpZmFjdHM7CjQuIHRoZSBsb2NrZWQgY29tcGFyaXNvbiBlbnZpcm9ubWVudDsKNS4gbm9ybWFsaXphdGlvbiBzb2Z0d2FyZS92ZXJzaW9uIGFuZCB0cmFuc2Zvcm1hdGlvbiB0aW1lOyBhbmQKNi4gdGhlIHNvdXJjZSBtZXRhZGF0YSB0aGF0IHN1cHBvcnRzIGV4cGVyaW1lbnQgY2xhaW1zLgoKQ29waWVkLCB0cmFuc2Zvcm1lZCwgcmVnZW5lcmF0ZWQsIGFuZCBpbmZlcnJlZCBhc3NlcnRpb25zIGFyZSBkaXN0aW5ndWlzaGVkLgpObyBvYnNlcnZhdGlvbiBpcyBwdWJsaXNoYWJsZSBpZiBpdHMgcmF3IGFydGlmYWN0IGxpbmVhZ2UgaXMgYnJva2VuLgoKIyMgQ29tcGFyYWJpbGl0eQoKVGhlIHR3byBQaGFzZSAxQSBydW5zIGFyZSBjb21wYXJhYmxlIG9ubHkgYXQgdGhlIGxldmVsIGV4cGxpY2l0bHkgc2hhcmVkIGJ5CnRoZWlyIHJ1biBjb250cmFjdHM6IHNhbXBsZSwgcmVmZXJlbmNlLCB0cnV0aCwgQkVELCBpbnRlcnZhbCwgY29tcGFyYXRvciwKcGFyYW1ldGVycywgYW5kIG5vcm1hbGl6YXRpb24gY29kZS4gUGlwZWxpbmUgYW5kIHNlcXVlbmNpbmcgdGVjaG5vbG9neSBkaWZmZXIuCkV4YWN0IHNvdXJjZS1yZWNvcmQgbWF0Y2hlcyBhY3Jvc3MgcnVucyBtYXkgYmUgZGlzcGxheWVkIHRvZ2V0aGVyIGFzIGNhbmRpZGF0ZQpjb3JyZXNwb25kZW5jZXMsIGJ1dCB0aGUgcnVuLXNjb3BlZCBldmVudCByb3dzIHN0YXkgZGlzdGluY3QuCgo=",
  "docs/conceptual_schema.md": "IyBDb25jZXB0dWFsIHNjaGVtYQoKVGhlIGV4ZWN1dGFibGUgSlNPTiBTY2hlbWFzIGluIGBzY2hlbWFzL2AgYXJlIHRoZSBQaGFzZSAxQSBjb250cmFjdC4gUGFycXVldAp0YWJsZXMgdXNlIHRoZSBzYW1lIHNpeCBlbnRpdGllcyBhbmQgRHVja0RCIGV4cG9zZXMgdGhlbSB3aXRob3V0IGNvbGxhcHNpbmcKdGhlaXIgaWRlbnRpZmllcnMuCgp8IEVudGl0eSB8IEdyYWluIHwgSWRlbnRpdHkgYW5kIGVzc2VudGlhbCBjb250ZW50IHwKfC0tLXwtLS18LS0tfAp8IGBFVkVOVGAgfCBPbmUgcnVuLXNjb3BlZCBjb21wYXJhdG9yIGdyb3VwaW5nIG9yIGZhbGxiYWNrIHNvdXJjZS1yZWNvcmQgZ3JvdXAgfCBgZXZlbnRfaWRgOyBhc3NlbWJseS9yZWZlcmVuY2U7IDAtYmFzZWQgaW50ZXJ2YWw7IGlkZW50aXR5IHNjb3BlOyBiZW5jaG1hcmsgcnVuOyBzb3VyY2UgYEJTYCB3aGVuIHByZXNlbnQ7IGRlcml2YXRpb24gbWV0aG9kOyBvcHRpb25hbCBsb29rdXAga2V5IHRoYXQgaXMgZXhwbGljaXRseSBub24tZXF1aXZhbGVuY2UtYmVhcmluZy4gfAp8IGBFWFBFUklNRU5UYCB8IE9uZSBzdWJtaXR0ZWQgcGlwZWxpbmUgYXBwbGllZCB0byBvbmUgc2FtcGxlIGFuZCBpbnB1dCB0ZWNobm9sb2d5IGNvbmRpdGlvbiB8IGBleHBlcmltZW50X2lkYDsgc291cmNlIGNvcnB1cy9zdWJtaXNzaW9uOyBzYW1wbGU7IHRlY2hub2xvZ3kvcGxhdGZvcm0vY292ZXJhZ2U7IHBpcGVsaW5lLCBjYWxsZXIsIGFsaWduZXIsIGJhc2VjYWxsZXIgYW5kIHZlcnNpb25zOyBtZXRhZGF0YSBldmlkZW5jZSBhcnRpZmFjdHM7IGV4cGxpY2l0IHVua25vd24gZmllbGRzLiB8CnwgYEJFTkNITUFSS19SVU5gIHwgT25lIGltbXV0YWJsZSBjb21wYXJpc29uIGludm9jYXRpb24gfCBgYmVuY2htYXJrX3J1bl9pZGA7IGV4cGVyaW1lbnQ7IGV4YWN0IHF1ZXJ5L3RydXRoL0JFRC9yZWZlcmVuY2UgYXJ0aWZhY3RzOyBjb3JlIGFuZCBwYWRkZWQgaW50ZXJ2YWxzOyBjb21wYXJhdG9yIGFuZCBlbmdpbmUgdmVyc2lvbnM7IGVudmlyb25tZW50IGxvY2svZGlnZXN0OyBhcmd2OyBydW4gc3RhdGU7IG91dHB1dCBhcnRpZmFjdHMgYW5kIHRpbWVzdGFtcHMuIHwKfCBgT0JTRVJWQVRJT05gIHwgT25lIHRydXRoLSBvciBxdWVyeS1zaWRlIHNvdXJjZSByZWNvcmQgZGVjaXNpb24gd2l0aGluIG9uZSBldmVudCBhbmQgcnVuIHwgYG9ic2VydmF0aW9uX2lkYDsgZXZlbnQvcnVuOyBzaWRlOyByYXcgYW5kIG5vcm1hbGl6ZWQgZGVjaXNpb247IG1hdGNoIGtpbmQvZGV0YWlsOyByYXcgVkNGIGxvY3VzL2FsbGVsZXMvZ2Vub3R5cGUvZmlsdGVyOyByZWdpb24gc3RhdGU7IHNvdXJjZSBvdXRwdXQgYXJ0aWZhY3QgYW5kIHJlY29yZCBvcmRpbmFsOyBvcmlnaW4uIE11bHRpcGxlIHJvd3MgcGVyIGV2ZW50IGFuZCBzaWRlIGFyZSBhbGxvd2VkLiB8CnwgYFNPVVJDRV9BUlRJRkFDVGAgfCBPbmUgYnl0ZS1hZGRyZXNzYWJsZSBpbnB1dCwgbWV0YWRhdGEsIHNvZnR3YXJlLCBvciBvdXRwdXQgb2JqZWN0IHwgYHNvdXJjZV9hcnRpZmFjdF9pZGA7IFVSSTsgcm9sZTsgbWVkaWEgdHlwZTsgYnl0ZSBzaXplOyB2ZXJzaW9uOyBoYXNoZXMgYW5kIHZlcmlmaWNhdGlvbiBzdGF0ZTsgcmV0cmlldmFsIHRpbWU7IGxpY2Vuc2UvcmV1c2UgYmFzaXM7IGxvY2FsIGRlcml2YXRpb24gc3RhdGUuIHwKfCBgUFJPVkVOQU5DRV9MSU5LYCB8IE9uZSBkaXJlY3RlZCBsaW5lYWdlIGVkZ2UgYmV0d2VlbiB0eXBlZCByZWdpc3RyeSBlbnRpdGllcyB8IGBwcm92ZW5hbmNlX2xpbmtfaWRgOyB0eXBlZCBzb3VyY2UvdGFyZ2V0IHJlZnM7IHJlbGF0aW9uOyB0cmFuc2Zvcm1hdGlvbi9zb2Z0d2FyZS9wYXJhbWV0ZXJzOyB0aW1lc3RhbXA7IHJlc3BvbnNpYmxlIGFnZW50LiBUaGlzIHJlcHJlc2VudHMgbGluZWFnZSwgbm90IGJpb2xvZ2ljYWwgZXF1aXZhbGVuY2UuIHwKCiMjIENhcmRpbmFsaXRpZXMKCi0gT25lIGBFWFBFUklNRU5UYCBoYXMgbWFueSBgQkVOQ0hNQVJLX1JVTmAgcm93cyBvdmVyIHRpbWUuCi0gT25lIGBCRU5DSE1BUktfUlVOYCB1c2VzIGV4YWN0bHkgb25lIHF1ZXJ5IFZDRiwgdHJ1dGggVkNGLCBiZW5jaG1hcmsgQkVELCBhbmQKICByZWZlcmVuY2UgYXJ0aWZhY3QgaW4gUGhhc2UgMUEuCi0gT25lIGBCRU5DSE1BUktfUlVOYCBoYXMgbWFueSBydW4tc2NvcGVkIGBFVkVOVGAgcm93cy4KLSBPbmUgYEVWRU5UYCBoYXMgb25lIG9yIG1vcmUgYE9CU0VSVkFUSU9OYCByb3dzOyBlYWNoIHNpZGUgbWF5IGhhdmUgemVybywgb25lLAogIG9yIG1hbnkgc291cmNlIHJlY29yZHMuCi0gRXZlcnkgbWF0ZXJpYWwgZW50aXR5IG1heSBoYXZlIG1hbnkgYFBST1ZFTkFOQ0VfTElOS2Agcm93cy4KLSBBbiBhcnRpZmFjdCBtYXkgYmUgdXNlZCBieSBtYW55IHJ1bnMsIGJ1dCBhIHJ1biBuZXZlciByZWZlcnMgdG8gYW4gYXJ0aWZhY3QgYnkKICBtdXRhYmxlIGZpbGVuYW1lIGFsb25lLgoKIyMgU3RvcmFnZSBrZXlzCgpJZGVudGlmaWVycyBhcmUgb3BhcXVlIHN0YWJsZSBzdHJpbmdzLiBUaGV5IGFyZSBub3QgZGVyaXZlZCBzb2xlbHkgZnJvbSBnZW5vbWljCmNvb3JkaW5hdGVzLiBBIGZ1dHVyZSBjb250ZW50LWFkZHJlc3NlZCBpZGVudGlmaWVyIHNjaGVtZSBtYXkgYmUgYWRkZWQgb25seQphZnRlciBpdHMgc2VyaWFsaXphdGlvbiBpcyBzcGVjaWZpZWQgYW5kIHZlcnNpb25lZC4KClBhcnF1ZXQgcGFydGl0aW9ucyBtYXkgdXNlIGBjb3JwdXNgLCBgc2FtcGxlYCwgYW5kIGBiZW5jaG1hcmtfcnVuX2lkYCBmb3IKcGh5c2ljYWwgb3JnYW5pemF0aW9uLCBidXQgdGhvc2UgcGF0aHMgYXJlIG5vdCBzZW1hbnRpYyBpZGVudGl0eS4gRHVja0RCCmZvcmVpZ24ta2V5LWxpa2UgY2hlY2tzIGFyZSBwZXJmb3JtZWQgZHVyaW5nIGluZ2VzdGlvbiBldmVuIHdoZXJlIFBhcnF1ZXQgZG9lcwpub3QgZW5mb3JjZSB0aGVtLgoKIyMgUmVxdWlyZWQgZXZlbnQtbGV2ZWwgcXVlcnkgYmVoYXZpb3IKCkdpdmVuIGEgcnVuLXNjb3BlZCBgZXZlbnRfaWRgLCByZXR1cm4gaXRzIGV2ZW50IGJvdW5kcywgYWxsIHRydXRoL3F1ZXJ5Cm9ic2VydmF0aW9ucywgZXhwZXJpbWVudCBjb25kaXRpb24sIGNvbXBsZXRlIGJlbmNobWFyay1ydW4gc2V0dGluZ3MsIGFuZCBzb3VyY2UKYXJ0aWZhY3QgbGluZWFnZS4gQSBzZXBhcmF0ZSBjYW5kaWRhdGUtc2VhcmNoIHF1ZXJ5IG1heSB1c2UgYXNzZW1ibHksIGNvbnRpZywKaW50ZXJ2YWwgb3ZlcmxhcCwgYW5kIG5vcm1hbGl6ZWQgYWxsZWxlIGxvb2t1cCwgYnV0IGl0cyByZXN1bHRzIG11c3QgcmVtYWluCnNlcGFyYXRlIGV2ZW50cyB1bnRpbCBhbiBleHBsaWNpdCBmdXR1cmUgZXF1aXZhbGVuY2UgYXNzZXJ0aW9uIGV4aXN0cy4KClRoZSBjb25jcmV0ZSBEdWNrREIgcmV0cmlldmFsIHNoYXBlIGlzIGxvY2tlZCBpbiBgc3FsL2V2ZW50X2V2aWRlbmNlLnNxbGAuCg==",
  "sql/event_evidence.sql": "LS0gUGhhc2UgMUEgZXZlbnQtbGV2ZWwgZXZpZGVuY2UgcXVlcnkgY29udHJhY3QuCi0tIEJpbmQgJGV2ZW50X2lkIHRvIG9uZSBydW4tc2NvcGVkIEVWRU5UIGlkZW50aWZpZXIuClNFTEVDVAogICAgZS5ldmVudF9pZCwKICAgIGUuaWRlbnRpdHlfc2NvcGUsCiAgICBlLmFzc2VtYmx5LAogICAgZS5jb250aWcsCiAgICBlLnN0YXJ0XzBiYXNlZCwKICAgIGUuZW5kXzBiYXNlZCwKICAgIGUuc291cmNlX2V2ZW50X2lkLAogICAgby5vYnNlcnZhdGlvbl9pZCwKICAgIG8uc2lkZSwKICAgIG8ucmF3X2RlY2lzaW9uLAogICAgby5ub3JtYWxpemVkX2RlY2lzaW9uLAogICAgby5yYXdfbWF0Y2hfa2luZCwKICAgIG8ucmVnaW9uX3N0YXR1cywKICAgIG8uc291cmNlX2NvbnRpZywKICAgIG8uc291cmNlX3Bvc18xYmFzZWQsCiAgICBvLnNvdXJjZV9yZWYsCiAgICBvLnNvdXJjZV9hbHQsCiAgICBvLnNvdXJjZV9nZW5vdHlwZSwKICAgIG8uc291cmNlX2ZpbHRlciwKICAgIHIuYmVuY2htYXJrX3J1bl9pZCwKICAgIHIub2JzZXJ2YXRpb25fb3JpZ2luLAogICAgci5jb21wYXJhdG9yLAogICAgci5jb21wYXJhdG9yX3ZlcnNpb24sCiAgICByLmVuZ2luZSwKICAgIHIuZW5naW5lX3ZlcnNpb24sCiAgICByLmVudmlyb25tZW50X2xvY2tfaWQsCiAgICByLnJ1bnRpbWVfaW1hZ2VfZGlnZXN0LAogICAgci5xdWVyeV9hcnRpZmFjdF9pZCwKICAgIHIudHJ1dGhfYXJ0aWZhY3RfaWQsCiAgICByLmJlbmNobWFya19iZWRfYXJ0aWZhY3RfaWQsCiAgICByLnJlZmVyZW5jZV9hcnRpZmFjdF9pZCwKICAgIHguZXhwZXJpbWVudF9pZCwKICAgIHguc291cmNlX3N1Ym1pc3Npb25faWQsCiAgICB4LnNhbXBsZV9pZCwKICAgIHgudGVjaG5vbG9neSwKICAgIHgucGxhdGZvcm0sCiAgICB4LmNvdmVyYWdlLAogICAgeC5waXBlbGluZV9uYW1lLAogICAgeC5jYWxsZXJfbmFtZXMsCiAgICB4LmNhbGxlcl92ZXJzaW9ucywKICAgIHAucHJvdmVuYW5jZV9saW5rX2lkIEFTIHJhd19yZWNvcmRfcHJvdmVuYW5jZV9saW5rX2lkLAogICAgYS5zb3VyY2VfYXJ0aWZhY3RfaWQgQVMgYW5ub3RhdGVkX291dHB1dF9hcnRpZmFjdF9pZCwKICAgIGEudXJpIEFTIGFubm90YXRlZF9vdXRwdXRfdXJpLAogICAgYS5jaGVja3N1bXMgQVMgYW5ub3RhdGVkX291dHB1dF9jaGVja3N1bXMKRlJPTSBldmVudHMgQVMgZQpKT0lOIG9ic2VydmF0aW9ucyBBUyBvCiAgT04gby5ldmVudF9pZCA9IGUuZXZlbnRfaWQKIEFORCBvLmJlbmNobWFya19ydW5faWQgPSBlLmJlbmNobWFya19ydW5faWQKSk9JTiBiZW5jaG1hcmtfcnVucyBBUyByCiAgT04gci5iZW5jaG1hcmtfcnVuX2lkID0gZS5iZW5jaG1hcmtfcnVuX2lkCkpPSU4gZXhwZXJpbWVudHMgQVMgeAogIE9OIHguZXhwZXJpbWVudF9pZCA9IHIuZXhwZXJpbWVudF9pZApKT0lOIHByb3ZlbmFuY2VfbGlua3MgQVMgcAogIE9OIHAudGFyZ2V0X2VudGl0eV90eXBlID0gJ09CU0VSVkFUSU9OJwogQU5EIHAudGFyZ2V0X2VudGl0eV9pZCA9IG8ub2JzZXJ2YXRpb25faWQKIEFORCBwLnNvdXJjZV9lbnRpdHlfdHlwZSA9ICdTT1VSQ0VfQVJUSUZBQ1QnCiBBTkQgcC5yZWxhdGlvbiA9ICdOT1JNQUxJWkVEX0ZST00nCkpPSU4gc291cmNlX2FydGlmYWN0cyBBUyBhCiAgT04gYS5zb3VyY2VfYXJ0aWZhY3RfaWQgPSBwLnNvdXJjZV9lbnRpdHlfaWQKV0hFUkUgZS5ldmVudF9pZCA9ICRldmVudF9pZApPUkRFUiBCWSBvLnNpZGUsIG8uc291cmNlX3JlY29yZF9vcmRpbmFsLCBvLm9ic2VydmF0aW9uX2lkOwoK",
  "schemas/benchmark_run.schema.json": "ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL2JlbmNobWFya19ydW4uc2NoZW1hLmpzb24iLAogICJ0aXRsZSI6ICJCRU5DSE1BUktfUlVOIiwKICAiZGVzY3JpcHRpb24iOiAiT25lIGltbXV0YWJsZSBjb21wYXJhdG9yIGludm9jYXRpb24gYW5kIGl0cyBleGFjdCBzY2llbnRpZmljL3J1bnRpbWUgY29udGV4dC4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWwogICAgInNjaGVtYV92ZXJzaW9uIiwKICAgICJiZW5jaG1hcmtfcnVuX2lkIiwKICAgICJleHBlcmltZW50X2lkIiwKICAgICJvYnNlcnZhdGlvbl9vcmlnaW4iLAogICAgInF1ZXJ5X2FydGlmYWN0X2lkIiwKICAgICJ0cnV0aF9hcnRpZmFjdF9pZCIsCiAgICAiYmVuY2htYXJrX2JlZF9hcnRpZmFjdF9pZCIsCiAgICAicmVmZXJlbmNlX2FydGlmYWN0X2lkIiwKICAgICJjb3JlX2ludGVydmFsIiwKICAgICJwYWRkZWRfaW50ZXJ2YWwiLAogICAgImNvbXBhcmF0b3IiLAogICAgImNvbXBhcmF0b3JfdmVyc2lvbiIsCiAgICAiZW5naW5lIiwKICAgICJlbmdpbmVfdmVyc2lvbiIsCiAgICAiZW52aXJvbm1lbnRfbG9ja19pZCIsCiAgICAicnVuX3N0YXR1cyIsCiAgICAiY29tbWFuZF9hcmd2IiwKICAgICJvdXRwdXRfYXJ0aWZhY3RfaWRzIgogIF0sCiAgInByb3BlcnRpZXMiOiB7CiAgICAic2NoZW1hX3ZlcnNpb24iOiB7ImNvbnN0IjogIjEuMC4wIn0sCiAgICAiYmVuY2htYXJrX3J1bl9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiZXhwZXJpbWVudF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAib2JzZXJ2YXRpb25fb3JpZ2luIjogeyJjb25zdCI6ICJSRUdFTkVSQVRFRCJ9LAogICAgInF1ZXJ5X2FydGlmYWN0X2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJ0cnV0aF9hcnRpZmFjdF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiYmVuY2htYXJrX2JlZF9hcnRpZmFjdF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAicmVmZXJlbmNlX2FydGlmYWN0X2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJjb3JlX2ludGVydmFsIjogeyIkcmVmIjogIiMvJGRlZnMvaW50ZXJ2YWwifSwKICAgICJwYWRkZWRfaW50ZXJ2YWwiOiB7IiRyZWYiOiAiIy8kZGVmcy9pbnRlcnZhbCJ9LAogICAgImNvbXBhcmF0b3IiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgImNvbXBhcmF0b3JfdmVyc2lvbiI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiZW5naW5lIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJlbmdpbmVfdmVyc2lvbiI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiZW52aXJvbm1lbnRfbG9ja19pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAicnVudGltZV9pbWFnZV9kaWdlc3QiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl0sICJwYXR0ZXJuIjogIl5zaGEyNTY6WzAtOWEtZl17NjR9JCJ9LAogICAgImNvbW1hbmRfYXJndiI6IHsidHlwZSI6ICJhcnJheSIsICJpdGVtcyI6IHsidHlwZSI6ICJzdHJpbmcifX0sCiAgICAicGFyYW1ldGVycyI6IHsidHlwZSI6ICJvYmplY3QifSwKICAgICJydW5fc3RhdHVzIjogeyJlbnVtIjogWyJQTEFOTkVEIiwgIlJVTk5JTkciLCAiU1VDQ0VFREVEIiwgIkZBSUxFRCJdfSwKICAgICJzdGFydGVkX2F0IjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdLCAiZm9ybWF0IjogImRhdGUtdGltZSJ9LAogICAgImNvbXBsZXRlZF9hdCI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXSwgImZvcm1hdCI6ICJkYXRlLXRpbWUifSwKICAgICJvdXRwdXRfYXJ0aWZhY3RfaWRzIjogewogICAgICAidHlwZSI6ICJhcnJheSIsCiAgICAgICJ1bmlxdWVJdGVtcyI6IHRydWUsCiAgICAgICJpdGVtcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0KICAgIH0sCiAgICAiZmFpbHVyZV9yZWFzb24iOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgIm5vdGVzIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdfQogIH0sCiAgIiRkZWZzIjogewogICAgImludGVydmFsIjogewogICAgICAidHlwZSI6ICJvYmplY3QiLAogICAgICAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBmYWxzZSwKICAgICAgInJlcXVpcmVkIjogWyJhc3NlbWJseSIsICJjb250aWciLCAic3RhcnRfMGJhc2VkIiwgImVuZF8wYmFzZWQiXSwKICAgICAgInByb3BlcnRpZXMiOiB7CiAgICAgICAgImFzc2VtYmx5IjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICAgICAiY29udGlnIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICAgICAic3RhcnRfMGJhc2VkIjogeyJ0eXBlIjogImludGVnZXIiLCAibWluaW11bSI6IDB9LAogICAgICAgICJlbmRfMGJhc2VkIjogeyJ0eXBlIjogImludGVnZXIiLCAibWluaW11bSI6IDF9CiAgICAgIH0KICAgIH0KICB9LAogICJhbGxPZiI6IFsKICAgIHsKICAgICAgImlmIjogeyJwcm9wZXJ0aWVzIjogeyJydW5fc3RhdHVzIjogeyJjb25zdCI6ICJTVUNDRUVERUQifX19LAogICAgICAidGhlbiI6IHsKICAgICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAgICJjb21tYW5kX2FyZ3YiOiB7Im1pbkl0ZW1zIjogMX0sCiAgICAgICAgICAib3V0cHV0X2FydGlmYWN0X2lkcyI6IHsibWluSXRlbXMiOiAxfSwKICAgICAgICAgICJydW50aW1lX2ltYWdlX2RpZ2VzdCI6IHsidHlwZSI6ICJzdHJpbmcifSwKICAgICAgICAgICJzdGFydGVkX2F0IjogeyJ0eXBlIjogInN0cmluZyJ9LAogICAgICAgICAgImNvbXBsZXRlZF9hdCI6IHsidHlwZSI6ICJzdHJpbmcifQogICAgICAgIH0KICAgICAgfQogICAgfQogIF0KfQoK",
  "schemas/event.schema.json": "ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL2V2ZW50LnNjaGVtYS5qc29uIiwKICAidGl0bGUiOiAiRVZFTlQiLAogICJkZXNjcmlwdGlvbiI6ICJBIHJ1bi1zY29wZWQgY29tcGFyaXNvbiBldmVudC4gSXQgaXMgbm90IGEgdW5pdmVyc2FsIGJpb2xvZ2ljYWwtdmFyaWFudCBpZGVudGl0eS4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWwogICAgInNjaGVtYV92ZXJzaW9uIiwKICAgICJldmVudF9pZCIsCiAgICAiaWRlbnRpdHlfc2NvcGUiLAogICAgImJlbmNobWFya19ydW5faWQiLAogICAgInJlZmVyZW5jZV9hcnRpZmFjdF9pZCIsCiAgICAiYXNzZW1ibHkiLAogICAgImNvbnRpZyIsCiAgICAic3RhcnRfMGJhc2VkIiwKICAgICJlbmRfMGJhc2VkIiwKICAgICJkZXJpdmF0aW9uX21ldGhvZCIsCiAgICAiY3Jvc3NfcnVuX2VxdWl2YWxlbmNlX2Fzc2VydGVkIgogIF0sCiAgInByb3BlcnRpZXMiOiB7CiAgICAic2NoZW1hX3ZlcnNpb24iOiB7ImNvbnN0IjogIjEuMC4wIn0sCiAgICAiZXZlbnRfaWQiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgImlkZW50aXR5X3Njb3BlIjogewogICAgICAiZW51bSI6IFsiQ09NUEFSQVRPUl9TVVBFUkxPQ1VTIiwgIlNPVVJDRV9SRUNPUkRfRkFMTEJBQ0siLCAiTk9STUFMSVpFRF9BTExFTEVfTE9PS1VQIl0KICAgIH0sCiAgICAiYmVuY2htYXJrX3J1bl9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAicmVmZXJlbmNlX2FydGlmYWN0X2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJhc3NlbWJseSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiY29udGlnIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzdGFydF8wYmFzZWQiOiB7InR5cGUiOiAiaW50ZWdlciIsICJtaW5pbXVtIjogMH0sCiAgICAiZW5kXzBiYXNlZCI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAxfSwKICAgICJzb3VyY2VfZXZlbnRfaWQiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgIm5vcm1hbGl6ZWRfYWxsZWxlX2xvb2t1cF9rZXkiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgImRlcml2YXRpb25fbWV0aG9kIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJkZXJpdmF0aW9uX3ZlcnNpb24iOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgImNyb3NzX3J1bl9lcXVpdmFsZW5jZV9hc3NlcnRlZCI6IHsiY29uc3QiOiBmYWxzZX0sCiAgICAibm90ZXMiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19CiAgfSwKICAiYWxsT2YiOiBbCiAgICB7CiAgICAgICJpZiI6IHsicHJvcGVydGllcyI6IHsiaWRlbnRpdHlfc2NvcGUiOiB7ImNvbnN0IjogIkNPTVBBUkFUT1JfU1VQRVJMT0NVUyJ9fX0sCiAgICAgICJ0aGVuIjogewogICAgICAgICJyZXF1aXJlZCI6IFsic291cmNlX2V2ZW50X2lkIl0sCiAgICAgICAgInByb3BlcnRpZXMiOiB7InNvdXJjZV9ldmVudF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX19CiAgICAgIH0KICAgIH0KICBdCn0K",
  "schemas/experiment.schema.json": "ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL2V4cGVyaW1lbnQuc2NoZW1hLmpzb24iLAogICJ0aXRsZSI6ICJFWFBFUklNRU5UIiwKICAiZGVzY3JpcHRpb24iOiAiT25lIHN1Ym1pdHRlZCBwaXBlbGluZSBhcHBsaWVkIHRvIG9uZSBzYW1wbGUgYW5kIHNlcXVlbmNpbmcgY29uZGl0aW9uLiIsCiAgInR5cGUiOiAib2JqZWN0IiwKICAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBmYWxzZSwKICAicmVxdWlyZWQiOiBbCiAgICAic2NoZW1hX3ZlcnNpb24iLAogICAgImV4cGVyaW1lbnRfaWQiLAogICAgInNvdXJjZV9jb3JwdXMiLAogICAgInNvdXJjZV9zdWJtaXNzaW9uX2lkIiwKICAgICJzYW1wbGVfaWQiLAogICAgInRlY2hub2xvZ3kiLAogICAgInBsYXRmb3JtIiwKICAgICJjb3ZlcmFnZSIsCiAgICAicGlwZWxpbmVfbmFtZSIsCiAgICAiY2FsbGVyX25hbWVzIiwKICAgICJjYWxsZXJfdmVyc2lvbnMiLAogICAgInJlZmVyZW5jZV9hc3NlbWJseSIsCiAgICAibWV0YWRhdGFfYXJ0aWZhY3RfaWRzIgogIF0sCiAgInByb3BlcnRpZXMiOiB7CiAgICAic2NoZW1hX3ZlcnNpb24iOiB7ImNvbnN0IjogIjEuMC4wIn0sCiAgICAiZXhwZXJpbWVudF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAic291cmNlX2NvcnB1cyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAic291cmNlX3N1Ym1pc3Npb25faWQiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgInNhbXBsZV9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAidGVjaG5vbG9neSI6IHsiZW51bSI6IFsiSUxMVU1JTkEiLCAiT05UIiwgIlBBQ0JJT19ISUZJIiwgIk1VTFRJIiwgIlVOS05PV04iXX0sCiAgICAicGxhdGZvcm0iOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgImNvdmVyYWdlIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJsaWJyYXJ5X3ByZXBhcmF0aW9uIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxLCAiZGVmYXVsdCI6ICJVTktOT1dOIn0sCiAgICAiYmFzZWNhbGxlciI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMSwgImRlZmF1bHQiOiAiVU5LTk9XTiJ9LAogICAgImJhc2VjYWxsZXJfdmVyc2lvbiI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMSwgImRlZmF1bHQiOiAiVU5LTk9XTiJ9LAogICAgImFsaWduZXJfbmFtZXMiOiB7InR5cGUiOiAiYXJyYXkiLCAiaXRlbXMiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LCAidW5pcXVlSXRlbXMiOiB0cnVlfSwKICAgICJhbGlnbmVyX3ZlcnNpb25zIjogeyJ0eXBlIjogImFycmF5IiwgIml0ZW1zIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfX0sCiAgICAicGlwZWxpbmVfbmFtZSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAicGlwZWxpbmVfdmVyc2lvbiI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMSwgImRlZmF1bHQiOiAiVU5LTk9XTiJ9LAogICAgImNhbGxlcl9uYW1lcyI6IHsidHlwZSI6ICJhcnJheSIsICJtaW5JdGVtcyI6IDEsICJpdGVtcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX19LAogICAgImNhbGxlcl92ZXJzaW9ucyI6IHsidHlwZSI6ICJhcnJheSIsICJtaW5JdGVtcyI6IDEsICJpdGVtcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX19LAogICAgInJlZmVyZW5jZV9hc3NlbWJseSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAibWV0YWRhdGFfYXJ0aWZhY3RfaWRzIjogewogICAgICAidHlwZSI6ICJhcnJheSIsCiAgICAgICJtaW5JdGVtcyI6IDEsCiAgICAgICJ1bmlxdWVJdGVtcyI6IHRydWUsCiAgICAgICJpdGVtcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0KICAgIH0sCiAgICAic2VsZWN0aW9uX2Jhc2lzIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJwZXJmb3JtYW5jZV9jb25zdWx0ZWRfZm9yX3NlbGVjdGlvbiI6IHsiY29uc3QiOiBmYWxzZX0sCiAgICAibm90ZXMiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19CiAgfQp9Cgo=",
  "schemas/observation.schema.json": "ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL29ic2VydmF0aW9uLnNjaGVtYS5qc29uIiwKICAidGl0bGUiOiAiT0JTRVJWQVRJT04iLAogICJkZXNjcmlwdGlvbiI6ICJPbmUgc2lkZS1zcGVjaWZpYyBzb3VyY2UtcmVjb3JkIGRlY2lzaW9uIHdpdGhpbiBvbmUgcnVuLXNjb3BlZCBldmVudC4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWwogICAgInNjaGVtYV92ZXJzaW9uIiwKICAgICJvYnNlcnZhdGlvbl9pZCIsCiAgICAiYmVuY2htYXJrX3J1bl9pZCIsCiAgICAiZXZlbnRfaWQiLAogICAgInNpZGUiLAogICAgIm9ic2VydmF0aW9uX29yaWdpbiIsCiAgICAicmF3X2RlY2lzaW9uIiwKICAgICJub3JtYWxpemVkX2RlY2lzaW9uIiwKICAgICJyYXdfbWF0Y2hfa2luZCIsCiAgICAicmVnaW9uX3N0YXR1cyIsCiAgICAic291cmNlX291dHB1dF9hcnRpZmFjdF9pZCIsCiAgICAic291cmNlX3JlY29yZF9vcmRpbmFsIiwKICAgICJzb3VyY2VfY29udGlnIiwKICAgICJzb3VyY2VfcG9zXzFiYXNlZCIsCiAgICAic291cmNlX3JlZiIsCiAgICAic291cmNlX2FsdCIsCiAgICAic291cmNlX2ZpbHRlciIsCiAgICAibm9ybWFsaXphdGlvbl9tZXRob2QiLAogICAgIm5vcm1hbGl6YXRpb25fdmVyc2lvbiIKICBdLAogICJwcm9wZXJ0aWVzIjogewogICAgInNjaGVtYV92ZXJzaW9uIjogeyJjb25zdCI6ICIxLjAuMCJ9LAogICAgIm9ic2VydmF0aW9uX2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJiZW5jaG1hcmtfcnVuX2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJldmVudF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAic2lkZSI6IHsiZW51bSI6IFsiVFJVVEgiLCAiUVVFUlkiXX0sCiAgICAib2JzZXJ2YXRpb25fb3JpZ2luIjogeyJjb25zdCI6ICJSRUdFTkVSQVRFRCJ9LAogICAgInJhd19kZWNpc2lvbiI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAibm9ybWFsaXplZF9kZWNpc2lvbiI6IHsiZW51bSI6IFsiVFAiLCAiRlAiLCAiRk4iLCAiTiIsICJVTksiLCAiSUdOIiwgIk9USEVSIl19LAogICAgInJhd19tYXRjaF9raW5kIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJyYXdfZW5naW5lX2RldGFpbCI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0sCiAgICAicmF3X3ZhcmlhbnRfdHlwZSI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0sCiAgICAicmF3X2xvY2F0aW9uX3R5cGUiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgInJlZ2lvbl9zdGF0dXMiOiB7ImVudW0iOiBbIklOU0lERSIsICJPVVRTSURFIiwgIkJPUkRFUiIsICJVTktOT1dOIl19LAogICAgInNvdXJjZV9vdXRwdXRfYXJ0aWZhY3RfaWQiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgInNvdXJjZV9yZWNvcmRfb3JkaW5hbCI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAxfSwKICAgICJzb3VyY2VfY29udGlnIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzb3VyY2VfcG9zXzFiYXNlZCI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAxfSwKICAgICJzb3VyY2VfcmVmIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzb3VyY2VfYWx0IjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzb3VyY2VfZ2Vub3R5cGUiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgInNvdXJjZV9maWx0ZXIiOiB7InR5cGUiOiAiYXJyYXkiLCAiaXRlbXMiOiB7InR5cGUiOiAic3RyaW5nIn19LAogICAgInF1YWxpdHlfc2NvcmUiOiB7InR5cGUiOiBbIm51bWJlciIsICJudWxsIl19LAogICAgIm5vcm1hbGl6YXRpb25fbWV0aG9kIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJub3JtYWxpemF0aW9uX3ZlcnNpb24iOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgIm5vcm1hbGl6ZWRfc3RhcnRfMGJhc2VkIjogeyJ0eXBlIjogWyJpbnRlZ2VyIiwgIm51bGwiXSwgIm1pbmltdW0iOiAwfSwKICAgICJub3JtYWxpemVkX2VuZF8wYmFzZWQiOiB7InR5cGUiOiBbImludGVnZXIiLCAibnVsbCJdLCAibWluaW11bSI6IDF9LAogICAgIm5vcm1hbGl6ZWRfcmVmIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdfSwKICAgICJub3JtYWxpemVkX2FsdCI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0sCiAgICAibm90ZXMiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19CiAgfQp9Cgo=",
  "schemas/provenance_link.schema.json": "ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL3Byb3ZlbmFuY2VfbGluay5zY2hlbWEuanNvbiIsCiAgInRpdGxlIjogIlBST1ZFTkFOQ0VfTElOSyIsCiAgImRlc2NyaXB0aW9uIjogIkEgZGlyZWN0ZWQgbGluZWFnZSBlZGdlLiBUaGlzIHNjaGVtYSBkb2VzIG5vdCBlbmNvZGUgYmlvbG9naWNhbCBlcXVpdmFsZW5jZS4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWwogICAgInNjaGVtYV92ZXJzaW9uIiwKICAgICJwcm92ZW5hbmNlX2xpbmtfaWQiLAogICAgInNvdXJjZV9lbnRpdHlfdHlwZSIsCiAgICAic291cmNlX2VudGl0eV9pZCIsCiAgICAidGFyZ2V0X2VudGl0eV90eXBlIiwKICAgICJ0YXJnZXRfZW50aXR5X2lkIiwKICAgICJyZWxhdGlvbiIsCiAgICAiYXNzZXJ0ZWRfYXQiLAogICAgImFzc2VydGVkX2J5IiwKICAgICJ0cmFuc2Zvcm1hdGlvbiIKICBdLAogICJwcm9wZXJ0aWVzIjogewogICAgInNjaGVtYV92ZXJzaW9uIjogeyJjb25zdCI6ICIxLjAuMCJ9LAogICAgInByb3ZlbmFuY2VfbGlua19pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAic291cmNlX2VudGl0eV90eXBlIjogeyIkcmVmIjogIiMvJGRlZnMvZW50aXR5X3R5cGUifSwKICAgICJzb3VyY2VfZW50aXR5X2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJ0YXJnZXRfZW50aXR5X3R5cGUiOiB7IiRyZWYiOiAiIy8kZGVmcy9lbnRpdHlfdHlwZSJ9LAogICAgInRhcmdldF9lbnRpdHlfaWQiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgInJlbGF0aW9uIjogewogICAgICAiZW51bSI6IFsiVVNFRCIsICJHRU5FUkFURUQiLCAiREVSSVZFRF9GUk9NIiwgIk5PUk1BTElaRURfRlJPTSIsICJERVNDUklCRVMiLCAiSEFTX01FTUJFUiJdCiAgICB9LAogICAgImFzc2VydGVkX2F0IjogeyJ0eXBlIjogInN0cmluZyIsICJmb3JtYXQiOiAiZGF0ZS10aW1lIn0sCiAgICAiYXNzZXJ0ZWRfYnkiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgInRyYW5zZm9ybWF0aW9uIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzb2Z0d2FyZV9uYW1lIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdfSwKICAgICJzb2Z0d2FyZV92ZXJzaW9uIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdfSwKICAgICJwYXJhbWV0ZXJzIjogeyJ0eXBlIjogIm9iamVjdCJ9LAogICAgIm5vdGVzIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdfQogIH0sCiAgIiRkZWZzIjogewogICAgImVudGl0eV90eXBlIjogewogICAgICAiZW51bSI6IFsiRVZFTlQiLCAiRVhQRVJJTUVOVCIsICJCRU5DSE1BUktfUlVOIiwgIk9CU0VSVkFUSU9OIiwgIlNPVVJDRV9BUlRJRkFDVCIsICJQUk9WRU5BTkNFX0xJTksiXQogICAgfQogIH0KfQoK",
  "schemas/source_artifact.schema.json": "ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL3NvdXJjZV9hcnRpZmFjdC5zY2hlbWEuanNvbiIsCiAgInRpdGxlIjogIlNPVVJDRV9BUlRJRkFDVCIsCiAgImRlc2NyaXB0aW9uIjogIkEgYnl0ZS1hZGRyZXNzYWJsZSBpbnB1dCwgbWV0YWRhdGEsIHNvZnR3YXJlLCBvciBnZW5lcmF0ZWQgb3V0cHV0IG9iamVjdC4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWwogICAgInNjaGVtYV92ZXJzaW9uIiwKICAgICJzb3VyY2VfYXJ0aWZhY3RfaWQiLAogICAgImFydGlmYWN0X25hbWUiLAogICAgImFydGlmYWN0X3JvbGUiLAogICAgInVyaSIsCiAgICAibWVkaWFfdHlwZSIsCiAgICAiYnl0ZV9zaXplIiwKICAgICJ2ZXJzaW9uIiwKICAgICJjaGVja3N1bXMiLAogICAgImNoZWNrc3VtX3N0YXR1cyIsCiAgICAicmV0cmlldmFsX3N0YXR1cyIsCiAgICAibGljZW5zZV9leHByZXNzaW9uIiwKICAgICJyZXVzZV9iYXNpcyIsCiAgICAiZGVyaXZhdGlvbl9zdGF0ZSIKICBdLAogICJwcm9wZXJ0aWVzIjogewogICAgInNjaGVtYV92ZXJzaW9uIjogeyJjb25zdCI6ICIxLjAuMCJ9LAogICAgInNvdXJjZV9hcnRpZmFjdF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiYXJ0aWZhY3RfbmFtZSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiYXJ0aWZhY3Rfcm9sZSI6IHsKICAgICAgImVudW0iOiBbCiAgICAgICAgIlFVRVJZX1ZDRiIsICJUUlVUSF9WQ0YiLCAiVFJVVEhfSU5ERVgiLCAiQkVOQ0hNQVJLX0JFRCIsCiAgICAgICAgIlJFRkVSRU5DRV9GQVNUQSIsICJSRUZFUkVOQ0VfSU5ERVgiLCAiRVhQRVJJTUVOVF9NRVRBREFUQSIsCiAgICAgICAgIkRBVEFTRVRfTUFOSUZFU1QiLCAiU09GVFdBUkVfSU1BR0UiLCAiU09GVFdBUkVfQVJDSElWRSIsCiAgICAgICAgIkRFUklWRURfU0xJQ0UiLCAiQ09NUEFSQVRPUl9PVVRQVVQiLCAiUlVOX0xPRyIsICJQQVJRVUVUIiwKICAgICAgICAiRFVDS0RCIiwgIlZBTElEQVRJT05fUkVQT1JUIiwgIk9USEVSIgogICAgICBdCiAgICB9LAogICAgInVyaSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAibWVkaWFfdHlwZSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiYnl0ZV9zaXplIjogeyJ0eXBlIjogWyJpbnRlZ2VyIiwgIm51bGwiXSwgIm1pbmltdW0iOiAwfSwKICAgICJ2ZXJzaW9uIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJjaGVja3N1bXMiOiB7CiAgICAgICJ0eXBlIjogImFycmF5IiwKICAgICAgIml0ZW1zIjogewogICAgICAgICJ0eXBlIjogIm9iamVjdCIsCiAgICAgICAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgICAgICAgInJlcXVpcmVkIjogWyJhbGdvcml0aG0iLCAidmFsdWUiLCAic291cmNlIl0sCiAgICAgICAgInByb3BlcnRpZXMiOiB7CiAgICAgICAgICAiYWxnb3JpdGhtIjogeyJlbnVtIjogWyJzaGEyNTYiLCAibWQ1IiwgIm9jaS1tYW5pZmVzdC1kaWdlc3QiXX0sCiAgICAgICAgICAidmFsdWUiOiB7InR5cGUiOiAic3RyaW5nIiwgInBhdHRlcm4iOiAiXlswLTlhLWZdezMyLDY0fSQifSwKICAgICAgICAgICJzb3VyY2UiOiB7ImVudW0iOiBbIlBVQkxJU0hFUiIsICJDT01QVVRFRF9PTl9SRVRSSUVWQUwiLCAiUkVHSVNUUlkiXX0KICAgICAgICB9CiAgICAgIH0KICAgIH0sCiAgICAiY2hlY2tzdW1fc3RhdHVzIjogeyJlbnVtIjogWyJQVUJMSVNIRVJfVkVSSUZJRUQiLCAiQ09NUFVURURfTk9UX1lFVF9WRVJJRklFRCIsICJOT1RfWUVUX1JFVFJJRVZFRCJdfSwKICAgICJyZXRyaWV2YWxfc3RhdHVzIjogeyJlbnVtIjogWyJOT1RfRE9XTkxPQURFRCIsICJET1dOTE9BREVEIiwgIkdFTkVSQVRFRCJdfSwKICAgICJyZXRyaWV2ZWRfYXQiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl0sICJmb3JtYXQiOiAiZGF0ZS10aW1lIn0sCiAgICAibGljZW5zZV9leHByZXNzaW9uIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJyZXVzZV9iYXNpcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiZGVyaXZhdGlvbl9zdGF0ZSI6IHsiZW51bSI6IFsiU09VUkNFIiwgIlRSQU5TRk9STUVEIiwgIkdFTkVSQVRFRCJdfSwKICAgICJwdWJsaXNoZXJfbGFzdF9tb2RpZmllZCI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXSwgImZvcm1hdCI6ICJkYXRlLXRpbWUifSwKICAgICJub3RlcyI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0KICB9LAogICJhbGxPZiI6IFsKICAgIHsKICAgICAgImlmIjogeyJwcm9wZXJ0aWVzIjogeyJjaGVja3N1bV9zdGF0dXMiOiB7ImNvbnN0IjogIlBVQkxJU0hFUl9WRVJJRklFRCJ9fX0sCiAgICAgICJ0aGVuIjogeyJwcm9wZXJ0aWVzIjogeyJjaGVja3N1bXMiOiB7Im1pbkl0ZW1zIjogMX19fQogICAgfSwKICAgIHsKICAgICAgImlmIjogeyJwcm9wZXJ0aWVzIjogeyJyZXRyaWV2YWxfc3RhdHVzIjogeyJlbnVtIjogWyJET1dOTE9BREVEIiwgIkdFTkVSQVRFRCJdfX19LAogICAgICAidGhlbiI6IHsKICAgICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAgICJjaGVja3N1bXMiOiB7Im1pbkl0ZW1zIjogMX0sCiAgICAgICAgICAicmV0cmlldmVkX2F0IjogeyJ0eXBlIjogInN0cmluZyJ9CiAgICAgICAgfQogICAgICB9CiAgICB9CiAgXQp9Cgo="
}
EMBEDDED_SHA256 = {
    name: hashlib.sha256(base64.b64decode(payload)).hexdigest()
    for name, payload in EMBEDDED_B64.items()
}
for rel, payload in EMBEDDED_B64.items():
    raw = base64.b64decode(payload)
    if hashlib.sha256(raw).hexdigest() != EMBEDDED_SHA256[rel]:
        raise RuntimeError(f"embedded snapshot checksum mismatch: {rel}")
    target = CONFIG_DIR / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(raw)

LOCK_PATH = CONFIG_DIR / "environment/comparator-lock.yaml"
SOURCE_MANIFEST_PATH = CONFIG_DIR / "data/manifests/source_artifacts.tsv"
EXPERIMENTS_PATH = CONFIG_DIR / "data/manifests/selected_experiments.tsv"
VALIDATION_PATH = CONFIG_DIR / "docs/phase1a_validation.md"
SCHEMA_DIR = CONFIG_DIR / "schemas"
QUERY_SQL_PATH = CONFIG_DIR / "sql/event_evidence.sql"

def read_tsv(path):
    with open(path, newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle, delimiter="\t"))

source_manifest_rows = read_tsv(SOURCE_MANIFEST_PATH)
source_manifest = {r["source_artifact_id"]: r for r in source_manifest_rows}
selected_rows = read_tsv(EXPERIMENTS_PATH)
selected = {r["source_submission_id"].lower(): r for r in selected_rows}

CORE = {"assembly":"GRCh38","contig":"chr20","start_0based":10000000,"end_0based":11000000}
PADDED = {"assembly":"GRCh38","contig":"chr20","start_0based":9999000,"end_0based":11001000}
CORE_VCF_REGION = "chr20:10000001-11000000"
PADDED_VCF_REGION = "chr20:9999001-11001000"
EXPECTED_SUBMISSIONS = {"60z59", "ru88n"}
assert set(selected) == EXPECTED_SUBMISSIONS
assert all(r["sample_id"] == "HG002" for r in selected_rows)
print("Embedded authoritative snapshots:")
for k in sorted(EMBEDDED_SHA256):
    print(EMBEDDED_SHA256[k], k)


## 4. Comparator/runtime installation

Install the pinned Python dependencies and the pinned rootless runner `udocker==1.3.17`, verify the locked hap.py OCI manifest digest with Skopeo, pull and expand the image under udocker P1, and mount the separately pinned RTG Tools 3.12.1 archive.

Google Colab kernels run as root, so every udocker invocation uses the documented global `--allow-root` option through one centralized command prefix.

Comparator identity is fail-closed. The authoritative identity is the **immutable OCI manifest digest** together with the exact Bioconda build tag `0.3.15--py27hcb73b3d_0`. The notebook additionally probes the running container (`hap.py --version`, `hap.py --help`, executable path, Conda metadata where present). If in-container metadata reports a contradictory hap.py version, the notebook blocks. If the legacy BioContainer does not expose package metadata, the version may still be established from the exact digest plus build-tag identity and a successful `hap.py` executable sanity probe; this fallback is recorded explicitly and is never misrepresented as a CLI-reported version.

All probe outputs are written to `hap_py_version_probes.json`. Any comparator identity or startup failure emits `COMPARATOR_ENVIRONMENT_BLOCKED` and stops before genomic source processing.


In [ ]:
COMMAND_LOG = RESULTS_DIR / "commands.log"
COMMAND_LOG.write_text("", encoding="utf-8")
runtime_identity = {}
retrieval_rows = []
source_artifacts = []
provenance_links = []
derived_paths = {}
runs_state = {}

def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def md5_file(path, chunk=8*1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def tree_sha256(path):
    h = hashlib.sha256()
    root = Path(path)
    for p in sorted(x for x in root.rglob("*") if x.is_file()):
        h.update(str(p.relative_to(root)).encode())
        h.update(b"\0")
        h.update(sha256_file(p).encode())
        h.update(b"\n")
    return h.hexdigest()

def append_command(argv, cwd=None):
    line = json.dumps({
        "at": now_iso(), "cwd": str(cwd or Path.cwd()),
        "argv": [str(x) for x in argv],
    }, sort_keys=True)
    with COMMAND_LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")

def run(argv, *, cwd=None, check=True, capture=True, env=None):
    argv = [str(x) for x in argv]
    append_command(argv, cwd)
    cp = subprocess.run(
        argv, cwd=cwd, env=env, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
    )
    if check and cp.returncode:
        raise RuntimeError(
            f"command failed ({cp.returncode}): {shlex.join(argv)}\n"
            f"stdout:\n{(cp.stdout or '')[-4000:]}\n"
            f"stderr:\n{(cp.stderr or '')[-4000:]}"
        )
    return cp

def write_tsv(path, rows, fieldnames=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0]) if rows else []
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t", extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)

def download_with_resume(url, dest, expected_size):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size == int(expected_size):
        print(f"reuse complete cached download: {dest}")
        return
    run(["curl", "-L", "--fail", "--retry", "5", "--retry-delay", "2",
         "--continue-at", "-", "--output", str(dest), url])

def refresh_checksums():
    out = RESULTS_DIR / "checksums.sha256"
    lines = []
    for p in sorted(RESULTS_DIR.rglob("*")):
        if p.is_file() and p != out:
            lines.append(f"{sha256_file(p)}  {p.relative_to(RESULTS_DIR)}")
    out.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

def package_partial(status, exc):
    diagnostic = {
        "status": status,
        "captured_at": now_iso(),
        "error_type": type(exc).__name__,
        "error": str(exc),
        "traceback": traceback.format_exc(),
        "runtime_identity": runtime_identity,
        "pipeline_abort_stage": PIPELINE_ABORT_STAGE,
    }
    package_errors = []
    try:
        (RESULTS_DIR / "blocked_or_failed_diagnostics.json").write_text(
            json.dumps(diagnostic, indent=2, sort_keys=True), encoding="utf-8")
        if retrieval_rows:
            write_tsv(RESULTS_DIR / "retrieved_artifacts.tsv", retrieval_rows)
        refresh_checksums()
    except Exception as package_exc:
        package_errors.append(
            f"diagnostic/checksum stage: {type(package_exc).__name__}: {package_exc}"
        )
    try:
        tar_cp = subprocess.run(
            FINAL_TAR_ARGV, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
        if tar_cp.returncode:
            package_errors.append(
                f"archive creation failed ({tar_cp.returncode}): {tar_cp.stderr.strip()}"
            )
        elif ARCHIVE_PATH.exists():
            print(f"archive path: {ARCHIVE_PATH}")
            print(f"archive size: {ARCHIVE_PATH.stat().st_size}")
            print(f"archive SHA-256: {sha256_file(ARCHIVE_PATH)}")
    except Exception as package_exc:
        package_errors.append(
            f"archive stage: {type(package_exc).__name__}: {package_exc}"
        )
    for package_error in package_errors:
        print(f"PARTIAL_PACKAGE_ERROR: {package_error}", file=sys.stderr)
    emit_status_once(status)

from contextlib import contextmanager

@contextmanager
def stage_guard(stage, blocked=False):
    global PIPELINE_ABORTED, PIPELINE_ABORT_STATUS, PIPELINE_ABORT_STAGE
    if PIPELINE_ABORTED:
        raise Phase1AAbort(
            f"Pipeline already stopped at {PIPELINE_ABORT_STAGE} "
            f"with status {PIPELINE_ABORT_STATUS}; do not continue."
        )
    try:
        yield
    except Phase1AAbort:
        raise
    except Exception as exc:
        status = "COMPARATOR_ENVIRONMENT_BLOCKED" if blocked else "PHASE1A_COMPUTE_FAIL"
        PIPELINE_ABORTED = True
        PIPELINE_ABORT_STATUS = status
        PIPELINE_ABORT_STAGE = stage
        original = RuntimeError(f"{stage}: {exc}")
        print(f"{status}: {original}", file=sys.stderr)
        package_partial(status, original)
        raise Phase1AAbort(f"{status}: {original}") from None

def _record_probe(name, cp=None, note=None):
    rec = {"name": name, "captured_at": now_iso()}
    if cp is not None:
        rec.update({
            "returncode": cp.returncode,
            "stdout": cp.stdout or "",
            "stderr": cp.stderr or "",
        })
    if note is not None:
        rec["note"] = note
    return rec

def _parse_conda_json_version(text):
    try:
        payload = json.loads(text)
    except Exception:
        return []
    versions = []
    if isinstance(payload, list):
        for row in payload:
            if not isinstance(row, dict):
                continue
            name = str(row.get("name", "")).strip().lower()
            if name in {"hap.py", "hap-py", "happy"}:
                version = str(row.get("version", "")).strip()
                if version:
                    versions.append(version)
    return versions

def _parse_conda_text_versions(text):
    versions = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        if len(parts) >= 2 and parts[0].lower() in {"hap.py", "hap-py", "happy"}:
            versions.append(parts[1])
    return versions

def _parse_conda_meta_versions(text):
    versions = []
    for raw in text.splitlines():
        p = raw.strip()
        if not p:
            continue
        name = Path(p).name
        m = re.match(r"^hap\.py-(\d+\.\d+\.\d+)(?:[-_].*)?\.json$", name, re.I)
        if m:
            versions.append(m.group(1))
    return versions

def establish_happy_runtime_version(container_run, expected_version, expected_tag,
                                    expected_digest, observed_digest_before,
                                    observed_digest_after):
    """Establish pinned hap.py identity without inventing a CLI-reported version."""
    probes = []
    candidates = []

    version_cp = container_run(["hap.py", "--version"], check=False)
    probes.append(_record_probe("hap.py --version", version_cp))
    version_text = ((version_cp.stdout or "") + "\n" + (version_cp.stderr or "")).strip()
    cli_match = re.search(r"(?i)hap(?:\.py)?[^0-9]{0,20}v?(\d+\.\d+\.\d+)", version_text)
    if cli_match:
        candidates.append({"version": cli_match.group(1), "source": "hap.py --version"})
    elif re.search(rf"(?<![0-9.]){re.escape(expected_version)}(?![0-9.])", version_text):
        candidates.append({"version": expected_version, "source": "hap.py --version"})

    help_cp = container_run(["hap.py", "--help"], check=False)
    probes.append(_record_probe("hap.py --help", help_cp))
    if help_cp.returncode != 0:
        raise RuntimeError(
            "hap.py executable sanity probe failed; "
            f"returncode={help_cp.returncode}; stderr={(help_cp.stderr or '')[-2000:]}"
        )

    which_cp = container_run(
        ["bash", "-lc", "command -v hap.py || command -v happy || true"], check=False
    )
    probes.append(_record_probe("hap.py executable path", which_cp))
    executable_lines = (which_cp.stdout or "").strip().splitlines()
    executable_path = executable_lines[0].strip() if executable_lines else ""
    if not executable_path:
        raise RuntimeError("hap.py executable path could not be established")

    conda_json_cp = container_run(
        ["bash", "-lc", "if command -v conda >/dev/null 2>&1; then conda list --json hap.py 2>/dev/null; else exit 127; fi"],
        check=False,
    )
    probes.append(_record_probe("conda list --json hap.py", conda_json_cp))
    for version in _parse_conda_json_version(conda_json_cp.stdout or ""):
        candidates.append({"version": version, "source": "conda list --json hap.py"})

    conda_text_cp = container_run(
        ["bash", "-lc", "if command -v conda >/dev/null 2>&1; then conda list hap.py 2>/dev/null; else exit 127; fi"],
        check=False,
    )
    probes.append(_record_probe("conda list hap.py", conda_text_cp))
    for version in _parse_conda_text_versions(conda_text_cp.stdout or ""):
        candidates.append({"version": version, "source": "conda list hap.py"})

    meta_script = """for root in /usr/local /opt /conda /usr; do
  [ -d "$root" ] || continue
  find "$root" -type f -path '*/conda-meta/hap.py-*.json' -print 2>/dev/null
done | sort -u | head -200"""
    meta_cp = container_run(["bash", "-lc", meta_script], check=False)
    probes.append(_record_probe("recursive conda-meta hap.py package records", meta_cp))
    for version in _parse_conda_meta_versions(meta_cp.stdout or ""):
        candidates.append({"version": version, "source": "conda-meta package filename"})

    package_name_script = """for root in /usr/local /opt /conda; do
  [ -d "$root" ] || continue
  find "$root" -maxdepth 6 \
    \( -type f -o -type d \) \
    \( -name 'hap.py-0.3.*' -o -name 'hap.py-*.tar.bz2' -o -name 'hap.py-*.conda' \) \
    -print 2>/dev/null
done | sort -u | head -200"""
    pkg_cp = container_run(["bash", "-lc", package_name_script], check=False)
    probes.append(_record_probe("hap.py package/build-name search", pkg_cp))
    for raw in (pkg_cp.stdout or "").splitlines():
        m = re.search(r"hap\.py-(\d+\.\d+\.\d+)(?:[-_]|$)", Path(raw.strip()).name, re.I)
        if m:
            candidates.append({"version": m.group(1), "source": "package/build-name search"})

    unique = []
    seen = set()
    for item in candidates:
        key = (item["version"], item["source"])
        if key not in seen:
            seen.add(key)
            unique.append(item)
    candidates = unique
    observed_versions = sorted({x["version"] for x in candidates})

    tag_match = re.match(r"^(\d+\.\d+\.\d+)--(.+)$", expected_tag)
    tag_version = tag_match.group(1) if tag_match else None
    tag_build = tag_match.group(2) if tag_match else None
    digest_identity_ok = (
        observed_digest_before == expected_digest and
        observed_digest_after == expected_digest
    )

    conflicting = [v for v in observed_versions if v != expected_version]
    if conflicting:
        evidence_mode = "CONFLICT"
        evidence_sources = []
        established_version = None
        runtime_reported_version = None
    elif expected_version in observed_versions:
        evidence_mode = "IN_CONTAINER_VERSION_EVIDENCE"
        evidence_sources = sorted({x["source"] for x in candidates if x["version"] == expected_version})
        established_version = expected_version
        runtime_reported_version = expected_version
    elif tag_version == expected_version and digest_identity_ok:
        evidence_mode = "IMMUTABLE_OCI_DIGEST_PLUS_BIOCONDA_BUILD_TAG"
        evidence_sources = [
            "exact OCI manifest digest",
            f"Bioconda container build tag {expected_tag}",
            "successful hap.py --help runtime probe",
        ]
        established_version = expected_version
        runtime_reported_version = None
    else:
        evidence_mode = "UNRESOLVED"
        evidence_sources = []
        established_version = None
        runtime_reported_version = None

    probe_payload = {
        "expected_version": expected_version,
        "expected_tag": expected_tag,
        "expected_digest": expected_digest,
        "established_version": established_version,
        "runtime_reported_version": runtime_reported_version,
        "identity_evidence_mode": evidence_mode,
        "identity_evidence_sources": evidence_sources,
        "candidate_evidence": candidates,
        "observed_versions": observed_versions,
        "executable_path": executable_path,
        "tag_version": tag_version,
        "tag_build": tag_build,
        "digest_identity_ok": digest_identity_ok,
        "probes": probes,
    }
    (RESULTS_DIR / "hap_py_version_probes.json").write_text(
        json.dumps(probe_payload, indent=2, sort_keys=True), encoding="utf-8"
    )

    if conflicting:
        raise RuntimeError(
            f"runtime hap.py version conflict: expected {expected_version}, observed {observed_versions}; "
            "see hap_py_version_probes.json"
        )
    if established_version != expected_version:
        raise RuntimeError(
            f"hap.py {expected_version} identity could not be established: "
            f"tag_version={tag_version}, digest_identity_ok={digest_identity_ok}, "
            f"observed_versions={observed_versions}; see hap_py_version_probes.json"
        )

    return {
        "established_version": established_version,
        "runtime_reported_version": runtime_reported_version,
        "identity_evidence_mode": evidence_mode,
        "evidence_sources": evidence_sources,
        "cli_returncode": version_cp.returncode,
        "cli_output": version_text,
        "help_returncode": help_cp.returncode,
        "executable_path": executable_path,
        "probe_artifact": str(RESULTS_DIR / "hap_py_version_probes.json"),
    }

with stage_guard("comparator/runtime installation", blocked=True):
    run(["apt-get", "update"], capture=False)
    run(["apt-get", "install", "-y", "--no-install-recommends",
         "bcftools", "samtools", "tabix", "skopeo", "unzip", "curl", "time"], capture=False)
    run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
         "udocker==1.3.17", "duckdb==1.5.6", "pyarrow==25.0.1",
         "pysam==0.23.3", "jsonschema==4.26.0", "PyYAML==6.0.3"], capture=False)

    import yaml
    lock = yaml.safe_load(LOCK_PATH.read_text())
    assert lock["hap_py"]["version"] == "0.3.15"
    assert lock["rtg_tools"]["version"] == "3.12.1"
    assert lock["comparison"]["engine"] == "vcfeval"
    assert lock["comparison"]["annotated_vcf"] is True
    assert lock["comparison"]["threads"] == 2
    assert lock["comparison"]["core_interval"] == CORE
    assert lock["comparison"]["padded_interval"] == PADDED
    assert lock["hap_py"]["container"]["manifest_digest"] == (
        "sha256:" + source_manifest["artifact-happy-container"]["checksum_value"])
    assert lock["rtg_tools"]["linux_x64_archive_url"] == source_manifest["artifact-rtg-tools-3121"]["uri"]
    assert str(lock["rtg_tools"]["linux_x64_archive_bytes"]) == source_manifest["artifact-rtg-tools-3121"]["byte_size"]
    assert lock["rtg_tools"]["linux_x64_archive_sha256"] == source_manifest["artifact-rtg-tools-3121"]["checksum_value"]

    runner_version = "1.3.17"
    image_tag = "quay.io/biocontainers/hap.py:" + lock["hap_py"]["container"]["tag_for_humans_only"]
    expected_digest = lock["hap_py"]["container"]["manifest_digest"]
    image_by_digest = "docker://" + lock["hap_py"]["container"]["reference"]

    observed_digest_before = run(
        ["skopeo", "inspect", "--format", "{{.Digest}}", image_by_digest]).stdout.strip()
    observed_platform = run(
        ["skopeo", "inspect", "--format", "{{.Os}}/{{.Architecture}}", image_by_digest]).stdout.strip()
    if observed_digest_before != expected_digest:
        raise RuntimeError(
            f"registry digest mismatch: {observed_digest_before} != {expected_digest}")
    if observed_platform != "linux/amd64":
        raise RuntimeError(f"comparator platform mismatch: {observed_platform}")

    rtg_row = source_manifest["artifact-rtg-tools-3121"]
    rtg_zip = TOOLS_DIR / rtg_row["artifact_name"]
    download_with_resume(rtg_row["uri"], rtg_zip, rtg_row["byte_size"])
    if rtg_zip.stat().st_size != int(rtg_row["byte_size"]):
        raise RuntimeError("RTG archive size mismatch")
    if sha256_file(rtg_zip) != rtg_row["checksum_value"]:
        raise RuntimeError("RTG archive SHA-256 mismatch")
    run(["unzip", "-q", "-o", str(rtg_zip), "-d", str(TOOLS_DIR)])
    rtg_dir = TOOLS_DIR / "rtg-tools-3.12.1"
    rtg_exe = rtg_dir / "rtg"
    if not rtg_exe.exists():
        raise RuntimeError("RTG executable missing after extraction")

    udocker_env = dict(os.environ, UDOCKER_DIR=str(TOOLS_DIR / "udocker"))
    UDOCKER_BASE = ["udocker", "--allow-root"]
    runtime_identity.update({
        "host_euid": HOST_EUID,
        "host_user": WHOAMI,
        "colab_host_root": COLAB_HOST_ROOT,
        "udocker_allow_root": True,
    })

    udocker_version_cp = run(UDOCKER_BASE + ["--version"], env=udocker_env)
    udocker_version_text = (
        (udocker_version_cp.stdout or "") + (udocker_version_cp.stderr or "")
    ).strip()
    if runner_version not in udocker_version_text:
        raise RuntimeError(f"udocker runtime version mismatch: {udocker_version_text}")

    run(UDOCKER_BASE + ["install"], env=udocker_env)
    run(UDOCKER_BASE + ["pull", image_tag], env=udocker_env)

    observed_digest_after = run(
        ["skopeo", "inspect", "--format", "{{.Digest}}", "docker://" + image_tag]
    ).stdout.strip()
    if observed_digest_after != expected_digest:
        raise RuntimeError(
            f"tag digest changed or mismatched: {observed_digest_after} != {expected_digest}")

    container_name = "phase1a-happy-0315"
    run(UDOCKER_BASE + ["rm", container_name], check=False, env=udocker_env)
    run(UDOCKER_BASE + ["create", "--name=" + container_name, image_tag], env=udocker_env)
    run(UDOCKER_BASE + ["setup", "--execmode=P1", container_name], env=udocker_env)

    inspect_text = run(
        UDOCKER_BASE + ["inspect", "-p", container_name], env=udocker_env
    ).stdout
    (RESULTS_DIR / "udocker_inspect.txt").write_text(inspect_text, encoding="utf-8")
    container_dir_candidates = [
        Path(line.strip().strip(chr(34)))
        for line in inspect_text.splitlines()
        if line.strip().strip(chr(34)).startswith("/")
    ]
    container_dir = next((p for p in container_dir_candidates if p.exists()), None)
    if container_dir is None:
        raise RuntimeError("cannot establish expanded udocker container identity")
    local_tree_digest = "sha256:" + tree_sha256(container_dir)

    def container_argv(inner):
        return (
            UDOCKER_BASE
            + [
                "run",
                "--volume=/content:/content",
                f"--volume={rtg_dir}:/opt/rtg-tools-3.12.1",
                container_name,
            ]
            + [str(x) for x in inner]
        )

    def container_run(inner, **kwargs):
        return run(container_argv(inner), env=udocker_env, **kwargs)

    happy_runtime = establish_happy_runtime_version(
        container_run,
        lock["hap_py"]["version"],
        lock["hap_py"]["container"]["tag_for_humans_only"],
        expected_digest,
        observed_digest_before,
        observed_digest_after,
    )

    rtg_version_cp = run([str(rtg_exe), "version"], check=False)
    rtg_version_text = (
        (rtg_version_cp.stdout or "") + "\n" + (rtg_version_cp.stderr or "")
    ).strip()
    if rtg_version_cp.returncode != 0:
        raise RuntimeError(
            f"RTG runtime probe failed ({rtg_version_cp.returncode}): {rtg_version_text}")
    if "3.12.1" not in rtg_version_text:
        raise RuntimeError(
            f"runtime RTG does not report 3.12.1: {rtg_version_text[:2000]}")

    runtime_identity.update({
        "runner": "udocker",
        "runner_version": runner_version,
        "runner_version_output": udocker_version_text,
        "expected_comparator_digest": expected_digest,
        "observed_registry_digest_before_pull": observed_digest_before,
        "observed_registry_digest_after_pull": observed_digest_after,
        "observed_image_platform": observed_platform,
        "observed_expanded_container_tree_digest": local_tree_digest,
        "expanded_tree_digest_semantics":
            "local deterministic file-tree hash; not an OCI manifest digest",
        "hap_py_runtime_established_version": happy_runtime["established_version"],
        "hap_py_runtime_reported_version": happy_runtime["runtime_reported_version"],
        "hap_py_runtime_identity_evidence_mode": happy_runtime["identity_evidence_mode"],
        "hap_py_runtime_version_evidence_sources": happy_runtime["evidence_sources"],
        "hap_py_executable_path": happy_runtime["executable_path"],
        "hap_py_bioconda_build_tag": lock["hap_py"]["container"]["tag_for_humans_only"],
        "hap_py_version_output": happy_runtime["cli_output"],
        "hap_py_version_cli_returncode": happy_runtime["cli_returncode"],
        "hap_py_help_returncode": happy_runtime["help_returncode"],
        "hap_py_version_probe_artifact": happy_runtime["probe_artifact"],
        "rtg_version_output": rtg_version_text,
        "bcftools_version": run(["bcftools", "--version"]).stdout.splitlines()[0],
        "samtools_version": run(["samtools", "--version"]).stdout.splitlines()[0],
        "skopeo_version": run(["skopeo", "--version"]).stdout.strip(),
        "python_version": sys.version,
        "architecture": platform.machine(),
    })
    (RESULTS_DIR / "runtime_identity.json").write_text(
        json.dumps(runtime_identity, indent=2, sort_keys=True), encoding="utf-8"
    )

print("Comparator environment established.")
print("hap.py established version:", runtime_identity["hap_py_runtime_established_version"])
print("hap.py identity mode:", runtime_identity["hap_py_runtime_identity_evidence_mode"])
print("hap.py version evidence:",
      ", ".join(runtime_identity["hap_py_runtime_version_evidence_sources"]))
print("RTG runtime:", runtime_identity["rtg_version_output"].splitlines()[0])
print("OCI digest:", runtime_identity["observed_registry_digest_after_pull"])


## 5. Download source artifacts

Download only the exact manifest objects needed for metadata, the two full unindexed query VCFs, GIAB truth/TBI/BED, and the pinned reference/FAI/GZI. Resume downloads when possible. No FASTQ, BAM, or CRAM is requested.

In [ ]:
DOWNLOAD_IDS = [
    "artifact-pfda-readme",
    "artifact-pfda-submission-methods",
    "artifact-pfda-anonymized-metadata",
    "artifact-query-60z59-hg002",
    "artifact-query-ru88n-hg002",
    "artifact-truth-hg002-grch38-v421",
    "artifact-truth-hg002-grch38-v421-tbi",
    "artifact-bed-hg002-grch38-v421",
    "artifact-reference-grch38-no-alt",
    "artifact-reference-grch38-no-alt-fai",
    "artifact-reference-grch38-no-alt-gzi",
]
downloaded = {}
with stage_guard("source artifact download"):
    for artifact_id in DOWNLOAD_IDS:
        row = source_manifest[artifact_id]
        dest = SOURCE_DIR / row["artifact_name"]
        download_with_resume(row["uri"], dest, row["byte_size"])
        downloaded[artifact_id] = dest
print("Downloaded manifest artifacts:", len(downloaded))


## 6. Verify artifact size/checksum

Fail on every expected-size mismatch and every supplied checksum mismatch. Compute SHA-256 immediately for every retrieved object; retain computed GIAB hashes in `retrieved_artifacts.tsv` without mutating the source manifest. Verify publisher MD5 for the reference and its indexes.

In [ ]:
MEDIA_TYPES = {
    ".vcf.gz":"application/gzip", ".tbi":"application/octet-stream",
    ".bed":"text/tab-separated-values", ".fasta.gz":"application/gzip",
    ".fai":"text/plain", ".gzi":"application/octet-stream",
    ".tsv":"text/tab-separated-values", ".txt":"text/plain", ".md":"text/markdown",
    ".zip":"application/zip",
}
def media_type_for(name):
    for suffix, mt in MEDIA_TYPES.items():
        if name.endswith(suffix):
            return mt
    return "application/octet-stream"

def source_record_from_manifest(row, path=None, status="NOT_DOWNLOADED"):
    checksums = []
    alg = row["checksum_algorithm"]
    val = row["checksum_value"]
    if alg in {"sha256", "md5", "oci-manifest-digest"} and val != "UNKNOWN":
        checksums.append({
            "algorithm": alg,
            "value": val.removeprefix("sha256:"),
            "source": "PUBLISHER" if row["checksum_source"] == "PUBLISHER" else
                      "REGISTRY" if row["checksum_source"] == "REGISTRY" else
                      "COMPUTED_ON_RETRIEVAL",
        })
    if path is not None:
        computed = sha256_file(path)
        if not any(c["algorithm"] == "sha256" for c in checksums):
            checksums.append({"algorithm":"sha256","value":computed,"source":"COMPUTED_ON_RETRIEVAL"})
    return {
        "schema_version":"1.0.0",
        "source_artifact_id":row["source_artifact_id"],
        "artifact_name":row["artifact_name"],
        "artifact_role":row["artifact_role"],
        "uri":row["uri"],
        "media_type":media_type_for(row["artifact_name"]),
        "byte_size":path.stat().st_size if path is not None else int(row["byte_size"]),
        "version":row["version"],
        "checksums":checksums,
        "checksum_status":("PUBLISHER_VERIFIED" if row["checksum_status"] == "PUBLISHER_VERIFIED"
                           else "COMPUTED_NOT_YET_VERIFIED" if path is not None
                           else "NOT_YET_RETRIEVED"),
        "retrieval_status":status,
        "retrieved_at":now_iso() if status in {"DOWNLOADED","GENERATED"} else None,
        "license_expression":"UNKNOWN",
        "reuse_basis":row["license_or_reuse_basis"] or "UNKNOWN",
        "derivation_state":"SOURCE",
        "publisher_last_modified":None,
        "notes":row["notes"] or None,
    }

with stage_guard("source artifact integrity verification"):
    for artifact_id, path in downloaded.items():
        row = source_manifest[artifact_id]
        actual_size = path.stat().st_size
        expected_size = int(row["byte_size"])
        if actual_size != expected_size:
            raise RuntimeError(f"{artifact_id} size {actual_size} != {expected_size}")
        actual_sha = sha256_file(path)
        actual_md5 = md5_file(path)
        alg, expected = row["checksum_algorithm"], row["checksum_value"]
        if alg == "sha256" and actual_sha != expected:
            raise RuntimeError(f"{artifact_id} SHA-256 mismatch")
        if alg == "md5" and actual_md5 != expected:
            raise RuntimeError(f"{artifact_id} publisher MD5 mismatch")
        retrieval_rows.append({
            "source_artifact_id":artifact_id, "artifact_name":row["artifact_name"],
            "uri":row["uri"], "expected_byte_size":expected_size,
            "actual_byte_size":actual_size, "publisher_checksum_algorithm":alg,
            "publisher_checksum_value":expected, "computed_sha256":actual_sha,
            "computed_md5":actual_md5, "verified_at":now_iso(), "status":"VERIFIED",
        })
        source_artifacts.append(source_record_from_manifest(row, path, "DOWNLOADED"))

    rtg_row = source_manifest["artifact-rtg-tools-3121"]
    retrieval_rows.append({
        "source_artifact_id":rtg_row["source_artifact_id"], "artifact_name":rtg_row["artifact_name"],
        "uri":rtg_row["uri"], "expected_byte_size":int(rtg_row["byte_size"]),
        "actual_byte_size":rtg_zip.stat().st_size, "publisher_checksum_algorithm":"sha256",
        "publisher_checksum_value":rtg_row["checksum_value"],
        "computed_sha256":sha256_file(rtg_zip), "computed_md5":md5_file(rtg_zip),
        "verified_at":now_iso(), "status":"VERIFIED",
    })
    source_artifacts.append(source_record_from_manifest(rtg_row, rtg_zip, "DOWNLOADED"))

    image_row = source_manifest["artifact-happy-container"]
    image_record = source_record_from_manifest(image_row, None, "DOWNLOADED")
    image_record["retrieved_at"] = now_iso()
    image_record["checksums"] = [{
        "algorithm":"oci-manifest-digest",
        "value":runtime_identity["observed_registry_digest_after_pull"].removeprefix("sha256:"),
        "source":"REGISTRY",
    }]
    image_record["checksum_status"] = "PUBLISHER_VERIFIED"
    source_artifacts.append(image_record)

    write_tsv(RESULTS_DIR / "retrieved_artifacts.tsv", retrieval_rows)
print("All expected sizes and supplied checksums verified.")


## 7. Prepare/index VCFs

Record each query’s original checksum before indexing. Create a tabix index without rewriting the source VCF. Only if the original is not indexable, retain it and generate a separately named BGZF/sorted derivative with a separately recorded command, checksum, artifact, and provenance edge.

In [ ]:
def make_derived_artifact(artifact_id, path, role, version, notes):
    path = Path(path)
    rec = {
        "schema_version":"1.0.0", "source_artifact_id":artifact_id,
        "artifact_name":path.name, "artifact_role":role,
        "uri":"file://" + str(path), "media_type":media_type_for(path.name),
        "byte_size":path.stat().st_size, "version":version,
        "checksums":[{"algorithm":"sha256","value":sha256_file(path),"source":"COMPUTED_ON_RETRIEVAL"}],
        "checksum_status":"COMPUTED_NOT_YET_VERIFIED",
        "retrieval_status":"GENERATED", "retrieved_at":now_iso(),
        "license_expression":"UNKNOWN", "reuse_basis":"Phase 1A derived evidence",
        "derivation_state":"GENERATED", "publisher_last_modified":None, "notes":notes,
    }
    source_artifacts.append(rec)
    derived_paths[artifact_id] = path
    return rec

def add_link(source_type, source_id, target_type, target_id, relation, transformation,
             software=None, version=None, parameters=None, notes=None):
    provenance_links.append({
        "schema_version":"1.0.0",
        "provenance_link_id":"prov-" + uuid.uuid4().hex,
        "source_entity_type":source_type, "source_entity_id":source_id,
        "target_entity_type":target_type, "target_entity_id":target_id,
        "relation":relation, "asserted_at":now_iso(), "asserted_by":"phase1a_colab_poc",
        "transformation":transformation, "software_name":software,
        "software_version":version, "parameters":parameters or {}, "notes":notes,
    })

query_ready = {}
with stage_guard("query VCF preparation"):
    for submission in ("60z59", "ru88n"):
        exp = selected[submission]
        source_id = exp["query_artifact_id"]
        original = downloaded[source_id]
        original_sha = sha256_file(original)
        view = run(["bcftools", "view", "-h", str(original)], check=False)
        if view.returncode:
            raise RuntimeError(f"{source_id} is not readable as VCF")
        idx = Path(str(original) + ".tbi")
        index_cp = run(["bcftools", "index", "--tbi", "--force", str(original)], check=False)
        if index_cp.returncode == 0:
            ready = original
            idx_id = source_id + "-generated-tbi"
            make_derived_artifact(idx_id, idx, "REFERENCE_INDEX", "tabix 1.0",
                                  "Index generated without altering source VCF bytes.")
            add_link("SOURCE_ARTIFACT", source_id, "SOURCE_ARTIFACT", idx_id, "GENERATED",
                     "tabix index of byte-identical source VCF", "bcftools",
                     runtime_identity["bcftools_version"])
        else:
            derived = STAGED_DIR / f"{submission}_HG002.sorted.vcf.gz"
            run(["bcftools", "sort", "-Oz", "-o", str(derived), str(original)])
            run(["bcftools", "index", "--tbi", "--force", str(derived)])
            derived_id = source_id + "-sorted-bgzf"
            idx_id = derived_id + "-tbi"
            make_derived_artifact(derived_id, derived, "DERIVED_SLICE", "Phase 1A prerequisite",
                                  "Sorted/BGZF derivative; original source retained unchanged.")
            make_derived_artifact(idx_id, Path(str(derived)+".tbi"), "REFERENCE_INDEX",
                                  "tabix 1.0", "Index for sorted/BGZF derivative.")
            add_link("SOURCE_ARTIFACT", source_id, "SOURCE_ARTIFACT", derived_id, "DERIVED_FROM",
                     "bcftools sort -Oz", "bcftools", runtime_identity["bcftools_version"])
            add_link("SOURCE_ARTIFACT", derived_id, "SOURCE_ARTIFACT", idx_id, "GENERATED",
                     "tabix index", "bcftools", runtime_identity["bcftools_version"])
            ready = derived
        if sha256_file(original) != original_sha:
            raise RuntimeError(f"source query bytes changed: {source_id}")
        query_ready[submission] = ready


## 8. Prepare reference and RTG SDF

Use only the pinned BGZF GRCh38 no-alt reference. Extract chr20 with the supplied FAI/GZI, index the extracted FASTA, then build the RTG SDF from that exact chr20 sequence. Record all products separately.

In [ ]:
with stage_guard("reference and SDF preparation"):
    ref_gz = downloaded["artifact-reference-grch38-no-alt"]
    ref_chr20 = STAGED_DIR / "GCA_000001405.15_GRCh38_no_alt_analysis_set.chr20.fasta"
    cp = run(["samtools", "faidx", str(ref_gz), "chr20"])
    ref_chr20.write_text(cp.stdout, encoding="utf-8")
    run(["samtools", "faidx", str(ref_chr20)])
    fai = Path(str(ref_chr20) + ".fai")
    fields = fai.read_text().splitlines()[0].split("\t")
    if fields[0] != "chr20" or int(fields[1]) != 64444167:
        raise RuntimeError("unexpected chr20 identity/length in pinned reference")
    sdf = STAGED_DIR / "GCA_000001405.15_GRCh38_no_alt_analysis_set.chr20.sdf"
    if sdf.exists():
        shutil.rmtree(sdf)
    run([str(rtg_exe), "format", "-o", str(sdf), str(ref_chr20)])

    make_derived_artifact("artifact-reference-chr20", ref_chr20, "DERIVED_SLICE",
                          "Phase 1A padded-comparison reference", "chr20 extracted from pinned reference.")
    make_derived_artifact("artifact-reference-chr20-fai", fai, "REFERENCE_INDEX",
                          "samtools faidx", "FAI for extracted chr20.")
    sdf_manifest = STAGED_DIR / "reference_chr20_sdf_manifest.json"
    sdf_manifest.write_text(json.dumps({
        "tree_sha256":tree_sha256(sdf),
        "files":[str(p.relative_to(sdf)) for p in sorted(sdf.rglob("*")) if p.is_file()]
    }, indent=2), encoding="utf-8")
    make_derived_artifact("artifact-reference-chr20-sdf", sdf_manifest, "OTHER",
                          "RTG SDF 3.12.1", "Manifest and deterministic tree hash for RTG SDF directory.")
    add_link("SOURCE_ARTIFACT", "artifact-reference-grch38-no-alt", "SOURCE_ARTIFACT",
             "artifact-reference-chr20", "DERIVED_FROM", "samtools faidx chr20",
             "samtools", runtime_identity["samtools_version"])
    add_link("SOURCE_ARTIFACT", "artifact-reference-chr20", "SOURCE_ARTIFACT",
             "artifact-reference-chr20-fai", "GENERATED", "samtools faidx",
             "samtools", runtime_identity["samtools_version"])
    add_link("SOURCE_ARTIFACT", "artifact-reference-chr20", "SOURCE_ARTIFACT",
             "artifact-reference-chr20-sdf", "GENERATED", "rtg format",
             "RTG Tools", "3.12.1")


## 9. Extract padded chr20 interval

Convert the locked 0-based half-open padded interval to the bcftools 1-based inclusive region `chr20:9999001-11001000`. Slice and index truth and each prepared query; padding is context only.

In [ ]:
truth_source = downloaded["artifact-truth-hg002-grch38-v421"]
truth_slice = STAGED_DIR / "HG002_GRCh38_v4.2.1.chr20_9999000_11001000.vcf.gz"
query_slices = {}
with stage_guard("padded VCF extraction"):
    run(["bcftools", "view", "-r", PADDED_VCF_REGION, "-Oz", "-o", str(truth_slice), str(truth_source)])
    run(["bcftools", "index", "--tbi", "--force", str(truth_slice)])
    make_derived_artifact("artifact-truth-padded", truth_slice, "DERIVED_SLICE",
                          "GIAB v4.2.1 Phase 1A padded slice", f"bcftools region {PADDED_VCF_REGION}.")
    make_derived_artifact("artifact-truth-padded-tbi", Path(str(truth_slice)+".tbi"),
                          "REFERENCE_INDEX", "tabix 1.0", "Index for padded truth slice.")
    add_link("SOURCE_ARTIFACT", "artifact-truth-hg002-grch38-v421", "SOURCE_ARTIFACT",
             "artifact-truth-padded", "DERIVED_FROM", f"bcftools view -r {PADDED_VCF_REGION}",
             "bcftools", runtime_identity["bcftools_version"])
    add_link("SOURCE_ARTIFACT", "artifact-truth-padded", "SOURCE_ARTIFACT",
             "artifact-truth-padded-tbi", "GENERATED", "bcftools index --tbi",
             "bcftools", runtime_identity["bcftools_version"])
    for submission in ("60z59", "ru88n"):
        out = STAGED_DIR / f"{submission}_HG002.chr20_9999000_11001000.vcf.gz"
        run(["bcftools", "view", "-r", PADDED_VCF_REGION, "-Oz", "-o", str(out),
             str(query_ready[submission])])
        run(["bcftools", "index", "--tbi", "--force", str(out)])
        query_slices[submission] = out
        aid = f"artifact-query-{submission}-padded"
        make_derived_artifact(aid, out, "DERIVED_SLICE", "Phase 1A padded query slice",
                              f"bcftools region {PADDED_VCF_REGION}.")
        make_derived_artifact(aid+"-tbi", Path(str(out)+".tbi"), "REFERENCE_INDEX",
                              "tabix 1.0", "Index for padded query slice.")
        add_link("SOURCE_ARTIFACT", selected[submission]["query_artifact_id"],
                 "SOURCE_ARTIFACT", aid, "DERIVED_FROM",
                 f"bcftools view -r {PADDED_VCF_REGION}", "bcftools",
                 runtime_identity["bcftools_version"])
        add_link("SOURCE_ARTIFACT", aid, "SOURCE_ARTIFACT", aid+"-tbi", "GENERATED",
                 "bcftools index --tbi", "bcftools", runtime_identity["bcftools_version"])


## 10. Restrict benchmark BED

Intersect the GIAB BED with padded and core intervals using BED’s native 0-based half-open convention. The padded BED drives comparison; the core BED measures assessability and final classification. Zero core overlap is a locked failure.

In [ ]:
bed_source = downloaded["artifact-bed-hg002-grch38-v421"]
padded_bed = STAGED_DIR / "HG002_v4.2.1.chr20.padded.bed"
core_bed = STAGED_DIR / "HG002_v4.2.1.chr20.core.bed"

def clip_bed(src, dst, start, end):
    total = 0
    with open(src, encoding="utf-8") as inp, open(dst, "w", encoding="utf-8") as out:
        for line in inp:
            if not line.strip() or line.startswith(("#","track","browser")):
                continue
            f = line.rstrip("\n").split("\t")
            if f[0] != "chr20":
                continue
            s, e = max(int(f[1]), start), min(int(f[2]), end)
            if s < e:
                out.write("\t".join([f[0], str(s), str(e)] + f[3:]) + "\n")
                total += e - s
    return total

with stage_guard("benchmark BED restriction"):
    padded_assessable_bases = clip_bed(bed_source, padded_bed, PADDED["start_0based"], PADDED["end_0based"])
    core_assessable_bases = clip_bed(bed_source, core_bed, CORE["start_0based"], CORE["end_0based"])
    if core_assessable_bases <= 0:
        raise RuntimeError("locked core interval has zero GIAB benchmark-BED overlap")
    make_derived_artifact("artifact-bed-padded", padded_bed, "DERIVED_SLICE",
                          "GIAB v4.2.1 Phase 1A padded BED", "0-based half-open clipped padded BED.")
    make_derived_artifact("artifact-bed-core", core_bed, "DERIVED_SLICE",
                          "GIAB v4.2.1 Phase 1A core BED", "0-based half-open clipped core BED.")
    add_link("SOURCE_ARTIFACT", "artifact-bed-hg002-grch38-v421", "SOURCE_ARTIFACT",
             "artifact-bed-padded", "DERIVED_FROM", "BED half-open interval clipping",
             "phase1a_colab_poc", "1.0.0", {"interval":PADDED})
    add_link("SOURCE_ARTIFACT", "artifact-bed-hg002-grch38-v421", "SOURCE_ARTIFACT",
             "artifact-bed-core", "DERIVED_FROM", "BED half-open interval clipping",
             "phase1a_colab_poc", "1.0.0", {"interval":CORE})
print({"padded_assessable_bases":padded_assessable_bases,
       "core_assessable_bases":core_assessable_bases})


## 11. Run Illumina comparison

Run submission 60Z59 independently with the locked hap.py/vcfeval contract and annotated VCF writing enabled. Preserve exact argv, stdout, stderr, timing, and exit status. A successful marker makes reruns resumable only after required outputs are rechecked.

In [ ]:
def run_benchmark(submission):
    exp = selected[submission]
    run_id = f"run-phase1a-{submission}-hg002-grch38-v421"
    run_dir = RUNS_DIR / submission
    run_dir.mkdir(parents=True, exist_ok=True)
    prefix = run_dir / f"{submission}.happy"
    stdout_path = run_dir / "hap.py.stdout.log"
    stderr_path = run_dir / "hap.py.stderr.log"
    timing_path = run_dir / "hap.py.time.txt"
    marker = run_dir / "SUCCESS.json"
    inner = [
        "hap.py", str(truth_slice), str(query_slices[submission]),
        "-f", str(padded_bed), "-r", str(ref_chr20), "-o", str(prefix),
        "-V", "--engine=vcfeval",
        "--engine-vcfeval-path=/opt/rtg-tools-3.12.1/rtg",
        "--engine-vcfeval-template", str(sdf), "--threads=2",
    ]
    argv = container_argv(inner)
    started = now_iso()
    if marker.exists():
        marker_obj = json.loads(marker.read_text())
        if marker_obj.get("command_argv") != argv:
            raise RuntimeError(f"{submission} success marker command differs from current lock")
        exit_code = 0
    else:
        append_command(argv)
        with stdout_path.open("w") as out, stderr_path.open("w") as err, timing_path.open("w") as tf:
            timed = ["/usr/bin/time", "-v", "-o", str(timing_path)] + argv
            append_command(timed)
            proc = subprocess.run(timed, stdout=out, stderr=err, text=True, env=udocker_env)
        exit_code = proc.returncode
        if exit_code:
            raise RuntimeError(f"hap.py failed for {submission}; see {stderr_path}")
        marker.write_text(json.dumps({
            "completed_at":now_iso(), "command_argv":argv,
            "stdout_sha256":sha256_file(stdout_path),
            "stderr_sha256":sha256_file(stderr_path),
        }, indent=2), encoding="utf-8")
    completed = now_iso()
    runs_state[submission] = {
        "run_id":run_id, "experiment_id":exp["experiment_id"], "prefix":prefix,
        "run_dir":run_dir, "command_argv":argv, "started_at":started,
        "completed_at":completed, "exit_code":exit_code,
        "stdout":stdout_path, "stderr":stderr_path, "timing":timing_path,
    }

with stage_guard("Illumina hap.py comparison"):
    run_benchmark("60z59")


## 12. Run ONT comparison

Run submission RU88N independently under the identical scientific comparison contract.

In [ ]:
with stage_guard("ONT hap.py comparison"):
    run_benchmark("ru88n")


## 13. Verify annotated outputs

For each run, locate a non-empty annotated VCF, ensure it is readable and indexed, require samples `TRUTH` and `QUERY`, and require FORMAT definitions `BD` and `BK`. Preserve `BS` when present and use the locked source-record fallback only for records without it. Metrics and logs are mandatory. This gate runs before any observation construction.

In [ ]:
import pysam

def find_annotated_vcf(prefix):
    candidates = [
        Path(str(prefix)+".vcf.gz"),
        Path(str(prefix)+".happy.vcf.gz"),
    ]
    candidates.extend(sorted(prefix.parent.glob(prefix.name + "*.vcf.gz")))
    for p in candidates:
        if p.exists() and p.stat().st_size > 0:
            return p
    raise RuntimeError(f"no annotated VCF found for {prefix}")

annotation_gates = {}
with stage_guard("annotated output gate"):
    compact_dir = RESULTS_DIR / "comparison_outputs"
    compact_dir.mkdir(exist_ok=True)
    for submission in ("60z59", "ru88n"):
        st = runs_state[submission]
        annotated = find_annotated_vcf(st["prefix"])
        tbi = Path(str(annotated)+".tbi")
        if not tbi.exists():
            run(["bcftools", "index", "--tbi", "--force", str(annotated)])
        metrics = [p for p in sorted(st["run_dir"].glob(st["prefix"].name + "*"))
                   if p.is_file() and p not in {annotated, tbi} and
                   any(x in p.name for x in ("summary", "extended", "metrics"))]
        if not metrics:
            raise RuntimeError(f"metrics output absent for {submission}")
        vf = pysam.VariantFile(str(annotated))
        samples = list(vf.header.samples)
        format_ids = set(vf.header.formats)
        record_count = sum(1 for _ in vf)
        vf.close()
        missing = {"TRUTH","QUERY"} - set(samples)
        missing_formats = {"BD","BK"} - format_ids
        gate = {
            "submission":submission, "annotated_vcf":str(annotated),
            "samples":samples, "format_ids":sorted(format_ids),
            "has_TRUTH":not ("TRUTH" in missing), "has_QUERY":not ("QUERY" in missing),
            "has_BD":"BD" in format_ids, "has_BK":"BK" in format_ids,
            "has_BS":"BS" in format_ids, "record_count":record_count,
            "metrics":[str(p) for p in metrics],
        }
        annotation_gates[submission] = gate
        if missing or missing_formats or record_count == 0:
            raise RuntimeError(f"annotated event decisions absent/incomplete for {submission}: {gate}")
        st["annotated"] = annotated
        st["annotated_tbi"] = tbi
        st["metrics"] = metrics

        out_vcf = compact_dir / f"{submission}.annotated.vcf.gz"
        out_tbi = Path(str(out_vcf)+".tbi")
        shutil.copy2(annotated, out_vcf)
        shutil.copy2(tbi, out_tbi)
        st["compact_annotated"] = out_vcf
        st["compact_annotated_tbi"] = out_tbi
        for p in (st["stdout"], st["stderr"], st["timing"]):
            shutil.copy2(p, RESULTS_DIR / f"{submission}.{p.name}")
    (RESULTS_DIR / "annotation_gates.json").write_text(
        json.dumps(annotation_gates, indent=2, sort_keys=True), encoding="utf-8")


## 14. Inspect TRUTH/QUERY/BD/BK/BS

Print headers and a small raw preview only after the gate succeeds. This is diagnostic inspection, not inference from aggregates.

In [ ]:
with stage_guard("annotated field inspection"):
    for submission in ("60z59", "ru88n"):
        p = runs_state[submission]["annotated"]
        print(f"\n{submission}: {p}")
        vf = pysam.VariantFile(str(p))
        print("samples:", list(vf.header.samples))
        print("required FORMAT:", {k: str(vf.header.formats[k]) for k in ("BD","BK")})
        print("BS:", str(vf.header.formats["BS"]) if "BS" in vf.header.formats else "ABSENT -> source-record fallback")
        for i, rec in enumerate(vf):
            preview = {"locus":f"{rec.contig}:{rec.pos}", "ref":rec.ref, "alts":rec.alts,
                       "filter":list(rec.filter.keys()),
                       "TRUTH":{k:rec.samples["TRUTH"].get(k) for k in ("GT","BD","BK","BS") if k in rec.samples["TRUTH"]},
                       "QUERY":{k:rec.samples["QUERY"].get(k) for k in ("GT","BD","BK","BS") if k in rec.samples["QUERY"]}}
            print(json.dumps(preview, default=str, sort_keys=True))
            if i >= 2:
                break
        vf.close()


## 15. Parse observations

Create run-scoped events and side-specific observations. Retain raw source coordinates/alleles, FILTER, genotype, BD/BK/BS and all sample FORMAT values. Export only records wholly inside the core interval; retain all padded-output records in an accounting table. BS values never cross run boundaries. An exact normalized lookup key is emitted only for a single unambiguous REF/ALT representation.

In [ ]:
DECISION_MAP = {"TP":"TP","FP":"FP","FN":"FN","N":"N","UNK":"UNK","IGN":"IGN"}
events = []
observations = []
accounting_rows = []
event_by_id = {}
all_event_bounds = {}

def value_text(v):
    if v is None:
        return "UNKNOWN"
    if isinstance(v, tuple):
        return "/".join("." if x is None else str(x) for x in v)
    return str(v)

def genotype_text(sample):
    gt = sample.get("GT")
    if gt is None:
        return None
    sep = "|" if getattr(sample, "phased", False) else "/"
    return sep.join("." if x is None else str(x) for x in gt)

def exact_lookup(contig, pos, ref, alts):
    if len(alts) != 1:
        return None
    alt = alts[0]
    if not re.fullmatch(r"[ACGTN]+", ref, re.I) or not re.fullmatch(r"[ACGTN]+", alt, re.I):
        return None
    return f"GRCh38|{contig}|{pos}|{ref.upper()}|{alt.upper()}"

core_bed_intervals = []
for line in core_bed.read_text().splitlines():
    if line.strip() and not line.startswith(("#","track","browser")):
        fields = line.split("\t")
        core_bed_intervals.append((int(fields[1]), int(fields[2])))

def benchmark_region_status(start0, end0):
    if any(start0 >= s and end0 <= e for s, e in core_bed_intervals):
        return "INSIDE"
    if any(start0 < e and end0 > s for s, e in core_bed_intervals):
        return "BORDER"
    return "OUTSIDE"

with stage_guard("event observation parsing"):
    for submission in ("60z59", "ru88n"):
        st = runs_state[submission]
        run_id = st["run_id"]
        output_artifact_id = f"artifact-{submission}-annotated-vcf"
        vf = pysam.VariantFile(str(st["annotated"]))
        has_bs = "BS" in vf.header.formats
        for ordinal, rec in enumerate(vf, 1):
            start0 = rec.pos - 1
            end0 = start0 + len(rec.ref)
            wholly_core = (
                rec.contig == CORE["contig"] and
                start0 >= CORE["start_0based"] and end0 <= CORE["end_0based"]
            )
            alts = list(rec.alts or [])
            raw_alt = ",".join(alts) if alts else "."
            bs_values = []
            for side in ("TRUTH","QUERY"):
                if has_bs:
                    v = rec.samples[side].get("BS")
                    if v not in (None, ".", ""):
                        bs_values.append(value_text(v))
            source_bs = bs_values[0] if bs_values and len(set(bs_values)) == 1 else None
            safe_bs = re.sub(r"[^A-Za-z0-9_.-]+", "_", source_bs) if source_bs else None
            if source_bs:
                event_id = f"event-{run_id}-bs-{safe_bs}"
                identity_scope = "COMPARATOR_SUPERLOCUS"
                derivation = "hap.py BS, scoped to this benchmark run"
            else:
                event_id = f"event-{run_id}-record-{ordinal}"
                identity_scope = "SOURCE_RECORD_FALLBACK"
                derivation = "deterministic source-record fallback because BS absent"
            if event_id not in all_event_bounds:
                all_event_bounds[event_id] = [start0, end0]
            else:
                all_event_bounds[event_id][0] = min(all_event_bounds[event_id][0], start0)
                all_event_bounds[event_id][1] = max(all_event_bounds[event_id][1], end0)
            if wholly_core:
                if event_id not in event_by_id:
                    event = {
                        "schema_version":"1.0.0", "event_id":event_id,
                        "identity_scope":identity_scope, "benchmark_run_id":run_id,
                        "reference_artifact_id":"artifact-reference-grch38-no-alt",
                        "assembly":"GRCh38", "contig":rec.contig,
                        "start_0based":start0, "end_0based":end0,
                        "source_event_id":source_bs,
                        "normalized_allele_lookup_key":exact_lookup(rec.contig, rec.pos, rec.ref, alts),
                        "derivation_method":derivation, "derivation_version":"phase1a_colab_poc/1.0.0",
                        "cross_run_equivalence_asserted":False,
                        "notes":"Run-scoped event; lookup key does not assert biological equivalence.",
                    }
                    events.append(event)
                    event_by_id[event_id] = event
                else:
                    e = event_by_id[event_id]
                    e["start_0based"] = min(e["start_0based"], start0)
                    e["end_0based"] = max(e["end_0based"], end0)
                    if e["normalized_allele_lookup_key"] != exact_lookup(rec.contig, rec.pos, rec.ref, alts):
                        e["normalized_allele_lookup_key"] = None

                for side in ("TRUTH","QUERY"):
                    sample = rec.samples[side]
                    raw_bd = value_text(sample.get("BD"))
                    raw_bk = value_text(sample.get("BK"))
                    raw_bs = value_text(sample.get("BS")) if has_bs else "UNKNOWN"
                    formats = {k:value_text(sample.get(k)) for k in sample.keys()}
                    not_assessed = raw_bd in {"N","UNK","IGN",".","UNKNOWN"}
                    observations.append({
                        "schema_version":"1.0.0",
                        "observation_id":f"obs-{run_id}-{ordinal}-{side.lower()}",
                        "benchmark_run_id":run_id, "event_id":event_id, "side":side,
                        "observation_origin":"REGENERATED",
                        "raw_decision":raw_bd,
                        "normalized_decision":DECISION_MAP.get(raw_bd, "OTHER"),
                        "raw_match_kind":raw_bk,
                        "raw_engine_detail":json.dumps({
                            "BS":raw_bs, "all_format_values":formats,
                            "assessability":"NOT_ASSESSED" if not_assessed else "ASSESSED",
                        }, sort_keys=True),
                        "raw_variant_type":value_text(sample.get("BVT")) if "BVT" in sample else None,
                        "raw_location_type":value_text(sample.get("BLT")) if "BLT" in sample else None,
                        "region_status":benchmark_region_status(start0, end0),
                        "source_output_artifact_id":output_artifact_id,
                        "source_record_ordinal":ordinal, "source_contig":rec.contig,
                        "source_pos_1based":rec.pos, "source_ref":rec.ref,
                        "source_alt":raw_alt, "source_genotype":genotype_text(sample),
                        "source_filter":list(rec.filter.keys()) or ["."],
                        "quality_score":float(rec.qual) if rec.qual is not None else None,
                        "normalization_method":"source representation retained; exact lookup only when unambiguous",
                        "normalization_version":"phase1a_colab_poc/1.0.0",
                        "normalized_start_0based":start0,
                        "normalized_end_0based":end0,
                        "normalized_ref":rec.ref if len(alts) == 1 else None,
                        "normalized_alt":alts[0] if len(alts) == 1 else None,
                        "notes":"Final Phase 1A observation is wholly inside locked core interval.",
                    })
            accounting_rows.append({
                "benchmark_run_id":run_id, "source_record_ordinal":ordinal,
                "chromosome":rec.contig, "position_1based":rec.pos,
                "ref":rec.ref, "alt":raw_alt,
                "truth_raw_decision":value_text(rec.samples["TRUTH"].get("BD")),
                "query_raw_decision":value_text(rec.samples["QUERY"].get("BD")),
                "benchmark_region_status":benchmark_region_status(start0, end0),
                "classification":"CORE_EXPORTED" if wholly_core else "PADDING_OR_OUTSIDE_CORE_DIAGNOSTIC",
                "event_id":event_id,
            })
        vf.close()
    exportable_event_ids = {eid for eid, (s, e) in all_event_bounds.items()
                            if CORE["start_0based"] <= s and e <= CORE["end_0based"]}
    events[:] = [e for e in events if e["event_id"] in exportable_event_ids]
    observations[:] = [o for o in observations if o["event_id"] in exportable_event_ids]
    for event in events:
        event["start_0based"], event["end_0based"] = all_event_bounds[event["event_id"]]
    for row in accounting_rows:
        if row["event_id"] not in exportable_event_ids:
            row["classification"] = "BOUNDARY_EVENT_DIAGNOSTIC"
    write_tsv(RESULTS_DIR / "annotated_record_accounting.tsv", accounting_rows)


## 16. Validate provenance/accounting

Construct EXPERIMENT, BENCHMARK_RUN, SOURCE_ARTIFACT, and PROVENANCE_LINK entities. Every generated slice, index, SDF manifest, annotated VCF/index, metrics file, and log gets a checksum and lineage edge. Observation lineage resolves through its annotated output and run to all four scientific inputs.

In [ ]:
experiments = []
benchmark_runs = []

with stage_guard("provenance and accounting construction"):
    metadata_ids = ["artifact-pfda-readme","artifact-pfda-submission-methods",
                    "artifact-pfda-anonymized-metadata"]
    for row in selected_rows:
        experiments.append({
            "schema_version":"1.0.0", "experiment_id":row["experiment_id"],
            "source_corpus":row["source_corpus"], "source_submission_id":row["source_submission_id"],
            "sample_id":row["sample_id"], "technology":row["technology"],
            "platform":row["platform"], "coverage":row["coverage"],
            "library_preparation":"UNKNOWN",
            "basecaller":"Guppy" if row["technology"] == "ONT" else "UNKNOWN",
            "basecaller_version":"3.6" if row["technology"] == "ONT" else "UNKNOWN",
            "aligner_names":row["aligner_names"].split(";"),
            "aligner_versions":row["aligner_versions"].split(";"),
            "pipeline_name":row["pipeline_name"], "pipeline_version":"UNKNOWN",
            "caller_names":row["caller_names"].split(";"),
            "caller_versions":row["caller_versions"].split(";"),
            "reference_assembly":"GRCh38", "metadata_artifact_ids":metadata_ids,
            "selection_basis":row["selection_basis"],
            "performance_consulted_for_selection":False,
            "notes":row["metadata_status"],
        })
        for mid in metadata_ids:
            add_link("SOURCE_ARTIFACT", mid, "EXPERIMENT", row["experiment_id"],
                     "DESCRIBES", "public metadata supports experiment claim",
                     "phase1a_colab_poc", "1.0.0")

    for submission in ("60z59","ru88n"):
        st = runs_state[submission]
        run_id = st["run_id"]
        generated = []
        products = [
            (f"artifact-{submission}-annotated-vcf", st["compact_annotated"], "COMPARATOR_OUTPUT",
             "hap.py 0.3.15 annotated VCF"),
            (f"artifact-{submission}-annotated-vcf-tbi", st["compact_annotated_tbi"], "REFERENCE_INDEX",
             "tabix index for annotated VCF"),
            (f"artifact-{submission}-stdout", RESULTS_DIR / f"{submission}.{st['stdout'].name}", "RUN_LOG",
             "hap.py stdout"),
            (f"artifact-{submission}-stderr", RESULTS_DIR / f"{submission}.{st['stderr'].name}", "RUN_LOG",
             "hap.py stderr"),
            (f"artifact-{submission}-timing", RESULTS_DIR / f"{submission}.{st['timing'].name}", "RUN_LOG",
             "GNU time resource report"),
        ]
        for i, metric in enumerate(st["metrics"]):
            dest = RESULTS_DIR / "comparison_outputs" / f"{submission}.{metric.name}"
            shutil.copy2(metric, dest)
            products.append((f"artifact-{submission}-metric-{i+1}", dest, "COMPARATOR_OUTPUT",
                             "hap.py metrics output"))
        for aid, path, role, note in products:
            make_derived_artifact(aid, path, role, "Phase 1A regenerated", note)
            generated.append(aid)
            add_link("BENCHMARK_RUN", run_id, "SOURCE_ARTIFACT", aid, "GENERATED",
                     note, "hap.py", "0.3.15")
        benchmark_runs.append({
            "schema_version":"1.0.0", "benchmark_run_id":run_id,
            "experiment_id":st["experiment_id"], "observation_origin":"REGENERATED",
            "query_artifact_id":selected[submission]["query_artifact_id"],
            "truth_artifact_id":"artifact-truth-hg002-grch38-v421",
            "benchmark_bed_artifact_id":"artifact-bed-hg002-grch38-v421",
            "reference_artifact_id":"artifact-reference-grch38-no-alt",
            "core_interval":CORE, "padded_interval":PADDED,
            "comparator":"hap.py", "comparator_version":"0.3.15",
            "engine":"vcfeval", "engine_version":"3.12.1",
            "environment_lock_id":lock["lock_id"],
            "runtime_image_digest":runtime_identity["observed_registry_digest_after_pull"],
            "command_argv":st["command_argv"],
            "parameters":{"threads":2,"annotated_vcf":True,"immutable_args":lock["comparison"]["immutable_args"]},
            "run_status":"SUCCEEDED", "started_at":st["started_at"],
            "completed_at":st["completed_at"], "output_artifact_ids":generated,
            "failure_reason":None,
            "notes":"New OMICSEDGE regenerated run; not an original precisionFDA event result.",
        })
        for input_id in (
            selected[submission]["query_artifact_id"],
            "artifact-truth-hg002-grch38-v421",
            "artifact-bed-hg002-grch38-v421",
            "artifact-reference-grch38-no-alt",
            "artifact-happy-container", "artifact-rtg-tools-3121",
        ):
            add_link("SOURCE_ARTIFACT", input_id, "BENCHMARK_RUN", run_id, "USED",
                     "exact input to regenerated comparison", "phase1a_colab_poc", "1.0.0")
    for obs in observations:
        add_link("SOURCE_ARTIFACT", obs["source_output_artifact_id"], "OBSERVATION",
                 obs["observation_id"], "NORMALIZED_FROM",
                 "side-specific event observation parsed from annotated VCF record",
                 "phase1a_colab_poc", "1.0.0",
                 {"source_record_ordinal":obs["source_record_ordinal"],"side":obs["side"]})
    for event in events:
        add_link("BENCHMARK_RUN", event["benchmark_run_id"], "EVENT", event["event_id"],
                 "GENERATED", event["derivation_method"], "hap.py", "0.3.15")
    ancillary_logs = [
        ("artifact-runtime-identity", RESULTS_DIR / "runtime_identity.json", "captured runtime identity"),
        ("artifact-udocker-inspect", RESULTS_DIR / "udocker_inspect.txt", "rootless container inspection"),
        ("artifact-annotation-gates", RESULTS_DIR / "annotation_gates.json", "annotated VCF gate evidence"),
        ("artifact-record-accounting", RESULTS_DIR / "annotated_record_accounting.tsv", "padded-record accounting"),
    ]
    for aid, path, note in ancillary_logs:
        make_derived_artifact(aid, path, "RUN_LOG", "Phase 1A regenerated", note)
        for run_row in benchmark_runs:
            add_link("BENCHMARK_RUN", run_row["benchmark_run_id"], "SOURCE_ARTIFACT",
                     aid, "GENERATED", note, "phase1a_colab_poc", "1.0.0")


## 17. Write Parquet

Validate the six in-memory entity collections against the embedded versioned JSON Schemas. Write compact Parquet for events, observations, and provenance links; round-trip them and register each Parquet as a generated artifact.

In [ ]:
import jsonschema
import pyarrow as pa
import pyarrow.parquet as pq

collections = {
    "events":events, "experiments":experiments, "benchmark_runs":benchmark_runs,
    "observations":observations, "source_artifacts":source_artifacts,
    "provenance_links":provenance_links,
}
schema_files = {
    "events":"event.schema.json", "experiments":"experiment.schema.json",
    "benchmark_runs":"benchmark_run.schema.json", "observations":"observation.schema.json",
    "source_artifacts":"source_artifact.schema.json",
    "provenance_links":"provenance_link.schema.json",
}
schema_validation_errors = []
with stage_guard("schema validation and Parquet writing"):
    for collection_name, rows in collections.items():
        schema = json.loads((SCHEMA_DIR / schema_files[collection_name]).read_text())
        validator = jsonschema.Draft202012Validator(schema)
        for i, row in enumerate(rows):
            for error in validator.iter_errors(row):
                schema_validation_errors.append(f"{collection_name}[{i}]: {error.message}")
    if schema_validation_errors:
        raise RuntimeError("schema validation errors:\n" + "\n".join(schema_validation_errors[:30]))

    parquet_roundtrip = {}
    parquet_id_fields = {"events":"event_id", "observations":"observation_id",
                         "provenance_links":"provenance_link_id"}
    for name in ("events","observations","provenance_links"):
        aid = f"artifact-parquet-{name}"
        for run_row in benchmark_runs:
            add_link("BENCHMARK_RUN", run_row["benchmark_run_id"], "SOURCE_ARTIFACT", aid,
                     "GENERATED", f"write {name} Parquet", "pyarrow", pa.__version__)
    for name in ("events","observations","provenance_links"):
        path = RESULTS_DIR / f"{name}.parquet"
        pq.write_table(pa.Table.from_pylist(collections[name]), path, compression="zstd")
        returned = pq.read_table(path).to_pylist()
        parquet_roundtrip[name] = {
            "expected_rows":len(collections[name]), "actual_rows":len(returned),
            "ids_match":set(r[parquet_id_fields[name]] for r in returned) ==
                        set(r[parquet_id_fields[name]] for r in collections[name]),
        }
        if parquet_roundtrip[name]["expected_rows"] != parquet_roundtrip[name]["actual_rows"] or not parquet_roundtrip[name]["ids_match"]:
            raise RuntimeError(f"Parquet round-trip mismatch: {name}")
        aid = f"artifact-parquet-{name}"
        make_derived_artifact(aid, path, "PARQUET", "schema 1.0.0", f"{name} entity table")


## 18. Build/query DuckDB

Build a temporary DuckDB database under `/content/omicsedge_phase1a/`, create views over Parquet plus temporary tables for experiments/runs/artifacts, and verify view counts. The database is audit-registered but is not placed in the final compact bundle.

In [ ]:
import duckdb

duckdb_path = WORK_ROOT / "phase1a.duckdb"
if duckdb_path.exists():
    duckdb_path.unlink()
with stage_guard("DuckDB construction"):
    con = duckdb.connect(str(duckdb_path))
    con.execute(f"CREATE VIEW events AS SELECT * FROM read_parquet('{RESULTS_DIR / 'events.parquet'}')")
    con.execute(f"CREATE VIEW observations AS SELECT * FROM read_parquet('{RESULTS_DIR / 'observations.parquet'}')")
    con.execute(f"CREATE VIEW provenance_links AS SELECT * FROM read_parquet('{RESULTS_DIR / 'provenance_links.parquet'}')")
    con.register("_experiments_df", pa.Table.from_pylist(experiments))
    con.register("_runs_df", pa.Table.from_pylist(benchmark_runs))
    con.register("_artifacts_df", pa.Table.from_pylist(source_artifacts))
    con.execute("CREATE TABLE experiments AS SELECT * FROM _experiments_df")
    con.execute("CREATE TABLE benchmark_runs AS SELECT * FROM _runs_df")
    con.execute("CREATE TABLE source_artifacts AS SELECT * FROM _artifacts_df")
    duck_counts = {
        name:con.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
        for name in ("events","observations","provenance_links","experiments","benchmark_runs","source_artifacts")
    }
    if duck_counts["events"] != len(events) or duck_counts["observations"] != len(observations):
        raise RuntimeError("DuckDB/Parquet count mismatch")


## 19. Produce one event-evidence query

Return event evidence for one run-scoped event with its observations, experiment, benchmark-run context, annotated artifact, and raw-record provenance. Separately emit exact-representation cross-run candidates labeled `CROSS_RUN_EQUIVALENCE_UNRESOLVED`; never merge run-scoped events or compute a universal trust score.

In [ ]:
with stage_guard("event evidence query"):
    if not events:
        raise RuntimeError("no core events available for event-evidence query")
    chosen_event_id = events[0]["event_id"]
    query_sql = QUERY_SQL_PATH.read_text().replace("$event_id", "?")
    query_df = con.execute(query_sql, [chosen_event_id]).fetchdf()
    if query_df.empty:
        raise RuntimeError("event-evidence query returned no rows")
    query_df.to_csv(RESULTS_DIR / "example_event_query.tsv", sep="\t", index=False)

    candidate_sql = """
    SELECT a.benchmark_run_id AS run_a, a.event_id AS event_a,
           b.benchmark_run_id AS run_b, b.event_id AS event_b,
           a.normalized_allele_lookup_key,
           'CROSS_RUN_EQUIVALENCE_UNRESOLVED' AS equivalence_status
    FROM events a JOIN events b
      ON a.normalized_allele_lookup_key = b.normalized_allele_lookup_key
     AND a.benchmark_run_id < b.benchmark_run_id
    WHERE a.normalized_allele_lookup_key IS NOT NULL
      AND a.assembly=b.assembly AND a.contig=b.contig
      AND a.start_0based=b.start_0based AND a.end_0based=b.end_0based
    ORDER BY a.normalized_allele_lookup_key
    """
    candidate_df = con.execute(candidate_sql).fetchdf()
    candidate_df.to_csv(RESULTS_DIR / "cross_run_candidates.tsv", sep="\t", index=False)
    con.close()

    make_derived_artifact("artifact-duckdb-temporary", duckdb_path, "DUCKDB",
                          duckdb.__version__, "Temporary query database; excluded from final archive.")
    for run_row in benchmark_runs:
        add_link("BENCHMARK_RUN", run_row["benchmark_run_id"], "SOURCE_ARTIFACT",
                 "artifact-duckdb-temporary", "GENERATED", "build temporary DuckDB",
                 "duckdb", duckdb.__version__)


## 20. Run all locked Phase 1A validations

Evaluate the exact 20 pre-registered criteria. Performance metrics are never thresholds. Allowed states are `PASS`, `FAIL`, and `NOT_TRIGGERED`; overall compute passes only when all mandatory criteria pass.

In [ ]:
def parse_locked_criteria(markdown):
    criteria = {}
    current = None
    parts = []
    for raw in markdown.splitlines():
        m = re.match(r"^(\d+)\.\s+(.*)$", raw)
        if m:
            if current is not None:
                criteria[current] = " ".join(parts).strip()
            current = int(m.group(1)); parts = [m.group(2).strip()]
        elif current is not None:
            if raw.startswith("## ") or raw.startswith("Any failed"):
                criteria[current] = " ".join(parts).strip()
                current = None; parts = []
            elif raw.strip():
                parts.append(raw.strip())
    if current is not None:
        criteria[current] = " ".join(parts).strip()
    return criteria

LOCKED_CRITERIA = parse_locked_criteria(VALIDATION_PATH.read_text())
if set(LOCKED_CRITERIA) != set(range(1,21)):
    raise RuntimeError(f"expected 20 locked criteria, found {sorted(LOCKED_CRITERIA)}")

FINAL_TAR_ARGV = ["tar", "-C", "/content", "-czf", str(ARCHIVE_PATH), RESULTS_DIR.name]
append_command(FINAL_TAR_ARGV)
make_derived_artifact("artifact-commands-log", COMMAND_LOG, "RUN_LOG",
                      "Phase 1A regenerated", "Exact argv audit log, including packaging command.")
for run_row in benchmark_runs:
    add_link("BENCHMARK_RUN", run_row["benchmark_run_id"], "SOURCE_ARTIFACT",
             "artifact-commands-log", "GENERATED", "record exact command argv",
             "phase1a_colab_poc", "1.0.0")
provenance_parquet_path = RESULTS_DIR / "provenance_links.parquet"
pq.write_table(pa.Table.from_pylist(provenance_links), provenance_parquet_path, compression="zstd")
provenance_artifact = next(a for a in source_artifacts
                           if a["source_artifact_id"] == "artifact-parquet-provenance_links")
provenance_artifact["byte_size"] = provenance_parquet_path.stat().st_size
provenance_artifact["checksums"] = [{"algorithm":"sha256",
    "value":sha256_file(provenance_parquet_path), "source":"COMPUTED_ON_RETRIEVAL"}]
provenance_returned = pq.read_table(provenance_parquet_path).to_pylist()
parquet_roundtrip["provenance_links"] = {
    "expected_rows":len(provenance_links), "actual_rows":len(provenance_returned),
    "ids_match":{r["provenance_link_id"] for r in provenance_returned} ==
                {r["provenance_link_id"] for r in provenance_links},
}
with duckdb.connect(str(duckdb_path), read_only=True) as count_con:
    final_duck_counts = {name:count_con.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
                         for name in ("events","observations","provenance_links")}

def ids(rows, key):
    return [r[key] for r in rows]
artifact_ids = set(ids(source_artifacts, "source_artifact_id"))
run_ids = set(ids(benchmark_runs, "benchmark_run_id"))
event_ids = set(ids(events, "event_id"))
obs_ids = set(ids(observations, "observation_id"))
exp_ids = set(ids(experiments, "experiment_id"))
link_endpoints = {
    (p["source_entity_type"],p["source_entity_id"]) for p in provenance_links
} | {
    (p["target_entity_type"],p["target_entity_id"]) for p in provenance_links
}
derived_roles = {"DERIVED_SLICE","REFERENCE_INDEX","COMPARATOR_OUTPUT","RUN_LOG","PARQUET","DUCKDB"}
derived_ids = {a["source_artifact_id"] for a in source_artifacts if a["artifact_role"] in derived_roles}
derived_with_edges = {aid for typ,aid in link_endpoints if typ == "SOURCE_ARTIFACT"}

annotated_core_counts = {}
observation_counts = {}
annotated_decision_counts = {}
observation_decision_counts = {}
for r in accounting_rows:
    if r["classification"] == "CORE_EXPORTED":
        annotated_core_counts[r["benchmark_run_id"]] = annotated_core_counts.get(r["benchmark_run_id"],0) + 2
        for side, column in (("TRUTH","truth_raw_decision"),("QUERY","query_raw_decision")):
            key = (r["benchmark_run_id"], side, r[column])
            annotated_decision_counts[key] = annotated_decision_counts.get(key, 0) + 1
for o in observations:
    observation_counts[o["benchmark_run_id"]] = observation_counts.get(o["benchmark_run_id"],0) + 1
    key = (o["benchmark_run_id"], o["side"], o["raw_decision"])
    observation_decision_counts[key] = observation_decision_counts.get(key, 0) + 1

def all_unique():
    return all(len(x)==len(set(x)) for x in (
        ids(events,"event_id"), ids(experiments,"experiment_id"),
        ids(benchmark_runs,"benchmark_run_id"), ids(observations,"observation_id"),
        ids(source_artifacts,"source_artifact_id"), ids(provenance_links,"provenance_link_id")))

def endpoints_resolve():
    maps = {
        "EVENT":event_ids, "EXPERIMENT":exp_ids, "BENCHMARK_RUN":run_ids,
        "OBSERVATION":obs_ids, "SOURCE_ARTIFACT":artifact_ids,
        "PROVENANCE_LINK":set(ids(provenance_links,"provenance_link_id")),
    }
    return all(p["source_entity_id"] in maps[p["source_entity_type"]] and
               p["target_entity_id"] in maps[p["target_entity_type"]]
               for p in provenance_links)

def observation_lineage_complete(obs):
    run_row = next(r for r in benchmark_runs if r["benchmark_run_id"] == obs["benchmark_run_id"])
    normalized_edge = any(
        p["source_entity_type"] == "SOURCE_ARTIFACT" and
        p["source_entity_id"] == obs["source_output_artifact_id"] and
        p["target_entity_type"] == "OBSERVATION" and
        p["target_entity_id"] == obs["observation_id"] and
        p["relation"] == "NORMALIZED_FROM" for p in provenance_links)
    scientific_inputs = {run_row["query_artifact_id"], run_row["truth_artifact_id"],
                         run_row["benchmark_bed_artifact_id"], run_row["reference_artifact_id"]}
    used_inputs = {p["source_entity_id"] for p in provenance_links
                   if p["source_entity_type"] == "SOURCE_ARTIFACT" and
                      p["target_entity_type"] == "BENCHMARK_RUN" and
                      p["target_entity_id"] == run_row["benchmark_run_id"] and
                      p["relation"] == "USED"}
    return normalized_edge and scientific_inputs <= used_inputs

final_schema_errors = []
for collection_name, rows in collections.items():
    schema = json.loads((SCHEMA_DIR / schema_files[collection_name]).read_text())
    validator = jsonschema.Draft202012Validator(schema)
    for i, row in enumerate(rows):
        final_schema_errors.extend(
            f"{collection_name}[{i}]: {error.message}" for error in validator.iter_errors(row))

with stage_guard("locked Phase 1A validation"):
    supplied_checks_ok = all(r["status"]=="VERIFIED" for r in retrieval_rows)
    query_rows_ok = all(
        next(r for r in retrieval_rows if r["source_artifact_id"]==selected[s]["query_artifact_id"])["computed_sha256"]
        == source_manifest[selected[s]["query_artifact_id"]]["checksum_value"]
        for s in ("60z59","ru88n"))
    sci_ids = {
        "artifact-truth-hg002-grch38-v421","artifact-truth-hg002-grch38-v421-tbi",
        "artifact-bed-hg002-grch38-v421","artifact-reference-grch38-no-alt",
        "artifact-reference-grch38-no-alt-fai","artifact-reference-grch38-no-alt-gzi",
    }
    retrieved_ids = {r["source_artifact_id"] for r in retrieval_rows}

    results_bool = {
        1: query_rows_ok,
        2: sci_ids <= retrieved_ids and supplied_checks_ok,
        3: derived_ids <= derived_with_edges and all(a["checksums"] for a in source_artifacts if a["source_artifact_id"] in derived_ids),
        4: all(set(e["metadata_artifact_ids"]) <= artifact_ids for e in experiments) and
           selected["60z59"]["caller_versions"].startswith("UNKNOWN") and
           selected["ru88n"]["caller_versions"] == "UNKNOWN;UNKNOWN",
        5: all(o["observation_origin"]=="REGENERATED" for o in observations) and
           all(r["observation_origin"]=="REGENERATED" for r in benchmark_runs),
        6: all(r["core_interval"]==CORE and r["padded_interval"]==PADDED and
               r["comparator_version"]=="0.3.15" and r["engine_version"]=="3.12.1" and
               r["parameters"]["threads"]==2 and r["parameters"]["annotated_vcf"] is True
               for r in benchmark_runs),
        7: core_assessable_bases > 0,
        8: (runtime_identity.get("hap_py_runtime_established_version") == "0.3.15" and
            bool(runtime_identity.get("hap_py_runtime_version_evidence_sources")) and
            "3.12.1" in runtime_identity["rtg_version_output"] and
            runtime_identity["observed_registry_digest_after_pull"] ==
            lock["hap_py"]["container"]["manifest_digest"] and
            all(k in runtime_identity for k in
                ("bcftools_version","samtools_version","skopeo_version","python_version"))),
        9: all(runs_state[s]["exit_code"]==0 and annotation_gates[s]["record_count"]>0 and
               runs_state[s]["stdout"].exists() and runs_state[s]["stderr"].exists() and
               runs_state[s]["metrics"] for s in ("60z59","ru88n")),
        10: all(g["has_TRUTH"] and g["has_QUERY"] and g["has_BD"] and g["has_BK"]
                for g in annotation_gates.values()),
        11: (len(accounting_rows) == sum(g["record_count"] for g in annotation_gates.values()) and
             all(r["classification"] in {"CORE_EXPORTED","PADDING_OR_OUTSIDE_CORE_DIAGNOSTIC",
                                                "BOUNDARY_EVENT_DIAGNOSTIC"}
                 for r in accounting_rows)),
        12: all(o["raw_decision"] and o["raw_match_kind"] and o["source_record_ordinal"]>=1 and
                o["source_filter"] is not None and "source_genotype" in o
                for o in observations),
        13: all(o["event_id"] in event_ids and o["benchmark_run_id"] in run_ids and
                CORE["start_0based"] <= o["normalized_start_0based"] and
                o["normalized_end_0based"] <= CORE["end_0based"] and
                observation_lineage_complete(o)
                for o in observations),
        14: all(any(e["benchmark_run_id"]==rid for e in events) and
                any(o["benchmark_run_id"]==rid for o in observations) for rid in run_ids),
        15: (all(annotated_core_counts.get(rid,0)==observation_counts.get(rid,0) for rid in run_ids) and
             annotated_decision_counts == observation_decision_counts),
        16: not final_schema_errors and all_unique() and endpoints_resolve(),
        17: all(v["expected_rows"]==v["actual_rows"] and v["ids_match"] for v in parquet_roundtrip.values()),
        18: (final_duck_counts["events"]==len(events) and
             final_duck_counts["observations"]==len(observations) and
             final_duck_counts["provenance_links"]==len(provenance_links)
             and not query_df.empty and
             {"event_id","observation_id","experiment_id","benchmark_run_id","annotated_output_artifact_id"}
             <= set(query_df.columns)),
        19: (candidate_df.empty or
             set(candidate_df["equivalence_status"])=={"CROSS_RUN_EQUIVALENCE_UNRESOLVED"}) and
            all(e["cross_run_equivalence_asserted"] is False for e in events),
        20: False,
    }

    write_tsv(RESULTS_DIR / "experiments.tsv", experiments)
    write_tsv(RESULTS_DIR / "benchmark_runs.tsv", benchmark_runs)
    write_tsv(RESULTS_DIR / "retrieved_artifacts.tsv", retrieval_rows)

    run_manifest = {
        "project":"Project 003", "phase":"1A", "observation_origin":"REGENERATED",
        "created_at":now_iso(), "selected_submissions":["60Z59","RU88N"],
        "sample":"HG002", "assembly":"GRCh38", "truth_version":"GIAB v4.2.1",
        "core_interval":CORE, "core_display":"chr20:10,000,001-11,000,000",
        "padded_interval":PADDED, "padded_vcf_region":PADDED_VCF_REGION,
        "environment_lock":lock, "runtime_identity":runtime_identity,
        "authoritative_snapshot_sha256":EMBEDDED_SHA256,
        "source_artifacts":source_artifacts,
        "counts":{k:len(v) for k,v in collections.items()},
        "core_assessable_bases":core_assessable_bases,
        "padded_assessable_bases":padded_assessable_bases,
    }
    (RESULTS_DIR / "run_manifest.json").write_text(
        json.dumps(run_manifest, indent=2, sort_keys=True), encoding="utf-8")
    results_bool[20] = (RESULTS_DIR / "run_manifest.json").exists()

    validation_rows = [{
        "validation_id":str(i), "description":LOCKED_CRITERIA[i],
        "status":"PASS" if results_bool[i] else "FAIL",
        "evidence":json.dumps({"criterion":i,"result":results_bool[i]}, sort_keys=True),
        "notes":"Locked before genomic compute; no performance threshold applied.",
    } for i in range(1,21)]
    write_tsv(RESULTS_DIR / "validation_results.tsv", validation_rows,
              ["validation_id","description","status","evidence","notes"])
    results_bool[20] = results_bool[20] and (RESULTS_DIR / "validation_results.tsv").exists()
    validation_rows[-1]["status"] = "PASS" if results_bool[20] else "FAIL"
    validation_rows[-1]["evidence"] = json.dumps({"criterion":20,"result":results_bool[20]})
    write_tsv(RESULTS_DIR / "validation_results.tsv", validation_rows,
              ["validation_id","description","status","evidence","notes"])

    COMPUTE_STATUS = ("PHASE1A_COMPUTE_PASS"
                      if all(results_bool.values()) else "PHASE1A_COMPUTE_FAIL")


## 21. Package compact result archive

Write the compute report and detached checksums, package only compact derived evidence/provenance, and exclude source genomic inputs, the reference, full query/truth VCFs, RTG SDF, container caches, and temporary DuckDB. Print archive path, size, SHA-256, then exactly one final computational status.

In [ ]:
with stage_guard("compact result packaging"):
    report = f"""# Project 003 Phase 1A compute report

- Status: {COMPUTE_STATUS}
- Origin: OMICSEDGE REGENERATED (not original precisionFDA event-level results)
- Submissions: 60Z59 (Illumina), RU88N (ONT)
- Sample/reference/truth: HG002 / GRCh38 / GIAB v4.2.1
- Core: chr20:[10,000,000,11,000,000) (0-based half-open)
- Display: chr20:10,000,001-11,000,000 (1-based inclusive)
- Padding: chr20:[9,999,000,11,001,000) (context only)
- hap.py / RTG vcfeval: 0.3.15 / 3.12.1
- Core assessable bases: {core_assessable_bases}
- Events: {len(events)}
- Observations: {len(observations)}
- Cross-run identity: never asserted; exact-representation hits remain CROSS_RUN_EQUIVALENCE_UNRESOLVED.
"""
    (RESULTS_DIR / "phase1a_compute_report.md").write_text(report, encoding="utf-8")
    refresh_checksums()

    forbidden_suffixes = (".fasta",".fasta.gz",".fa",".fa.gz",".cram",".bam",".fastq",".fq")
    forbidden_names = {
        source_manifest["artifact-query-60z59-hg002"]["artifact_name"],
        source_manifest["artifact-query-ru88n-hg002"]["artifact_name"],
        source_manifest["artifact-truth-hg002-grch38-v421"]["artifact_name"],
    }
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file() and (p.name in forbidden_names or p.name.endswith(forbidden_suffixes)):
            raise RuntimeError(f"large/source genomic input leaked into result bundle: {p}")

    if ARCHIVE_PATH.exists():
        ARCHIVE_PATH.unlink()
    run(["tar", "-C", "/content", "-czf", str(ARCHIVE_PATH), RESULTS_DIR.name])
    print(f"archive path: {ARCHIVE_PATH}")
    print(f"archive size: {ARCHIVE_PATH.stat().st_size}")
    print(f"archive SHA-256: {sha256_file(ARCHIVE_PATH)}")

emit_status_once(COMPUTE_STATUS)
